# Snow-cover modeling: annotated research notebook

**Research sample:** 20 western U.S. ski counties, 2004–2023.  
**Primary outcome in the final analysis:** seasonal days with ≥10 cm snow depth above 2,000 m.  
**Final sensitivity experiments:** +0, +1, +2, +3 °C; Fixed Effects, Random Forest, XGBoost.

## How to use this notebook
This is a **cleaned review copy**, not a fully refactored, top-to-bottom reproducible pipeline.
All original code cells and their order have been retained; stored outputs have been cleared.
The notebook documents several exploratory analyses, changing target definitions, and
diagnostic investigations. **Do not run all cells sequentially and assume every intermediate
model is part of the final specification.**

The final warming-scenario calculations and economic summaries are at the end.
Earlier cells provide historical modeling, robustness checks, exploratory economic work,
and reconstruction of temperature-dependent variables.

### Before reproducible reruns
- Consolidate input paths and confirm the intended source datasets.
- Verify the final target and features throughout model training.
- Explicitly refit the three final models before warming-scenario predictions.
- Verify BEA units and the precise snow-day construction.
- Earth Engine export cells require authentication, cloud access, and may create tasks.


# Predicting Snow Conditions in Western U.S. Ski Counties

## 1. Setup and Data Preparation

This notebook develops and evaluates machine learning models
to predict annual snow-cover duration in ski-dependent counties
of the western United States.

The primary outcome is the number of snow-cover days above
2,000 meters.

Models are evaluated using temporal and geographic validation
to assess their ability to predict snow conditions across
counties and years.

In [ ]:
# Core packages
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Machine learning
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import LeaveOneGroupOut

RANDOM_STATE = 42

pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")

In [ ]:
import pandas as pd

panel = pd.read_csv(
    "/Users/evabenson/Files/thesis/snow_ml/code/snow_county_panel.csv",
    dtype={"fips": str}
)

print(panel.shape)
display(panel.head())

In [ ]:
from pathlib import Path

project = Path("/Users/evabenson/Files/thesis/snow_ml")

for file in sorted(project.rglob("*")):
    if file.is_file() and file.suffix.lower() in [".csv", ".parquet"]:
        print(file.relative_to(project))

## 2. Data Preparation

Load county-level snow observations, historical climate
variables, and geographic characteristics.

In [ ]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path("/Users/evabenson/Files/thesis/snow_ml/ml_model")

snow = pd.read_csv(
    DATA_DIR / "county_snow_panel.csv",
    dtype={"fips": str}
)

climate = pd.read_csv(
    DATA_DIR / "county_climate_panel_v2.csv",
    dtype={"fips": str}
)

terrain = pd.read_csv(
    DATA_DIR / "county_terrain.csv",
    dtype={"fips": str}
)

print("Snow:", snow.shape)
print("Climate:", climate.shape)
print("Terrain:", terrain.shape)

In [ ]:
for name, df in [
    ("Snow", snow),
    ("Climate", climate),
    ("Terrain", terrain)
]:
    print(f"\n{name.upper()}")
    print("Columns:", df.columns.tolist())
    display(df.head())
    print("\nMissing values:")
    print(df.isna().sum())

In [ ]:
for name, df in [
    ("Snow", snow),
    ("Climate", climate),
    ("Terrain", terrain)
]:
    print(f"\n{name.upper()}")

    if "fips" in df.columns:
        print("Counties:", df["fips"].nunique())

    if "year" in df.columns:
        print("Years:", sorted(df["year"].dropna().unique()))

### 2.1 Construct the County-Year Modeling Panel

Merge historical snow-cover observations, climate variables,
and time-invariant terrain characteristics using county FIPS
codes and year.

In [ ]:
# Standardize year formats
snow["year"] = snow["year"].astype(int)
climate["year"] = climate["year"].astype(int)

# Check that merge keys are unique
assert not snow.duplicated(["fips", "year"]).any()
assert not climate.duplicated(["fips", "year"]).any()
assert not terrain.duplicated(["fips"]).any()

# Merge snow and climate
panel = snow.merge(
    climate.drop(columns=["state", "county"]),
    on=["fips", "year"],
    how="left",
    validate="one_to_one"
)

# Merge terrain
panel = panel.merge(
    terrain.drop(columns=["state", "county"]),
    on="fips",
    how="left",
    validate="many_to_one"
)

print("Merged panel shape:", panel.shape)
display(panel.head())

In [ ]:
missing_snow = panel[panel["snow_days_2000m"].isna()]

display(
    missing_snow.groupby(
        ["state", "county", "fips"]
    ).agg(
        missing_years=("year", "count"),
        first_year=("year", "min"),
        last_year=("year", "max")
    ).reset_index()
)

In [ ]:
print(panel.isna().sum())

### 2.2 Define the Modeling Sample

The dependent variable is annual snow-cover duration above
2,000 meters. Observations without a valid snow-cover
measurement are excluded from the primary modeling sample.

The resulting panel contains 21 counties observed annually
from 2001 through 2023.

In [ ]:
# Keep observations with valid snow-cover measurements
model_df = panel.dropna(
    subset=["snow_days_2000m"]
).copy()

# Define target variable
TARGET = "snow_days_2000m"

# Define climate predictors
CLIMATE_FEATURES = [
    "tmin_djf",
    "tmin_apr",
    "precip_on",
    "precip_djf"
]

# Define terrain predictors
TERRAIN_FEATURES = [
    "mean_elev_m",
    "mean_slope_deg",
    "mean_northness",
    "mean_eastness",
    "frac_above_2000m",
    "frac_above_2500m",
    "frac_above_3000m"
]

FEATURES = CLIMATE_FEATURES + TERRAIN_FEATURES

# Sort the panel
model_df = model_df.sort_values(
    ["fips", "year"]
).reset_index(drop=True)

print("Observations:", len(model_df))
print("Counties:", model_df["fips"].nunique())
print("Years:", model_df["year"].nunique())

assert not model_df[FEATURES + [TARGET]].isna().any().any()

display(model_df.head())

## 3. Exploratory Data Analysis

Examine the distribution of snow-cover duration and
differences across counties and years.

In [ ]:
print(model_df[TARGET].describe().round(2))

fig, ax = plt.subplots(figsize=(9, 5))

sns.histplot(
    data=model_df,
    x=TARGET,
    bins=25,
    kde=True,
    ax=ax
)

ax.set_title("Distribution of Annual Snow-Cover Duration")
ax.set_xlabel("Snow-cover days above 2,000 m")
ax.set_ylabel("Number of county-year observations")

plt.tight_layout()
plt.show()

In [ ]:
county_summary = (
    model_df.groupby(["state", "county"])[TARGET]
    .agg(["mean", "std", "min", "max"])
    .reset_index()
    .sort_values("mean", ascending=False)
)

display(county_summary.round(2))

plt.figure(figsize=(10, 8))

sns.barplot(
    data=county_summary,
    x="mean",
    y="county",
    hue="state",
    dodge=False
)

plt.xlabel("Average annual snow-cover days above 2,000 m")
plt.ylabel("County")
plt.title("Average Snow-Cover Duration by County")
plt.tight_layout()
plt.show()

### 3.3 Climate and Snow-Cover Relationships

Examine the relationships between historical climate conditions
and snow-cover duration, distinguishing variation across counties
from year-to-year variation within counties.

In [ ]:
# Correlations across all county-year observations
corr_vars = CLIMATE_FEATURES + TERRAIN_FEATURES + [TARGET]

correlations = (
    model_df[corr_vars]
    .corr()[TARGET]
    .drop(TARGET)
    .sort_values()
)

display(correlations.to_frame("Correlation with snow days").round(3))

plt.figure(figsize=(9, 6))

sns.barplot(
    x=correlations.values,
    y=correlations.index,
    color="steelblue"
)

plt.axvline(0, color="black", linewidth=0.8)
plt.xlabel("Pearson correlation")
plt.ylabel("")
plt.title("Predictor Correlations with Snow-Cover Duration")
plt.tight_layout()
plt.show()

In [ ]:
# Create county-demeaned variables
within_df = model_df.copy()

for col in CLIMATE_FEATURES + [TARGET]:
    within_df[col + "_dm"] = (
        within_df[col]
        - within_df.groupby("fips")[col].transform("mean")
    )

# Correlation between climate anomalies and snow anomalies
within_corr = (
    within_df[
        [c + "_dm" for c in CLIMATE_FEATURES]
        + [TARGET + "_dm"]
    ]
    .corr()[TARGET + "_dm"]
    .drop(TARGET + "_dm")
    .sort_values()
)

display(
    within_corr.to_frame(
        "Within-county correlation"
    ).round(3)
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sns.regplot(
    data=within_df,
    x="tmin_djf_dm",
    y=TARGET + "_dm",
    scatter_kws={"alpha": 0.35},
    ax=axes[0]
)

axes[0].set_title("Winter Temperature vs. Snow Anomalies")
axes[0].set_xlabel("Winter minimum temperature anomaly")
axes[0].set_ylabel("Snow-cover duration anomaly (days)")

sns.regplot(
    data=within_df,
    x="tmin_apr_dm",
    y=TARGET + "_dm",
    scatter_kws={"alpha": 0.35},
    ax=axes[1]
)

axes[1].set_title("April Temperature vs. Snow Anomalies")
axes[1].set_xlabel("April minimum temperature anomaly")
axes[1].set_ylabel("Snow-cover duration anomaly (days)")

plt.tight_layout()
plt.show()

## 4. Examining Cloud Cover



In [ ]:
from pathlib import Path
import pandas as pd

data_dir = Path.home() / "Files/thesis/snow_ml/ml_model"

snow = pd.read_csv(data_dir / "county_snow_validation_v3.csv")
climate = pd.read_csv(data_dir / "county_climate_panel_v3.csv")

print("Snow:", snow.shape)
print("Climate:", climate.shape)

In [ ]:
# Standardize FIPS and year
for data in [snow, climate]:
    data["fips"] = (
        data["fips"].astype(str)
        .str.replace(r"\.0$", "", regex=True)
        .str.zfill(5)
    )
    data["year"] = data["year"].astype(int)

# Merge snow and climate data
df = snow.merge(
    climate,
    on=["fips", "year"],
    how="left",
    validate="one_to_one",
    suffixes=("", "_climate")
)

print("Observations:", len(df))
print("Counties:", df["fips"].nunique())
print("Years:", df["year"].nunique())

print("\nColumns:")
print(df.columns.tolist())

In [ ]:
targets = [
    "snow_detected_2000m",
    "snow_adjusted_2000m",
    "observed_snow_fraction_2000m"
]

print(df[targets].describe().round(2))

print("\nCorrelations between snow measurements:")
print(df[targets].corr().round(3))

In [ ]:
climate_vars = [
    "tmin_djf",
    "tmax_djf",
    "tmin_mar",
    "tmax_mar",
    "tmin_apr",
    "tmax_apr",
    "precip_djf",
    "precip_mar_apr",
    "positive_degree_days",
    "thaw_days"
]

# Keep variables available in your dataset
climate_vars = [v for v in climate_vars if v in df.columns]

# Pooled correlations
print("POOLED CORRELATIONS")
print(
    df[targets + climate_vars]
    .corr()
    .loc[targets, climate_vars]
    .round(3)
    .to_string()
)

# Within-county correlations
within = df.copy()

for col in targets + climate_vars:
    within[col] = (
        within[col]
        - within.groupby("fips")[col].transform("mean")
    )

print("\nWITHIN-COUNTY CORRELATIONS")
print(
    within[targets + climate_vars]
    .corr()
    .loc[targets, climate_vars]
    .round(3)
    .to_string()
)

In [ ]:
import matplotlib.pyplot as plt

annual = df.groupby("year")[
    ["snow_detected_2000m", "snow_adjusted_2000m"]
].mean()

plt.figure(figsize=(11, 5))

plt.plot(
    annual.index,
    annual["snow_detected_2000m"],
    marker="o",
    label="Detected snow days"
)

plt.plot(
    annual.index,
    annual["snow_adjusted_2000m"],
    marker="o",
    label="Cloud-adjusted snow days"
)

plt.xlabel("Winter ending year")
plt.ylabel("Average snow days")
plt.title("Historical Snow Duration in Western U.S. Ski Counties")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# STEP 3: Compare snow duration with climate variables

climate_vars = [
    "tmin_djf_2000m",
    "tmax_djf_2000m",
    "tmin_mar_2000m",
    "tmax_mar_2000m",
    "tmin_apr_2000m",
    "tmax_apr_2000m",
    "precip_djf_2000m",
    "precip_mar_apr_2000m",
    "positive_degree_days_2000m",
    "freezing_degree_days_2000m",
    "thaw_days_2000m",
    "precip_freezing_fraction_2000m"
]

targets = [
    "snow_detected_2000m",
    "snow_adjusted_2000m",
    "observed_snow_fraction_2000m"
]

# Keep observations with complete snow and climate data
analysis = df.dropna(subset=targets + climate_vars).copy()

print("Complete observations:", len(analysis))
print("Counties:", analysis["fips"].nunique())

# 1. Pooled correlations
pooled_corr = (
    analysis[targets + climate_vars]
    .corr()
    .loc[targets, climate_vars]
)

print("\nPOOLED CORRELATIONS")
print(pooled_corr.round(3).to_string())

# 2. Within-county correlations
within = analysis.copy()

for col in targets + climate_vars:
    within[col] = (
        within[col]
        - within.groupby("fips")[col].transform("mean")
    )

within_corr = (
    within[targets + climate_vars]
    .corr()
    .loc[targets, climate_vars]
)

print("\nWITHIN-COUNTY CORRELATIONS")
print(within_corr.round(3).to_string())

In [ ]:
coverage_vars = [
    "valid_fraction_2000m",
    "cloud_fraction_2000m",
    "valid_days_2000m",
    "cloud_days_2000m"
]

print("CLOUD COVERAGE CORRELATIONS")

print(
    analysis[targets + coverage_vars]
    .corr()
    .loc[targets, coverage_vars]
    .round(3)
    .to_string()
)

# Within-county coverage correlations
coverage_within = analysis.copy()

for col in targets + coverage_vars:
    coverage_within[col] = (
        coverage_within[col]
        - coverage_within.groupby("fips")[col].transform("mean")
    )

print("\nWITHIN-COUNTY CLOUD COVERAGE CORRELATIONS")

print(
    coverage_within[targets + coverage_vars]
    .corr()
    .loc[targets, coverage_vars]
    .round(3)
    .to_string()
)

In [ ]:
import statsmodels.formula.api as smf

regression = analysis.copy()

model = smf.ols(
    formula=(
        "snow_adjusted_2000m ~ "
        "positive_degree_days_2000m + "
        "precip_djf_2000m + "
        "precip_mar_apr_2000m + "
        "valid_fraction_2000m + "
        "C(fips) + C(year)"
    ),
    data=regression
).fit(
    cov_type="cluster",
    cov_kwds={"groups": regression["fips"]}
)

print(model.summary())

## 5. Random Forest Model

In [ ]:
# Standardize county identifiers
terrain["fips"] = (
    terrain["fips"].astype(str)
    .str.replace(r"\.0$", "", regex=True)
    .str.zfill(5)
)

# Merge terrain with the snow-climate panel
model_df = df.merge(
    terrain,
    on="fips",
    how="left",
    validate="many_to_one",
    suffixes=("", "_terrain")
)

print("Observations:", len(model_df))
print("Columns:", model_df.columns.tolist())

In [ ]:
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

target = "snow_adjusted_2000m"

climate_features = [
    "tmin_djf_2000m",
    "tmax_djf_2000m",
    "tmin_mar_2000m",
    "tmax_mar_2000m",
    "tmin_apr_2000m",
    "tmax_apr_2000m",
    "precip_djf_2000m",
    "precip_mar_apr_2000m",
    "positive_degree_days_2000m",
    "freezing_degree_days_2000m",
    "thaw_days_2000m"
]

terrain_features = [
    "mean_elev_m",
    "mean_slope_deg",
    "mean_northness",
    "mean_eastness",
    "frac_above_2000m",
    "frac_above_2500m",
    "frac_above_3000m"
]

features = climate_features + terrain_features

# Verify columns before fitting
missing = [c for c in features if c not in model_df.columns]
if missing:
    raise ValueError(f"Missing columns: {missing}")

# Keep complete observations
data = model_df.dropna(subset=[target] + features).copy()

X = data[features]
y = data[target]
groups = data["year"]

print("Training observations:", len(data))
print("Counties:", data["fips"].nunique())
print("Years:", data["year"].nunique())

logo = LeaveOneGroupOut()
predictions = np.full(len(data), np.nan)

for train_idx, test_idx in logo.split(X, y, groups):
    rf = RandomForestRegressor(
        n_estimators=500,
        min_samples_leaf=3,
        max_features=0.7,
        random_state=42,
        n_jobs=-1
    )

    rf.fit(X.iloc[train_idx], y.iloc[train_idx])
    predictions[test_idx] = rf.predict(X.iloc[test_idx])

print("\nLEAVE-ONE-YEAR-OUT RESULTS")
print("R²:", round(r2_score(y, predictions), 3))
print("MAE:", round(mean_absolute_error(y, predictions), 2), "days")
print("RMSE:", round(np.sqrt(mean_squared_error(y, predictions)), 2), "days")

data["predicted_snow_days"] = predictions

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(7, 7))

plt.scatter(
    data[target],
    data["predicted_snow_days"],
    alpha=0.6
)

plt.plot([0, 185], [0, 185], "k--")

plt.xlabel("Observed cloud-adjusted snow days")
plt.ylabel("Predicted snow days")
plt.title("Random Forest: Leave-One-Year-Out Predictions")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
baseline = np.full(len(data), np.nan)

for train_idx, test_idx in logo.split(X, y, groups):
    train = data.iloc[train_idx]
    test = data.iloc[test_idx]

    county_means = train.groupby("fips")[target].mean()
    baseline[test_idx] = (
        test["fips"].map(county_means).to_numpy()
    )

print("COUNTY-MEAN BASELINE")
print("R²:", round(r2_score(y, baseline), 3))
print("MAE:", round(mean_absolute_error(y, baseline), 2), "days")

In [ ]:
# Train final descriptive model on all historical observations
final_rf = RandomForestRegressor(
    n_estimators=500,
    min_samples_leaf=3,
    max_features=0.7,
    random_state=42,
    n_jobs=-1
)

final_rf.fit(X, y)

importance = pd.Series(
    final_rf.feature_importances_,
    index=features
).sort_values(ascending=False)

print(importance.round(3).to_string())

plt.figure(figsize=(10, 7))
importance.sort_values().plot(kind="barh")
plt.xlabel("Random forest impurity-based importance")
plt.title("Snow Duration Model: Feature Importance")
plt.tight_layout()
plt.show()

In [ ]:
# Remove each county's average from observed and predicted snow days
data["observed_anomaly"] = (
    data[target]
    - data.groupby("fips")[target].transform("mean")
)

data["predicted_anomaly"] = (
    data["predicted_snow_days"]
    - data.groupby("fips")["predicted_snow_days"].transform("mean")
)

print("Within-county anomaly R²:",
      round(r2_score(
          data["observed_anomaly"],
          data["predicted_anomaly"]
      ), 3))

print("Within-county anomaly correlation:",
      round(data[
          ["observed_anomaly", "predicted_anomaly"]
      ].corr().iloc[0, 1], 3))

plt.figure(figsize=(7, 7))

plt.scatter(
    data["observed_anomaly"],
    data["predicted_anomaly"],
    alpha=0.6
)

limits = [
    min(data["observed_anomaly"].min(),
        data["predicted_anomaly"].min()),
    max(data["observed_anomaly"].max(),
        data["predicted_anomaly"].max())
]

plt.plot(limits, limits, "k--")

plt.xlabel("Observed snow-duration anomaly (days)")
plt.ylabel("Predicted snow-duration anomaly (days)")
plt.title("Within-County Snow Duration Predictions")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## SNODAS Data

In [ ]:
from pathlib import Path
import pandas as pd

file_path = (
    Path.home()
    / "Files"
    / "thesis"
    / "snow_ml"
    / "ml_model"
    / "county_snodas_climate_terrain_merged.csv"
)

df = pd.read_csv(file_path, dtype={"fips": str})

print("Full dataset:", df.shape)
print("Counties:", df["fips"].nunique())
print("Years:", df["year"].min(), "-", df["year"].max())

In [ ]:

logo = LeaveOneGroupOut()

predictions = np.full(len(model_df), np.nan)
baseline_predictions = np.full(len(model_df), np.nan)

for train_idx, test_idx in logo.split(X, y, groups):

    X_train = X.iloc[train_idx]
    X_test = X.iloc[test_idx]

    y_train = y.iloc[train_idx]

    rf = RandomForestRegressor(
        n_estimators=500,
        min_samples_leaf=3,
        max_features=0.7,
        random_state=42,
        n_jobs=-1
    )

    rf.fit(X_train, y_train)

    predictions[test_idx] = rf.predict(X_test)

    # County historical-mean baseline.
    # IMPORTANT: only use training years.
    train_counties = model_df.iloc[train_idx]["fips"]

    county_means = y_train.groupby(
        train_counties
    ).mean()

    test_counties = model_df.iloc[test_idx]["fips"]

    baseline_predictions[test_idx] = (
        test_counties.map(county_means)
        .fillna(y_train.mean())
        .to_numpy()
    )

model_df["rf_predicted"] = predictions
model_df["baseline_predicted"] = baseline_predictions

print("Cross-validation complete.")
print("Missing predictions:", np.isnan(predictions).sum())


In [ ]:

def evaluate_model(actual, predicted):
    return {
        "R2": r2_score(actual, predicted),
        "MAE": mean_absolute_error(actual, predicted),
        "RMSE": np.sqrt(
            mean_squared_error(actual, predicted)
        )
    }

rf_results = evaluate_model(
    model_df[TARGET],
    model_df["rf_predicted"]
)

baseline_results = evaluate_model(
    model_df[TARGET],
    model_df["baseline_predicted"]
)

performance = pd.DataFrame({
    "Random Forest": rf_results,
    "County Mean Baseline": baseline_results
}).T

print(performance.round(3))

mae_improvement = (
    1 - rf_results["MAE"] / baseline_results["MAE"]
) * 100

print(
    f"\nRF MAE improvement over baseline: "
    f"{mae_improvement:.1f}%"
)


In [ ]:

# Training-only county means were calculated
# during cross-validation.
# They are stored in baseline_predicted.

model_df["observed_anomaly"] = (
    model_df[TARGET] -
    model_df["baseline_predicted"]
)

model_df["predicted_anomaly"] = (
    model_df["rf_predicted"] -
    model_df["baseline_predicted"]
)

anomaly_r2 = r2_score(
    model_df["observed_anomaly"],
    model_df["predicted_anomaly"]
)

anomaly_mae = mean_absolute_error(
    model_df["observed_anomaly"],
    model_df["predicted_anomaly"]
)

anomaly_corr = model_df[
    ["observed_anomaly", "predicted_anomaly"]
].corr().iloc[0, 1]

print(f"Anomaly R2: {anomaly_r2:.3f}")
print(f"Anomaly MAE: {anomaly_mae:.2f} days")
print(f"Anomaly correlation: {anomaly_corr:.3f}")


In [ ]:

plt.figure(figsize=(7, 6))

plt.scatter(
    model_df[TARGET],
    model_df["rf_predicted"],
    alpha=0.6,
    s=25
)

limits = [
    min(model_df[TARGET].min(),
        model_df["rf_predicted"].min()),
    max(model_df[TARGET].max(),
        model_df["rf_predicted"].max())
]

plt.plot(limits, limits, "k--", linewidth=1)

plt.xlabel("Observed SNODAS snow days")
plt.ylabel("Predicted snow days")
plt.title(
    "Random Forest: SNODAS Snow Duration\n"
    "Leave-One-Year-Out Cross-Validation"
)

plt.text(
    0.05, 0.95,
    f"R² = {rf_results['R2']:.3f}\n"
    f"MAE = {rf_results['MAE']:.2f} days",
    transform=plt.gca().transAxes,
    verticalalignment="top"
)

plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()


In [ ]:

# Examine prediction errors by observed snow duration

model_df["prediction_error"] = (
    model_df["rf_predicted"] - model_df[TARGET]
)

model_df["snow_category"] = pd.cut(
    model_df[TARGET],
    bins=[0, 90, 120, 150, 200],
    labels=[
        "Under 90 days",
        "90–120 days",
        "120–150 days",
        "Over 150 days"
    ],
    include_lowest=True
)

error_summary = (
    model_df.groupby(
        "snow_category",
        observed=True
    )
    .agg(
        observations=(TARGET, "count"),
        mean_error=("prediction_error", "mean"),
        MAE=("prediction_error", lambda x: x.abs().mean())
    )
)

print(error_summary.round(2))


## County FE Model

In [ ]:

from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Start with a parsimonious climate specification
fe_features = [
    "positive_degree_days_2000m",
    "precip_djf_2000m",
    "precip_mar_apr_2000m"
]

X_fe = model_df[fe_features + ["fips"]]
y_fe = model_df["snow_days_10cm_2000m"]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "county",
            OneHotEncoder(handle_unknown="ignore"),
            ["fips"]
        ),
        (
            "climate",
            "passthrough",
            fe_features
        )
    ]
)

fe_model = Pipeline([
    ("preprocessor", preprocessor),
    ("regression", LinearRegression())
])

fe_predictions = np.full(len(model_df), np.nan)

logo = LeaveOneGroupOut()

for train_idx, test_idx in logo.split(
    X_fe, y_fe, groups=model_df["year"]
):
    fe_model.fit(
        X_fe.iloc[train_idx],
        y_fe.iloc[train_idx]
    )

    fe_predictions[test_idx] = fe_model.predict(
        X_fe.iloc[test_idx]
    )

model_df["fe_predicted"] = fe_predictions

print("Fixed-effects cross-validation complete.")


In [ ]:

comparison = pd.DataFrame({
    "Random Forest": evaluate_model(
        model_df[TARGET],
        model_df["rf_predicted"]
    ),
    "County FE Linear Model": evaluate_model(
        model_df[TARGET],
        model_df["fe_predicted"]
    ),
    "County Mean Baseline": evaluate_model(
        model_df[TARGET],
        model_df["baseline_predicted"]
    )
}).T

print(comparison.round(3))


In [ ]:

model_df["fe_error"] = (
    model_df["fe_predicted"] - model_df[TARGET]
)

bias_comparison = model_df.groupby(
    "snow_category",
    observed=True
).agg(
    observations=(TARGET, "count"),
    rf_mean_error=("prediction_error", "mean"),
    fe_mean_error=("fe_error", "mean"),
    rf_mae=("prediction_error", lambda x: x.abs().mean()),
    fe_mae=("fe_error", lambda x: x.abs().mean())
)

print(bias_comparison.round(2))


In [ ]:

from sklearn.metrics import r2_score, mean_absolute_error

# Use the training-only county baseline from
# the leave-one-year-out cross-validation.

model_df["fe_predicted_anomaly"] = (
    model_df["fe_predicted"] -
    model_df["baseline_predicted"]
)

fe_anomaly_r2 = r2_score(
    model_df["observed_anomaly"],
    model_df["fe_predicted_anomaly"]
)

fe_anomaly_mae = mean_absolute_error(
    model_df["observed_anomaly"],
    model_df["fe_predicted_anomaly"]
)

fe_anomaly_corr = model_df[
    ["observed_anomaly", "fe_predicted_anomaly"]
].corr().iloc[0, 1]

anomaly_comparison = pd.DataFrame({
    "Random Forest": {
        "Anomaly R2": anomaly_r2,
        "Anomaly MAE": anomaly_mae,
        "Anomaly Correlation": anomaly_corr
    },
    "County FE Linear": {
        "Anomaly R2": fe_anomaly_r2,
        "Anomaly MAE": fe_anomaly_mae,
        "Anomaly Correlation": fe_anomaly_corr
    }
}).T

print(anomaly_comparison.round(3))


## Testing Non-Linear Temperature

In [ ]:
model_df["pdd_squared"] = (
    model_df["positive_degree_days_2000m"] ** 2
)

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import LeaveOneGroupOut

quad_features = [
    "positive_degree_days_2000m",
    "pdd_squared",
    "precip_djf_2000m",
    "precip_mar_apr_2000m"
]

X_quad = model_df[quad_features + ["fips"]]
y_quad = model_df[TARGET]

quad_preprocessor = ColumnTransformer(
    transformers=[
        (
            "county",
            OneHotEncoder(handle_unknown="ignore"),
            ["fips"]
        ),
        (
            "climate",
            "passthrough",
            quad_features
        )
    ]
)

quad_model = Pipeline([
    ("preprocessor", quad_preprocessor),
    ("regression", LinearRegression())
])

quad_predictions = np.full(len(model_df), np.nan)

logo = LeaveOneGroupOut()

for train_idx, test_idx in logo.split(
    X_quad,
    y_quad,
    groups=model_df["year"]
):
    quad_model.fit(
        X_quad.iloc[train_idx],
        y_quad.iloc[train_idx]
    )

    quad_predictions[test_idx] = quad_model.predict(
        X_quad.iloc[test_idx]
    )

model_df["quad_predicted"] = quad_predictions

In [ ]:
comparison = pd.DataFrame({
    "Random Forest": evaluate_model(
        model_df[TARGET],
        model_df["rf_predicted"]
    ),
    "County FE Linear": evaluate_model(
        model_df[TARGET],
        model_df["fe_predicted"]
    ),
    "County FE Quadratic": evaluate_model(
        model_df[TARGET],
        model_df["quad_predicted"]
    )
}).T

print(comparison.round(3))

In [ ]:
model_df["quad_predicted_anomaly"] = (
    model_df["quad_predicted"] -
    model_df["baseline_predicted"]
)

quad_anomaly_r2 = r2_score(
    model_df["observed_anomaly"],
    model_df["quad_predicted_anomaly"]
)

quad_anomaly_corr = model_df[
    ["observed_anomaly", "quad_predicted_anomaly"]
].corr().iloc[0, 1]

print(f"Quadratic anomaly R²: {quad_anomaly_r2:.3f}")
print(f"Quadratic anomaly correlation: {quad_anomaly_corr:.3f}")

## Can we identify unusually warm winters?

In [ ]:
# Calculate county-specific 75th percentile
pdd_threshold = model_df.groupby("fips")[
    "positive_degree_days_2000m"
].transform(lambda x: x.quantile(0.75))

model_df["warm_winter"] = (
    model_df["positive_degree_days_2000m"] >= pdd_threshold
)

print(model_df["warm_winter"].value_counts())

In [ ]:
warm_results = model_df.groupby(
    "warm_winter"
).agg(
    observations=(TARGET, "count"),
    observed_snow_days=(TARGET, "mean"),
    rf_mean_error=("prediction_error", "mean"),
    fe_mean_error=("fe_error", "mean"),
    rf_mae=("prediction_error", lambda x: x.abs().mean()),
    fe_mae=("fe_error", lambda x: x.abs().mean())
)

print(warm_results.round(2))

## 6. Adding XGBoost

In [ ]:
%pip install xgboost

In [ ]:
from xgboost import XGBRegressor
from sklearn.model_selection import LeaveOneGroupOut
import numpy as np

X = model_df[FEATURES]
y = model_df[TARGET]
groups = model_df["year"]

logo = LeaveOneGroupOut()

xgb_predictions = np.full(len(model_df), np.nan)

for train_idx, test_idx in logo.split(X, y, groups):

    xgb = XGBRegressor(
        n_estimators=300,
        max_depth=2,
        learning_rate=0.03,
        subsample=0.8,
        colsample_bytree=0.8,
        min_child_weight=5,
        reg_lambda=10,
        objective="reg:squarederror",
        random_state=42,
        n_jobs=-1
    )

    xgb.fit(
        X.iloc[train_idx],
        y.iloc[train_idx]
    )

    xgb_predictions[test_idx] = xgb.predict(
        X.iloc[test_idx]
    )

model_df["xgb_predicted"] = xgb_predictions

print("XGBoost cross-validation complete.")

In [ ]:
comparison = pd.DataFrame({
    "Random Forest": evaluate_model(
        model_df[TARGET],
        model_df["rf_predicted"]
    ),
    "County FE Linear": evaluate_model(
        model_df[TARGET],
        model_df["fe_predicted"]
    ),
    "County FE Quadratic": evaluate_model(
        model_df[TARGET],
        model_df["quad_predicted"]
    ),
    "XGBoost": evaluate_model(
        model_df[TARGET],
        model_df["xgb_predicted"]
    )
}).T

print(comparison.round(3))

In [ ]:
from sklearn.metrics import r2_score, mean_absolute_error

model_df["xgb_predicted_anomaly"] = (
    model_df["xgb_predicted"] -
    model_df["baseline_predicted"]
)

xgb_anomaly_r2 = r2_score(
    model_df["observed_anomaly"],
    model_df["xgb_predicted_anomaly"]
)

xgb_anomaly_corr = model_df[
    ["observed_anomaly", "xgb_predicted_anomaly"]
].corr().iloc[0, 1]

print(f"XGBoost anomaly R²: {xgb_anomaly_r2:.3f}")
print(f"XGBoost anomaly MAE: {mean_absolute_error(model_df['observed_anomaly'], model_df['xgb_predicted_anomaly']):.3f}")
print(f"XGBoost anomaly correlation: {xgb_anomaly_corr:.3f}")

## 7. Adding Forward Chain

In [ ]:

import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from xgboost import XGBRegressor

folds = [
    (2013, 2014, 2016),
    (2016, 2017, 2019),
    (2019, 2020, 2023)
]

fe_features = [
    "positive_degree_days_2000m",
    "precip_djf_2000m",
    "precip_mar_apr_2000m"
]

forward_df = model_df.copy()

for col in ["forward_rf", "forward_fe", "forward_xgb",
            "forward_baseline"]:
    forward_df[col] = np.nan

for train_end, test_start, test_end in folds:

    train = forward_df["year"] <= train_end
    test = forward_df["year"].between(test_start, test_end)

    X_train = forward_df.loc[train, FEATURES]
    X_test = forward_df.loc[test, FEATURES]

    y_train = forward_df.loc[train, TARGET]

    # Random Forest
    rf = RandomForestRegressor(
        n_estimators=500,
        min_samples_leaf=3,
        max_features=0.7,
        random_state=42,
        n_jobs=-1
    )

    rf.fit(X_train, y_train)
    forward_df.loc[test, "forward_rf"] = rf.predict(X_test)

    # XGBoost
    xgb = XGBRegressor(
        n_estimators=300,
        max_depth=2,
        learning_rate=0.03,
        subsample=0.8,
        colsample_bytree=0.8,
        min_child_weight=5,
        reg_lambda=10,
        objective="reg:squarederror",
        random_state=42,
        n_jobs=-1
    )

    xgb.fit(X_train, y_train)
    forward_df.loc[test, "forward_xgb"] = xgb.predict(X_test)

    # County FE linear model
    preprocessor = ColumnTransformer([
        ("county",
         OneHotEncoder(handle_unknown="ignore"),
         ["fips"]),
        ("climate", "passthrough", fe_features)
    ])

    fe = Pipeline([
        ("preprocessor", preprocessor),
        ("regression", LinearRegression())
    ])

    fe.fit(
        forward_df.loc[train, fe_features + ["fips"]],
        y_train
    )

    forward_df.loc[test, "forward_fe"] = fe.predict(
        forward_df.loc[test, fe_features + ["fips"]]
    )

    # Historical county mean baseline
    county_means = y_train.groupby(
        forward_df.loc[train, "fips"]
    ).mean()

    forward_df.loc[test, "forward_baseline"] = (
        forward_df.loc[test, "fips"]
        .map(county_means)
        .fillna(y_train.mean())
        .to_numpy()
    )

    print(f"Completed {test_start}-{test_end}")


In [ ]:

test_df = forward_df.dropna(
    subset=["forward_rf", "forward_fe",
            "forward_xgb", "forward_baseline"]
).copy()

results = []

for name, col in [
    ("Random Forest", "forward_rf"),
    ("County FE Linear", "forward_fe"),
    ("XGBoost", "forward_xgb"),
    ("County Mean Baseline", "forward_baseline")
]:
    results.append({
        "Model": name,
        "R2": r2_score(test_df[TARGET], test_df[col]),
        "MAE": mean_absolute_error(test_df[TARGET], test_df[col]),
        "RMSE": np.sqrt(mean_squared_error(
            test_df[TARGET], test_df[col]
        ))
    })

forward_results = pd.DataFrame(results).set_index("Model")
print(forward_results.round(3))


In [ ]:

observed_anomaly = (
    test_df[TARGET] - test_df["forward_baseline"]
)

for name, col in [
    ("Random Forest", "forward_rf"),
    ("County FE Linear", "forward_fe"),
    ("XGBoost", "forward_xgb")
]:
    predicted_anomaly = (
        test_df[col] - test_df["forward_baseline"]
    )

    print(
        name,
        "Anomaly R2:",
        round(r2_score(
            observed_anomaly, predicted_anomaly
        ), 3),
        "Correlation:",
        round(observed_anomaly.corr(
            predicted_anomaly
        ), 3)
    )


## Compare LOYO and Forward Chain on same years

In [ ]:

from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    mean_squared_error
)
import numpy as np
import pandas as pd

# Both prediction sets refer to the same model_df rows
comparison_df = forward_df[
    forward_df["year"].between(2014, 2023)
].copy()

models = {
    "Random Forest": ("rf_predicted", "forward_rf"),
    "County FE Linear": ("fe_predicted", "forward_fe"),
    "XGBoost": ("xgb_predicted", "forward_xgb")
}

results = []

for model_name, (loyo_col, forward_col) in models.items():
    for method, pred_col in [
        ("LOYO", loyo_col),
        ("Forward", forward_col)
    ]:
        valid = comparison_df[
            [TARGET, pred_col]
        ].dropna()

        results.append({
            "Model": model_name,
            "Validation": method,
            "R2": r2_score(valid[TARGET], valid[pred_col]),
            "MAE": mean_absolute_error(
                valid[TARGET], valid[pred_col]
            ),
            "RMSE": np.sqrt(mean_squared_error(
                valid[TARGET], valid[pred_col]
            ))
        })

matched_results = pd.DataFrame(results)

print(
    matched_results.round(3).to_string(index=False)
)


In [ ]:

results = []

for model_name, (loyo_col, forward_col) in models.items():
    for method, pred_col, baseline_col in [
        ("LOYO", loyo_col, "baseline_predicted"),
        ("Forward", forward_col, "forward_baseline")
    ]:
        valid = comparison_df[
            [TARGET, pred_col, baseline_col]
        ].dropna()

        observed_anomaly = (
            valid[TARGET] - valid[baseline_col]
        )
        predicted_anomaly = (
            valid[pred_col] - valid[baseline_col]
        )

        results.append({
            "Model": model_name,
            "Validation": method,
            "Anomaly_R2": r2_score(
                observed_anomaly,
                predicted_anomaly
            ),
            "Anomaly_Correlation": observed_anomaly.corr(
                predicted_anomaly
            )
        })

matched_anomalies = pd.DataFrame(results)

print(
    matched_anomalies.round(3).to_string(index=False)
)


## 8. Robustness Checks
Matched Feature Comparison

In [ ]:

import numpy as np
import pandas as pd

from sklearn.model_selection import LeaveOneGroupOut
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    mean_squared_error
)
from xgboost import XGBRegressor

# Same predictors for all three models
matched_features = FEATURES + ["fips"]

# Encode county identifiers
X_matched = pd.get_dummies(
    model_df[matched_features],
    columns=["fips"],
    dtype=float
)

y_matched = model_df[TARGET]
groups = model_df["year"]

print("Observations:", len(X_matched))
print("Predictors:", X_matched.shape[1])
print("Years:", groups.nunique())


In [ ]:

logo = LeaveOneGroupOut()

matched_predictions = {
    "Linear": np.full(len(model_df), np.nan),
    "Random Forest": np.full(len(model_df), np.nan),
    "XGBoost": np.full(len(model_df), np.nan)
}

for train_idx, test_idx in logo.split(
    X_matched, y_matched, groups
):

    models = {
        "Linear": LinearRegression(),

        "Random Forest": RandomForestRegressor(
            n_estimators=500,
            min_samples_leaf=3,
            max_features=0.7,
            random_state=42,
            n_jobs=-1
        ),

        "XGBoost": XGBRegressor(
            n_estimators=300,
            max_depth=2,
            learning_rate=0.03,
            subsample=0.8,
            colsample_bytree=0.8,
            min_child_weight=5,
            reg_lambda=10,
            objective="reg:squarederror",
            random_state=42,
            n_jobs=-1
        )
    }

    for name, model in models.items():

        model.fit(
            X_matched.iloc[train_idx],
            y_matched.iloc[train_idx]
        )

        matched_predictions[name][test_idx] = model.predict(
            X_matched.iloc[test_idx]
        )

for name, preds in matched_predictions.items():
    model_df[f"matched_{name.lower().replace(' ', '_')}"] = preds

print("Matched-feature LOYO validation complete.")


In [ ]:

matched_results = []

for name, preds in matched_predictions.items():

    observed_anomaly = (
        y_matched - model_df["baseline_predicted"]
    )

    predicted_anomaly = (
        preds - model_df["baseline_predicted"]
    )

    matched_results.append({
        "Model": name,
        "R2": r2_score(y_matched, preds),
        "MAE": mean_absolute_error(y_matched, preds),
        "RMSE": np.sqrt(
            mean_squared_error(y_matched, preds)
        ),
        "Anomaly_R2": r2_score(
            observed_anomaly,
            predicted_anomaly
        )
    })

matched_results = pd.DataFrame(
    matched_results
).set_index("Model")

print(matched_results.round(3))


Forward-Chaining w/ matched predictors

In [ ]:

folds = [
    (2013, 2014, 2016),
    (2016, 2017, 2019),
    (2019, 2020, 2023)
]

forward_matched = {
    "Linear": np.full(len(model_df), np.nan),
    "Random Forest": np.full(len(model_df), np.nan),
    "XGBoost": np.full(len(model_df), np.nan)
}

for train_end, test_start, test_end in folds:

    train_idx = np.where(
        model_df["year"] <= train_end
    )[0]

    test_idx = np.where(
        model_df["year"].between(
            test_start, test_end
        )
    )[0]

    models = {
        "Linear": LinearRegression(),

        "Random Forest": RandomForestRegressor(
            n_estimators=500,
            min_samples_leaf=3,
            max_features=0.7,
            random_state=42,
            n_jobs=-1
        ),

        "XGBoost": XGBRegressor(
            n_estimators=300,
            max_depth=2,
            learning_rate=0.03,
            subsample=0.8,
            colsample_bytree=0.8,
            min_child_weight=5,
            reg_lambda=10,
            objective="reg:squarederror",
            random_state=42,
            n_jobs=-1
        )
    }

    for name, model in models.items():

        model.fit(
            X_matched.iloc[train_idx],
            y_matched.iloc[train_idx]
        )

        forward_matched[name][test_idx] = model.predict(
            X_matched.iloc[test_idx]
        )

print("Matched-feature forward validation complete.")


In [ ]:

forward_matched_results = []

test_mask = model_df["year"].between(2014, 2023)

for name, preds in forward_matched.items():

    actual = y_matched.loc[test_mask]
    predicted = preds[test_mask.to_numpy()]

    baseline = forward_df.loc[
        test_mask, "forward_baseline"
    ]

    forward_matched_results.append({
        "Model": name,
        "R2": r2_score(actual, predicted),
        "MAE": mean_absolute_error(actual, predicted),
        "RMSE": np.sqrt(
            mean_squared_error(actual, predicted)
        ),
        "Anomaly_R2": r2_score(
            actual.to_numpy() - baseline.to_numpy(),
            predicted - baseline.to_numpy()
        )
    })

forward_matched_results = pd.DataFrame(
    forward_matched_results
).set_index("Model")

print(forward_matched_results.round(3))


# Spatial Robustness

In [ ]:

from sklearn.model_selection import LeaveOneGroupOut
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import r2_score, mean_absolute_error
import numpy as np
import pandas as pd

X_spatial = model_df[FEATURES]
y_spatial = model_df[TARGET]
county_groups = model_df["fips"]

logo_county = LeaveOneGroupOut()

spatial_predictions = {
    "Linear": np.full(len(model_df), np.nan),
    "Random Forest": np.full(len(model_df), np.nan),
    "XGBoost": np.full(len(model_df), np.nan)
}

for train_idx, test_idx in logo_county.split(
    X_spatial, y_spatial, county_groups
):

    models = {
        "Linear": LinearRegression(),

        "Random Forest": RandomForestRegressor(
            n_estimators=500,
            min_samples_leaf=3,
            max_features=0.7,
            random_state=42,
            n_jobs=-1
        ),

        "XGBoost": XGBRegressor(
            n_estimators=300,
            max_depth=2,
            learning_rate=0.03,
            subsample=0.8,
            colsample_bytree=0.8,
            min_child_weight=5,
            reg_lambda=10,
            objective="reg:squarederror",
            random_state=42,
            n_jobs=-1
        )
    }

    for name, model in models.items():

        model.fit(
            X_spatial.iloc[train_idx],
            y_spatial.iloc[train_idx]
        )

        spatial_predictions[name][test_idx] = model.predict(
            X_spatial.iloc[test_idx]
        )

print("Leave-one-county-out validation complete.")


In [ ]:

spatial_results = []

for name, preds in spatial_predictions.items():

    spatial_results.append({
        "Model": name,
        "R2": r2_score(y_spatial, preds),
        "MAE": mean_absolute_error(y_spatial, preds),
        "RMSE": np.sqrt(
            np.mean((y_spatial.to_numpy() - preds) ** 2)
        )
    })

spatial_results = pd.DataFrame(
    spatial_results
).set_index("Model")

print(spatial_results.round(3))


In [ ]:

county_errors = model_df[
    ["fips", "county", "state", TARGET]
].copy()

for name, preds in spatial_predictions.items():
    county_errors[f"{name}_abs_error"] = np.abs(
        county_errors[TARGET].to_numpy() - preds
    )

county_summary = county_errors.groupby(
    ["state", "county"]
).agg(
    observations=(TARGET, "count"),
    linear_mae=("Linear_abs_error", "mean"),
    rf_mae=("Random Forest_abs_error", "mean"),
    xgb_mae=("XGBoost_abs_error", "mean")
).reset_index()

print(
    county_summary.sort_values(
        "linear_mae", ascending=False
    ).round(2).to_string(index=False)
)


Box Elder Co

In [ ]:
box_elder = model_df[
    (model_df["county"] == "Box Elder")
]

print(
    box_elder[
        [
            "year",
            "snow_days_10cm_2000m",
            "mean_elev_m",
            "frac_above_2000m",
            "positive_degree_days_2000m"
        ]
    ].to_string(index=False)
)

In [ ]:

from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    mean_squared_error
)
import numpy as np
import pandas as pd

# Exclude Box Elder only from evaluation
mask = model_df["county"] != "Box Elder"

results = []

for name, preds in spatial_predictions.items():

    actual = model_df.loc[mask, TARGET]
    predicted = preds[mask.to_numpy()]

    results.append({
        "Model": name,
        "R2": r2_score(actual, predicted),
        "MAE": mean_absolute_error(actual, predicted),
        "RMSE": np.sqrt(
            mean_squared_error(actual, predicted)
        )
    })

sensitivity_results = pd.DataFrame(
    results
).set_index("Model")

print("Spatial validation excluding Box Elder:")
print(sensitivity_results.round(3))


In [ ]:
cols = [
    "year",
    "snow_days_10cm_2000m",
    "valid_days_2000m",
    "valid_fraction_2000m",
    "mean_depth_cm_2000m",
    "peak_depth_cm_2000m"
]

box_elder = model_df[
    model_df["county"] == "Box Elder"
]

print(box_elder[cols].round(2).to_string(index=False))

## 9. Pulling Economic Data

In [ ]:
%pip install requests

In [ ]:

from pathlib import Path
from io import BytesIO
import zipfile
import re
import time

import numpy as np
import pandas as pd
import requests

# ============================================================
# 1. SETTINGS
# ============================================================

START_YEAR = 2004
END_YEAR = 2023

INDUSTRY = "721"  # Accommodation
OWNERSHIP = "5"   # Private ownership

BASE = Path.cwd()
OUTPUT = BASE / "economic_data"
CACHE = OUTPUT / "download_cache"

OUTPUT.mkdir(parents=True, exist_ok=True)
CACHE.mkdir(parents=True, exist_ok=True)

# County FIPS codes for the 20-county snow sample.
# Verify these against model_df before downloading.
COUNTIES = {
    "08019", "08037", "08049", "08051", "08067",
    "08079", "08097", "08107", "08113",
    "16013", "16081",
    "30029", "30031", "30067",
    "49003", "49035", "49043", "49051", "49057",
    "56039"
}

# Prefer the county identifiers from the actual snow dataset.
if "model_df" in globals() and "fips" in model_df.columns:
    actual_counties = set(
        model_df["fips"]
        .astype(str)
        .str.replace(r"\.0$", "", regex=True)
        .str.zfill(5)
    )

    if actual_counties != COUNTIES:
        print("County FIPS list differs from model_df.")
        print("Using model_df FIPS codes instead.")
        print("Only in script:", sorted(COUNTIES - actual_counties))
        print("Only in model_df:", sorted(actual_counties - COUNTIES))
        COUNTIES = actual_counties

print("Counties:", len(COUNTIES))
print("Output folder:", OUTPUT)

# ============================================================
# 2. DOWNLOAD CONFIGURATION
# ============================================================

session = requests.Session()

session.headers.update({
    "User-Agent": "Academic research using public BLS QCEW data"
})

BASE_URL = "https://data.bls.gov/cew/data/files"

all_data = []
download_log = []

# ============================================================
# 3. DOWNLOAD AND PROCESS EACH YEAR
# ============================================================

for year in range(START_YEAR, END_YEAR + 1):

    url = (
        f"{BASE_URL}/{year}/csv/"
        f"{year}_qtrly_by_industry.zip"
    )

    zip_path = CACHE / f"{year}_qtrly_by_industry.zip"

    print(f"\nProcessing {year}...", flush=True)

    try:
        # ----------------------------------------
        # Download once; reuse cached ZIP files
        # ----------------------------------------

        if zip_path.exists():
            print("  Using cached ZIP file")

        else:
            print("  Downloading ZIP file...")

            response = session.get(
                url,
                timeout=(30, 180)
            )

            response.raise_for_status()

            # Verify the file is a valid ZIP before saving.
            with zipfile.ZipFile(BytesIO(response.content)) as test_zip:
                if test_zip.testzip() is not None:
                    raise ValueError("Corrupt ZIP archive")

            zip_path.write_bytes(response.content)

            print(
                f"  Downloaded "
                f"{len(response.content) / 1e6:.1f} MB"
            )

        # ----------------------------------------
        # Find NAICS 721 inside the ZIP
        # ----------------------------------------

        with zipfile.ZipFile(zip_path) as z:

            # Example:
            # 2004.q1-q4 721 Accommodation.csv
            #
            # Match industry code 721 exactly,
            # not 7211, 721110, etc.

            candidates = []

            for name in z.namelist():

                filename = Path(name).name

                if not filename.lower().endswith(".csv"):
                    continue

                if re.search(
                    rf"(?:^|\s){INDUSTRY}\s",
                    filename
                ):
                    candidates.append(name)

            if len(candidates) != 1:
                raise ValueError(
                    f"Expected one NAICS {INDUSTRY} file; "
                    f"found {len(candidates)}: {candidates[:10]}"
                )

            print("  Found:", Path(candidates[0]).name)

            with z.open(candidates[0]) as f:

                df = pd.read_csv(
                    f,
                    dtype={
                        "area_fips": str,
                        "own_code": str,
                        "industry_code": str,
                        "disclosure_code": str
                    },
                    low_memory=False
                )

        # ----------------------------------------
        # Standardize identifiers
        # ----------------------------------------

        for col in [
            "area_fips",
            "own_code",
            "industry_code"
        ]:
            df[col] = df[col].astype(str).str.strip()

        df["area_fips"] = df["area_fips"].str.zfill(5)

        # ----------------------------------------
        # Filter counties, ownership, industry
        # ----------------------------------------

        df = df.loc[
            df["area_fips"].isin(COUNTIES)
            & df["own_code"].eq(OWNERSHIP)
            & df["industry_code"].eq(INDUSTRY)
        ].copy()

        df["qtr"] = pd.to_numeric(
            df["qtr"], errors="coerce"
        )

        df = df.loc[
            df["qtr"].between(1, 4)
        ].copy()

        df["year"] = year

        # ----------------------------------------
        # Handle suppressed observations
        # ----------------------------------------

        df["disclosure_code"] = (
            df["disclosure_code"]
            .fillna("")
            .astype(str)
            .str.strip()
        )

        df["suppressed"] = df["disclosure_code"].eq("N")

        protected_columns = [
            "month1_emplvl",
            "month2_emplvl",
            "month3_emplvl",
            "total_qtrly_wages",
            "avg_wkly_wage"
        ]

        for col in protected_columns:

            df[col] = pd.to_numeric(
                df[col], errors="coerce"
            )

            df.loc[
                df["suppressed"], col
            ] = np.nan

        df["qtrly_estabs"] = pd.to_numeric(
            df["qtrly_estabs"],
            errors="coerce"
        )

        df["quarter_avg_employment"] = df[
            [
                "month1_emplvl",
                "month2_emplvl",
                "month3_emplvl"
            ]
        ].mean(axis=1)

        all_data.append(df)

        download_log.append({
            "year": year,
            "status": "success",
            "records": len(df),
            "counties": df["area_fips"].nunique(),
            "unsuppressed_records": int(
                (~df["suppressed"]).sum()
            )
        })

        print(
            f"  Success: {len(df)} quarterly records "
            f"across {df['area_fips'].nunique()} counties"
        )

    except Exception as e:

        print(f"  FAILED: {e}")

        download_log.append({
            "year": year,
            "status": f"failed: {e}",
            "records": 0,
            "counties": 0,
            "unsuppressed_records": 0
        })

    time.sleep(0.3)

# ============================================================
# 4. SAVE DOWNLOAD LOG
# ============================================================

log = pd.DataFrame(download_log)

log.to_csv(
    OUTPUT / "qcew_download_log.csv",
    index=False
)

print("\nDOWNLOAD SUMMARY")
print(log.to_string(index=False))

if not all_data:
    raise RuntimeError(
        "No data downloaded successfully. "
        "Inspect the download log."
    )

# ============================================================
# 5. BUILD QUARTERLY ECONOMIC PANEL
# ============================================================

panel = pd.concat(
    all_data,
    ignore_index=True
)

panel = panel.sort_values(
    ["area_fips", "year", "qtr"]
).reset_index(drop=True)

# Check for duplicate county-year-quarter observations.
duplicates = panel.duplicated(
    ["area_fips", "year", "qtr"]
).sum()

if duplicates:
    raise ValueError(
        f"Found {duplicates} duplicate county-year-quarter records."
    )

panel_path = (
    OUTPUT / "qcew_721_county_quarterly_2004_2023.csv"
)

panel.to_csv(
    panel_path,
    index=False
)

# ============================================================
# 6. CREATE COVERAGE GRID
# ============================================================

expected_grid = pd.MultiIndex.from_product(
    [
        sorted(COUNTIES),
        range(START_YEAR, END_YEAR + 1),
        range(1, 5)
    ],
    names=["area_fips", "year", "qtr"]
)

coverage = (
    panel.set_index(
        ["area_fips", "year", "qtr"]
    )
    .reindex(expected_grid)
    .reset_index()
)

coverage["record_present"] = (
    coverage["own_code"].notna()
)

coverage["employment_available"] = (
    coverage["quarter_avg_employment"].notna()
)

coverage["wages_available"] = (
    coverage["total_qtrly_wages"].notna()
)

coverage.to_csv(
    OUTPUT / "qcew_721_coverage_grid.csv",
    index=False
)

# ============================================================
# 7. SUMMARIZE COVERAGE BY COUNTY
# ============================================================

county_summary = (
    coverage.groupby("area_fips")
    .agg(
        quarters_expected=("year", "size"),
        quarters_present=("record_present", "sum"),
        employment_quarters=(
            "employment_available", "sum"
        ),
        wage_quarters=(
            "wages_available", "sum"
        )
    )
    .reset_index()
)

county_summary["employment_coverage_pct"] = (
    100
    * county_summary["employment_quarters"]
    / county_summary["quarters_expected"]
)

county_summary.to_csv(
    OUTPUT / "qcew_721_county_coverage_summary.csv",
    index=False
)

# ============================================================
# 8. PRINT FINAL RESULTS
# ============================================================

print("\n" + "=" * 60)
print("FINAL QCEW DOWNLOAD RESULTS")
print("=" * 60)

print("\nTotal quarterly observations:", len(panel))
print("Unique counties:", panel["area_fips"].nunique())
print("Years:", panel["year"].min(), "-", panel["year"].max())

print("\nCounty coverage:")
print(
    county_summary.round(1).to_string(index=False)
)

print("\nFiles saved to:")
print(OUTPUT)

print("\nMain panel:")
print(panel_path)


In [ ]:
import zipfile
import pandas as pd
from pathlib import Path

zip_path = (
    Path.cwd()
    / "economic_data"
    / "download_cache"
    / "2004_qtrly_by_industry.zip"
)

with zipfile.ZipFile(zip_path) as z:
    name = next(
        n for n in z.namelist()
        if "721 Accommodation.csv" in n
    )

    with z.open(name) as f:
        sample = pd.read_csv(f, nrows=5)

print(sample.columns.tolist())

In [ ]:

from pathlib import Path
from io import BytesIO
import zipfile
import re
import time

import numpy as np
import pandas as pd
import requests

# ============================================================
# 1. SETTINGS
# ============================================================

# Set TEST_MODE = True to process only 2004–2005.
# Once successful, change to False for 2004–2023.
TEST_MODE = False

START_YEAR = 2004
END_YEAR = 2005 if TEST_MODE else 2023

INDUSTRY = "721"  # Accommodation
OWNERSHIP = "5"   # Private sector

BASE = Path.cwd()
OUTPUT = BASE / "economic_data"
CACHE = OUTPUT / "download_cache"

OUTPUT.mkdir(parents=True, exist_ok=True)
CACHE.mkdir(parents=True, exist_ok=True)

# Use actual county FIPS from your snow model.
if "model_df" not in globals():
    raise NameError(
        "model_df is not defined. Run your snow-model "
        "data preparation cells first."
    )

COUNTIES = set(
    model_df["fips"]
    .astype(str)
    .str.replace(r"\.0$", "", regex=True)
    .str.zfill(5)
)

print("Counties:", len(COUNTIES))
print("Years:", START_YEAR, "-", END_YEAR)
print("Saving to:", OUTPUT)

if len(COUNTIES) != 20:
    print(
        "WARNING: Expected 20 counties, "
        f"but found {len(COUNTIES)}."
    )

# ============================================================
# 2. DOWNLOAD CONFIGURATION
# ============================================================

session = requests.Session()
session.headers.update({
    "User-Agent": "Academic research using public BLS QCEW data"
})

BASE_URL = "https://data.bls.gov/cew/data/files"

all_data = []
download_log = []

# ============================================================
# 3. DOWNLOAD AND PROCESS YEARS
# ============================================================

for year in range(START_YEAR, END_YEAR + 1):

    url = (
        f"{BASE_URL}/{year}/csv/"
        f"{year}_qtrly_by_industry.zip"
    )

    zip_path = CACHE / f"{year}_qtrly_by_industry.zip"

    print(f"\nProcessing {year}...", flush=True)

    try:

        # ----------------------------------------
        # Download or reuse cached archive
        # ----------------------------------------

        if zip_path.exists():
            print("  Using cached ZIP file")

        else:
            print("  Downloading ZIP file...")

            response = session.get(
                url,
                timeout=(30, 180)
            )
            response.raise_for_status()

            with zipfile.ZipFile(BytesIO(response.content)) as z:
                bad_file = z.testzip()
                if bad_file is not None:
                    raise ValueError(
                        f"Corrupt ZIP member: {bad_file}"
                    )

            zip_path.write_bytes(response.content)

            print(
                f"  Downloaded "
                f"{len(response.content) / 1e6:.1f} MB"
            )

        # ----------------------------------------
        # Locate NAICS 721 CSV
        # ----------------------------------------

        with zipfile.ZipFile(zip_path) as z:

            candidates = [
                name
                for name in z.namelist()
                if name.lower().endswith(".csv")
                and re.search(
                    rf"(?:^|\s){INDUSTRY}\s",
                    Path(name).name
                )
            ]

            if len(candidates) != 1:
                raise ValueError(
                    f"Expected one NAICS {INDUSTRY} CSV, "
                    f"found {len(candidates)}: "
                    f"{candidates[:10]}"
                )

            print("  Found:", Path(candidates[0]).name)

            with z.open(candidates[0]) as f:

                df = pd.read_csv(
                    f,
                    dtype={
                        "area_fips": str,
                        "own_code": str,
                        "industry_code": str,
                        "disclosure_code": str
                    },
                    low_memory=False
                )

        # ----------------------------------------
        # Standardize identifiers
        # ----------------------------------------

        for col in [
            "area_fips",
            "own_code",
            "industry_code"
        ]:
            df[col] = df[col].astype(str).str.strip()

        df["area_fips"] = (
            df["area_fips"].str.zfill(5)
        )

        df["qtr"] = pd.to_numeric(
            df["qtr"],
            errors="coerce"
        )

        # ----------------------------------------
        # Filter to our 20 counties
        # ----------------------------------------

        df = df.loc[
            df["area_fips"].isin(COUNTIES)
            & df["own_code"].eq(OWNERSHIP)
            & df["industry_code"].eq(INDUSTRY)
            & df["qtr"].between(1, 4)
        ].copy()

        df["year"] = year

        # ----------------------------------------
        # Suppression handling
        # ----------------------------------------

        df["disclosure_code"] = (
            df["disclosure_code"]
            .fillna("")
            .astype(str)
            .str.strip()
        )

        df["suppressed"] = (
            df["disclosure_code"].eq("N")
        )

        protected_columns = [
            "month1_emplvl",
            "month2_emplvl",
            "month3_emplvl",
            "total_qtrly_wages",
            "avg_wkly_wage"
        ]

        for col in protected_columns:

            df[col] = pd.to_numeric(
                df[col],
                errors="coerce"
            )

            df.loc[
                df["suppressed"], col
            ] = np.nan

        # Correct BLS establishment column name
        df["qtrly_estabs"] = pd.to_numeric(
            df["qtrly_estabs_count"],
            errors="coerce"
        )

        # Calculate average quarterly employment
        df["quarter_avg_employment"] = df[
            [
                "month1_emplvl",
                "month2_emplvl",
                "month3_emplvl"
            ]
        ].mean(axis=1)

        all_data.append(df)

        download_log.append({
            "year": year,
            "status": "success",
            "records": len(df),
            "counties": df["area_fips"].nunique(),
            "unsuppressed_records": int(
                (~df["suppressed"]).sum()
            )
        })

        print(
            f"  SUCCESS: {len(df)} quarterly records "
            f"across {df['area_fips'].nunique()} counties"
        )

    except Exception as e:

        print(f"  FAILED: {e}")

        download_log.append({
            "year": year,
            "status": f"failed: {e}",
            "records": 0,
            "counties": 0,
            "unsuppressed_records": 0
        })

    time.sleep(0.2)

# ============================================================
# 4. SAVE DOWNLOAD LOG
# ============================================================

log = pd.DataFrame(download_log)

log.to_csv(
    OUTPUT / "qcew_download_log.csv",
    index=False
)

print("\nDOWNLOAD SUMMARY")
print(log.to_string(index=False))

if not all_data:
    raise RuntimeError(
        "No data downloaded successfully. "
        "Check the download log."
    )

# ============================================================
# 5. BUILD QUARTERLY PANEL
# ============================================================

panel = pd.concat(
    all_data,
    ignore_index=True
)

panel = panel.sort_values(
    ["area_fips", "year", "qtr"]
).reset_index(drop=True)

# Ensure unique county-year-quarter observations
duplicates = panel.duplicated(
    ["area_fips", "year", "qtr"]
).sum()

if duplicates:
    raise ValueError(
        f"Found {duplicates} duplicate observations."
    )

panel_path = OUTPUT / (
    f"qcew_721_county_quarterly_"
    f"{START_YEAR}_{END_YEAR}.csv"
)

panel.to_csv(
    panel_path,
    index=False
)

# ============================================================
# 6. CREATE COVERAGE GRID
# ============================================================

expected_grid = pd.MultiIndex.from_product(
    [
        sorted(COUNTIES),
        range(START_YEAR, END_YEAR + 1),
        range(1, 5)
    ],
    names=["area_fips", "year", "qtr"]
)

coverage = (
    panel.set_index(
        ["area_fips", "year", "qtr"]
    )
    .reindex(expected_grid)
    .reset_index()
)

coverage["record_present"] = (
    coverage["own_code"].notna()
)

coverage["employment_available"] = (
    coverage["quarter_avg_employment"].notna()
)

coverage["wages_available"] = (
    coverage["total_qtrly_wages"].notna()
)

coverage.to_csv(
    OUTPUT / "qcew_721_coverage_grid.csv",
    index=False
)

# ============================================================
# 7. COUNTY COVERAGE SUMMARY
# ============================================================

county_summary = (
    coverage.groupby("area_fips")
    .agg(
        quarters_expected=("year", "size"),
        quarters_present=("record_present", "sum"),
        employment_quarters=(
            "employment_available", "sum"
        ),
        wage_quarters=(
            "wages_available", "sum"
        )
    )
    .reset_index()
)

county_summary["employment_coverage_pct"] = (
    100
    * county_summary["employment_quarters"]
    / county_summary["quarters_expected"]
)

county_summary.to_csv(
    OUTPUT / "qcew_721_county_coverage_summary.csv",
    index=False
)

# ============================================================
# 8. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 60)
print("FINAL QCEW DOWNLOAD RESULTS")
print("=" * 60)

print("\nTotal quarterly records:", len(panel))
print("Unique counties:", panel["area_fips"].nunique())
print("Years:", panel["year"].min(), "-", panel["year"].max())

print("\nCounty coverage:")
print(
    county_summary.round(1).to_string(index=False)
)

print("\nFiles saved to:")
print(OUTPUT)

print("\nMain panel:")
print(panel_path)


In [ ]:
failed_years = log.loc[
    log["status"] != "success", "year"
].tolist()

print("Years that need to be downloaded again:", failed_years)

In [ ]:
from pathlib import Path
import zipfile

CACHE = Path.cwd() / "economic_data" / "download_cache"

available = []
missing = []

for year in range(2004, 2024):
    path = CACHE / f"{year}_qtrly_by_industry.zip"

    if path.exists() and zipfile.is_zipfile(path):
        available.append(year)
    else:
        missing.append(year)

print("Available years:", available)
print("Missing years:", missing)
print(f"\nDownloaded {len(available)} of 20 years")

In [ ]:

import requests
import pandas as pd
from io import BytesIO

results = []

for year in range(2018, 2024):
    url = f"https://data.bls.gov/cew/data/api/{year}/1/industry/721.csv"

    try:
        response = requests.get(url, timeout=30)
        response.raise_for_status()

        sample = pd.read_csv(BytesIO(response.content), nrows=3)

        valid = {"area_fips", "industry_code", "qtr"}.issubset(sample.columns)

        results.append({
            "year": year,
            "available": valid,
            "size_mb": round(len(response.content) / 1e6, 2)
        })

    except Exception as e:
        results.append({
            "year": year,
            "available": False,
            "error": str(e)[:100]
        })

print(pd.DataFrame(results).to_string(index=False))


In [ ]:

from pathlib import Path
from io import BytesIO
import zipfile
import re
import time

import numpy as np
import pandas as pd
import requests

# ============================================================
# 1. SETTINGS
# ============================================================

START_YEAR = 2004
END_YEAR = 2023

INDUSTRY = "721"   # NAICS 721: Accommodation
OWNERSHIP = "5"    # Private sector

BASE = Path.cwd()
OUTPUT = BASE / "economic_data"
CACHE = OUTPUT / "download_cache"
API_CACHE = CACHE / "api_721"

OUTPUT.mkdir(parents=True, exist_ok=True)
CACHE.mkdir(parents=True, exist_ok=True)
API_CACHE.mkdir(parents=True, exist_ok=True)

# Use the county FIPS codes from the SNODAS model dataset.
if "model_df" not in globals():
    raise NameError(
        "model_df is not defined. Run the snow-model "
        "data preparation cells first."
    )

COUNTIES = set(
    model_df["fips"]
    .dropna()
    .astype(str)
    .str.strip()
    .str.replace(r"\.0$", "", regex=True)
    .str.zfill(5)
)

if len(COUNTIES) != 20:
    raise ValueError(
        f"Expected 20 counties, found {len(COUNTIES)}. "
        "Check model_df['fips']."
    )

print("=" * 60)
print("BLS QCEW ACCOMMODATION DATA")
print("=" * 60)
print("Counties:", len(COUNTIES))
print("Years:", START_YEAR, "-", END_YEAR)
print("Industry:", INDUSTRY)
print("Ownership:", OWNERSHIP)
print("Output:", OUTPUT)

session = requests.Session()
session.headers.update({
    "User-Agent": "Academic research - public BLS QCEW data"
})

all_data = []
download_log = []

# ============================================================
# 2. CLEAN AND STANDARDIZE QCEW DATA
# ============================================================

def clean_qcew(df):

    df = df.copy()

    # ----------------------------------------
    # Standardize ZIP vs API column names
    # ----------------------------------------

    # ZIP files: qtrly_estabs_count
    # API files: qtrly_estabs
    if "qtrly_estabs_count" in df.columns:
        df = df.rename(columns={
            "qtrly_estabs_count": "qtrly_estabs"
        })

    # ----------------------------------------
    # Standardize identifiers
    # ----------------------------------------

    required_columns = [
        "area_fips",
        "own_code",
        "industry_code",
        "year",
        "qtr",
        "disclosure_code",
        "qtrly_estabs",
        "month1_emplvl",
        "month2_emplvl",
        "month3_emplvl",
        "total_qtrly_wages",
        "avg_wkly_wage"
    ]

    missing_columns = [
        c for c in required_columns
        if c not in df.columns
    ]

    if missing_columns:
        raise ValueError(
            f"Missing required columns: {missing_columns}"
        )

    for col in ["area_fips", "own_code", "industry_code"]:
        df[col] = (
            df[col]
            .astype(str)
            .str.strip()
            .str.replace(r"\.0$", "", regex=True)
        )

    df["area_fips"] = df["area_fips"].str.zfill(5)

    df["year"] = pd.to_numeric(
        df["year"], errors="coerce"
    )

    df["qtr"] = pd.to_numeric(
        df["qtr"], errors="coerce"
    )

    # ----------------------------------------
    # Filter to study counties
    # ----------------------------------------

    df = df.loc[
        df["area_fips"].isin(COUNTIES)
        & df["own_code"].eq(OWNERSHIP)
        & df["industry_code"].eq(INDUSTRY)
        & df["qtr"].between(1, 4)
    ].copy()

    # ----------------------------------------
    # Disclosure suppression
    # ----------------------------------------

    df["disclosure_code"] = (
        df["disclosure_code"]
        .fillna("")
        .astype(str)
        .str.strip()
    )

    df["suppressed"] = (
        df["disclosure_code"].eq("N")
    )

    numeric_columns = [
        "qtrly_estabs",
        "month1_emplvl",
        "month2_emplvl",
        "month3_emplvl",
        "total_qtrly_wages",
        "avg_wkly_wage"
    ]

    for col in numeric_columns:
        df[col] = pd.to_numeric(
            df[col], errors="coerce"
        )

    protected_columns = [
        "month1_emplvl",
        "month2_emplvl",
        "month3_emplvl",
        "total_qtrly_wages",
        "avg_wkly_wage"
    ]

    df.loc[
        df["suppressed"],
        protected_columns
    ] = np.nan

    # ----------------------------------------
    # Quarterly average employment
    # ----------------------------------------

    # Require all three monthly employment values.
    df["quarter_avg_employment"] = (
        df[
            [
                "month1_emplvl",
                "month2_emplvl",
                "month3_emplvl"
            ]
        ].mean(axis=1, skipna=False)
    )

    return df

# ============================================================
# 3. READ CACHED ANNUAL ZIP FILES: 2004–2017
# ============================================================

print("\n" + "=" * 60)
print("READING CACHED DATA: 2004–2017")
print("=" * 60)

for year in range(2004, 2018):

    zip_path = CACHE / f"{year}_qtrly_by_industry.zip"

    try:

        if not zip_path.exists():
            raise FileNotFoundError(
                f"Missing cached ZIP: {zip_path}"
            )

        with zipfile.ZipFile(zip_path) as z:

            candidates = [
                name
                for name in z.namelist()
                if name.lower().endswith(".csv")
                and re.search(
                    rf"(?:^|\s){INDUSTRY}\s",
                    Path(name).name
                )
            ]

            if len(candidates) != 1:
                raise ValueError(
                    f"Expected one NAICS {INDUSTRY} CSV; "
                    f"found {len(candidates)}"
                )

            with z.open(candidates[0]) as f:

                df = pd.read_csv(
                    f,
                    dtype={
                        "area_fips": str,
                        "own_code": str,
                        "industry_code": str,
                        "disclosure_code": str
                    },
                    low_memory=False
                )

        df = clean_qcew(df)

        # Confirm that the file contains only this year.
        if not df["year"].eq(year).all():
            raise ValueError(
                f"Unexpected year values in {year} ZIP"
            )

        all_data.append(df)

        download_log.append({
            "year": year,
            "source": "cached ZIP",
            "quarters_loaded": df["qtr"].nunique(),
            "records": len(df),
            "status": "success"
        })

        print(
            f"{year}: {len(df)} records, "
            f"{df['area_fips'].nunique()} counties"
        )

    except Exception as e:

        print(f"{year}: FAILED - {e}")

        download_log.append({
            "year": year,
            "source": "cached ZIP",
            "quarters_loaded": 0,
            "records": 0,
            "status": f"failed: {e}"
        })

# ============================================================
# 4. DOWNLOAD API DATA: 2018–2023
# ============================================================

print("\n" + "=" * 60)
print("DOWNLOADING BLS API DATA: 2018–2023")
print("=" * 60)

for year in range(2018, 2024):

    for quarter in range(1, 5):

        url = (
            f"https://data.bls.gov/cew/data/api/"
            f"{year}/{quarter}/industry/{INDUSTRY}.csv"
        )

        cache_path = (
            API_CACHE / f"{year}_q{quarter}_721.csv"
        )

        success = False
        last_error = None

        for attempt in range(1, 4):

            try:

                # Use local API cache if available.
                if cache_path.exists():
                    df = pd.read_csv(
                        cache_path,
                        dtype={
                            "area_fips": str,
                            "own_code": str,
                            "industry_code": str,
                            "disclosure_code": str
                        },
                        low_memory=False
                    )

                else:

                    response = session.get(
                        url,
                        timeout=(30, 120)
                    )

                    response.raise_for_status()

                    df = pd.read_csv(
                        BytesIO(response.content),
                        dtype={
                            "area_fips": str,
                            "own_code": str,
                            "industry_code": str,
                            "disclosure_code": str
                        },
                        low_memory=False
                    )

                df = clean_qcew(df)

                # Validate year and quarter.
                if not (
                    df["year"].eq(year).all()
                    and df["qtr"].eq(quarter).all()
                ):
                    raise ValueError(
                        "Unexpected year or quarter"
                    )

                if df["area_fips"].nunique() != 20:
                    raise ValueError(
                        f"Expected 20 counties, found "
                        f"{df['area_fips'].nunique()}"
                    )

                # Cache only after successful validation.
                # Cache the original API response, not the
                # cleaned 20-county subset.
                if not cache_path.exists():
                    cache_path.write_bytes(response.content)

                all_data.append(df)

                download_log.append({
                    "year": year,
                    "quarter": quarter,
                    "source": "BLS API",
                    "quarters_loaded": 1,
                    "records": len(df),
                    "status": "success"
                })

                print(
                    f"{year} Q{quarter}: "
                    f"{len(df)} records"
                )

                success = True
                break

            except Exception as e:

                last_error = str(e)

                print(
                    f"{year} Q{quarter}, "
                    f"attempt {attempt}: {e}"
                )

                if attempt < 3:
                    time.sleep(3 * attempt)

        if not success:

            download_log.append({
                "year": year,
                "quarter": quarter,
                "source": "BLS API",
                "quarters_loaded": 0,
                "records": 0,
                "status": f"failed: {last_error}"
            })

# ============================================================
# 5. COMBINE ECONOMIC PANEL
# ============================================================

print("\n" + "=" * 60)
print("BUILDING ECONOMIC PANEL")
print("=" * 60)

if not all_data:
    raise RuntimeError("No QCEW data loaded.")

panel = pd.concat(
    all_data,
    ignore_index=True
)

panel = panel.sort_values(
    ["area_fips", "year", "qtr"]
).reset_index(drop=True)

# Check uniqueness.
duplicate_count = panel.duplicated(
    ["area_fips", "year", "qtr"]
).sum()

if duplicate_count:
    raise ValueError(
        f"Found {duplicate_count} duplicate "
        "county-year-quarter observations."
    )

# ============================================================
# 6. BUILD COMPLETE COVERAGE GRID
# ============================================================

expected_grid = pd.MultiIndex.from_product(
    [
        sorted(COUNTIES),
        range(START_YEAR, END_YEAR + 1),
        range(1, 5)
    ],
    names=["area_fips", "year", "qtr"]
)

coverage = (
    panel.set_index(
        ["area_fips", "year", "qtr"]
    )
    .reindex(expected_grid)
    .reset_index()
)

coverage["record_present"] = (
    coverage["own_code"].notna()
)

coverage["employment_available"] = (
    coverage["quarter_avg_employment"].notna()
)

coverage["wages_available"] = (
    coverage["total_qtrly_wages"].notna()
)

coverage["suppressed"] = (
    coverage["suppressed"].fillna(False)
)

# ============================================================
# 7. COUNTY COVERAGE SUMMARY
# ============================================================

county_summary = (
    coverage.groupby("area_fips")
    .agg(
        quarters_expected=("year", "size"),
        quarters_present=("record_present", "sum"),
        employment_quarters=(
            "employment_available", "sum"
        ),
        wage_quarters=(
            "wages_available", "sum"
        ),
        suppressed_quarters=(
            "suppressed", "sum"
        )
    )
    .reset_index()
)

county_summary["employment_coverage_pct"] = (
    100
    * county_summary["employment_quarters"]
    / county_summary["quarters_expected"]
)

# ============================================================
# 8. VERIFY PANEL COMPLETENESS
# ============================================================

expected_records = (
    len(COUNTIES)
    * (END_YEAR - START_YEAR + 1)
    * 4
)

missing_records = (
    expected_records - len(panel)
)

print("\nExpected records:", expected_records)
print("Actual records:", len(panel))
print("Missing records:", missing_records)

print(
    "Suppressed observations:",
    int(panel["suppressed"].sum())
)

print(
    "Missing employment:",
    int(panel["quarter_avg_employment"].isna().sum())
)

print(
    "Missing wages:",
    int(panel["total_qtrly_wages"].isna().sum())
)

# ============================================================
# 9. SAVE RESULTS
# ============================================================

log = pd.DataFrame(download_log)

log.to_csv(
    OUTPUT / "qcew_download_log.csv",
    index=False
)

coverage.to_csv(
    OUTPUT / "qcew_721_coverage_grid.csv",
    index=False
)

county_summary.to_csv(
    OUTPUT / "qcew_721_county_coverage_summary.csv",
    index=False
)

if missing_records == 0:

    panel_path = (
        OUTPUT /
        "qcew_721_county_quarterly_2004_2023.csv"
    )

    panel.to_csv(
        panel_path,
        index=False
    )

    print("\nCOMPLETE PANEL SAVED:")
    print(panel_path)

else:

    partial_path = (
        OUTPUT /
        "qcew_721_county_quarterly_PARTIAL.csv"
    )

    panel.to_csv(
        partial_path,
        index=False
    )

    print("\nWARNING: Panel is incomplete.")
    print("Partial panel saved to:", partial_path)
    print("The complete-panel filename was not overwritten.")

# ============================================================
# 10. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 60)
print("FINAL ECONOMIC PANEL RESULTS")
print("=" * 60)

print("\nTotal records:", len(panel))
print("Counties:", panel["area_fips"].nunique())
print(
    "Years:",
    int(panel["year"].min()),
    "-",
    int(panel["year"].max())
)

print("\nDownload failures:")
failures = log.loc[
    log["status"] != "success"
]

if failures.empty:
    print("None")
else:
    print(failures.to_string(index=False))

print("\nCounty coverage:")
print(
    county_summary.round(1).to_string(index=False)
)

print("\nFiles saved to:")
print(OUTPUT)


# Winter Economic Panel

In [ ]:

import numpy as np
import pandas as pd
from pathlib import Path

# Start with the completed quarterly BLS panel
qcew = panel.copy()

# Assign each quarter to the winter ending year
qcew["winter_year"] = np.where(
    qcew["qtr"] == 4,
    qcew["year"] + 1,
    qcew["year"]
)

# Keep Q4 (previous year) and Q1 (current year)
winter_qcew = qcew[
    qcew["qtr"].isin([1, 4])
].copy()

# Retain only winters with both quarters available
# and within our historical data coverage
winter_qcew = winter_qcew[
    winter_qcew["winter_year"].between(2005, 2023)
]

# Summarize by county and winter year
winter_econ = (
    winter_qcew
    .groupby(["area_fips", "winter_year"])
    .agg(
        quarters_present=("qtr", "size"),
        quarters_with_employment=(
            "quarter_avg_employment",
            "count"
        ),
        quarters_with_wages=(
            "total_qtrly_wages",
            "count"
        ),
        winter_employment=(
            "quarter_avg_employment",
            "mean"
        ),
        winter_wages=(
            "total_qtrly_wages",
            lambda x: x.sum(min_count=2)
        )
    )
    .reset_index()
)

# Do not calculate winter employment if either
# quarter has suppressed employment.
winter_econ.loc[
    winter_econ["quarters_with_employment"] < 2,
    "winter_employment"
] = np.nan

winter_econ.loc[
    winter_econ["quarters_with_wages"] < 2,
    "winter_wages"
] = np.nan

# Log outcomes for economic regression
winter_econ["log_winter_employment"] = np.where(
    winter_econ["winter_employment"] > 0,
    np.log(
        winter_econ["winter_employment"].clip(lower=1)
    ),
    np.nan
)

winter_econ["log_winter_wages"] = np.where(
    winter_econ["winter_wages"] > 0,
    np.log(
        winter_econ["winter_wages"].clip(lower=1)
    ),
    np.nan
)

# Save
output_path = (
    Path.cwd()
    / "economic_data"
    / "qcew_721_winter_county_panel_2005_2023.csv"
)

winter_econ.to_csv(output_path, index=False)

print("Winter economic observations:", len(winter_econ))
print("Counties:", winter_econ["area_fips"].nunique())
print("Years:", winter_econ["winter_year"].min(),
      "-", winter_econ["winter_year"].max())

print("\nMissing winter employment:",
      winter_econ["winter_employment"].isna().sum())

print("Missing winter wages:",
      winter_econ["winter_wages"].isna().sum())

print("\nPreview:")
print(winter_econ.head(10).to_string(index=False))


In [ ]:
# Inspect snow panel structure
print("Snow panel columns:")
print(model_df.columns.tolist())

print("\nSnow observations:")
print(model_df[
    ["fips", "year", "snow_days_10cm_2000m"]
].head(10).to_string(index=False))

# Check economic coverage by county
print("\nWinter economic coverage by county:")
print(
    winter_econ.groupby("area_fips")
    .agg(
        winters=("winter_year", "size"),
        usable=("winter_employment", "count")
    )
    .assign(
        coverage_pct=lambda x: (
            100 * x["usable"] / x["winters"]
        ).round(1)
    )
    .to_string()
)

Merge SNODAS w/ BLS Data

In [ ]:

import pandas as pd
import numpy as np

# Prepare snow dataset
snow_econ = model_df.copy()

snow_econ["fips"] = (
    snow_econ["fips"]
    .astype(str)
    .str.replace(r"\.0$", "", regex=True)
    .str.zfill(5)
)

snow_econ["year"] = pd.to_numeric(
    snow_econ["year"], errors="coerce"
)

# Prepare economic dataset
econ = winter_econ.copy()

econ["area_fips"] = (
    econ["area_fips"]
    .astype(str)
    .str.replace(r"\.0$", "", regex=True)
    .str.zfill(5)
)

# Merge county and winter year
merged = snow_econ.merge(
    econ,
    left_on=["fips", "year"],
    right_on=["area_fips", "winter_year"],
    how="inner",
    validate="one_to_one"
)

# Main economic sample: exclude Clear Creek
econ_model = merged.loc[
    (merged["fips"] != "08019")
    & merged["winter_employment"].notna()
    & merged["snow_days_10cm_2000m"].notna()
].copy()

print("Merged county-winters:", len(merged))
print("Main regression observations:", len(econ_model))
print("Counties:", econ_model["fips"].nunique())
print("Years:", econ_model["year"].min(),
      "-", econ_model["year"].max())

print("\nEconomic outcomes:")
print(
    econ_model[
        [
            "winter_employment",
            "winter_wages",
            "snow_days_10cm_2000m"
        ]
    ].describe().round(2)
)


In [ ]:

import statsmodels.formula.api as smf

# County and winter-year fixed effects
fe_employment = smf.ols(
    formula=(
        "log_winter_employment ~ "
        "snow_days_10cm_2000m + "
        "C(fips) + C(year)"
    ),
    data=econ_model
).fit(
    cov_type="cluster",
    cov_kwds={
        "groups": econ_model["fips"]
    }
)

print(fe_employment.summary())

# Interpret a 10-day increase in snow cover
beta = fe_employment.params["snow_days_10cm_2000m"]

pct_change = 100 * (
    np.exp(beta * 10) - 1
)

print(
    f"\nEstimated employment difference "
    f"associated with 10 additional snow days: "
    f"{pct_change:.2f}%"
)


In [ ]:

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

specifications = {
    "1. Employment, 10cm": {
        "outcome": "log_winter_employment",
        "snow": "snow_days_10cm_2000m",
        "exclude_covid": False
    },
    "2. Wages, 10cm": {
        "outcome": "log_winter_wages",
        "snow": "snow_days_10cm_2000m",
        "exclude_covid": False
    },
    "3. Employment, 5cm": {
        "outcome": "log_winter_employment",
        "snow": "snow_days_5cm_2000m",
        "exclude_covid": False
    },
    "4. Employment, depth": {
        "outcome": "log_winter_employment",
        "snow": "mean_depth_cm_2000m",
        "exclude_covid": False
    },
    "5. Employment, no COVID": {
        "outcome": "log_winter_employment",
        "snow": "snow_days_10cm_2000m",
        "exclude_covid": True
    }
}

results = []
fitted_models = {}

for name, spec in specifications.items():

    df = econ_model.copy()

    if spec["exclude_covid"]:
        df = df[
            ~df["year"].isin([2020, 2021])
        ].copy()

    outcome = spec["outcome"]
    snow = spec["snow"]

    df = df.dropna(
        subset=[outcome, snow, "fips", "year"]
    ).copy()

    formula = (
        f"{outcome} ~ {snow} "
        "+ C(fips) + C(year)"
    )

    model = smf.ols(
        formula=formula,
        data=df
    ).fit(
        cov_type="cluster",
        cov_kwds={
            "groups": df["fips"]
        }
    )

    fitted_models[name] = model

    beta = model.params[snow]
    se = model.bse[snow]
    pval = model.pvalues[snow]

    # Interpret a 10-unit increase:
    # 10 days for duration, 10 cm for depth.
    pct_change = 100 * (
        np.exp(beta * 10) - 1
    )

    results.append({
        "model": name,
        "coefficient": beta,
        "clustered_se": se,
        "p_value": pval,
        "effect_10_units_pct": pct_change,
        "observations": int(model.nobs),
        "counties": df["fips"].nunique(),
        "r_squared": model.rsquared
    })

comparison = pd.DataFrame(results)

print(
    comparison.round(4).to_string(index=False)
)


In [ ]:

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

df = econ_model.copy()

snow_col = "snow_days_10cm_2000m"

# County-specific average snow duration
df["county_mean_snow"] = (
    df.groupby("fips")[snow_col]
    .transform("mean")
)

# Snow anomaly relative to county mean
df["snow_anomaly"] = (
    df[snow_col] - df["county_mean_snow"]
)

# Separate negative and positive anomalies
df["negative_snow_anomaly"] = (
    df["snow_anomaly"].clip(upper=0)
)

df["positive_snow_anomaly"] = (
    df["snow_anomaly"].clip(lower=0)
)

# Estimate asymmetric fixed-effects regression
asymmetric_model = smf.ols(
    formula=(
        "log_winter_employment ~ "
        "negative_snow_anomaly + "
        "positive_snow_anomaly + "
        "C(fips) + C(year)"
    ),
    data=df
).fit(
    cov_type="cluster",
    cov_kwds={
        "groups": df["fips"]
    }
)

print("\nASYMMETRIC SNOW EFFECTS")
print("=" * 55)

for variable in [
    "negative_snow_anomaly",
    "positive_snow_anomaly"
]:
    beta = asymmetric_model.params[variable]
    se = asymmetric_model.bse[variable]
    p = asymmetric_model.pvalues[variable]

    print(f"\n{variable}")
    print(f"Coefficient: {beta:.5f}")
    print(f"Clustered SE: {se:.5f}")
    print(f"P-value: {p:.4f}")

# Wald test: are the two slopes equal?
test = asymmetric_model.t_test(
    "negative_snow_anomaly = positive_snow_anomaly"
)

print("\nTEST OF EQUAL SLOPES")
print(test)

print("\nObservations:", int(asymmetric_model.nobs))
print("R-squared:", round(asymmetric_model.rsquared, 4))


In [ ]:
start = f"{year-1}-10-01"
end = f"{year}-04-30"

In [ ]:

import numpy as np
import pandas as pd
from pathlib import Path

# =====================================================
# 1. LOAD QUARTERLY ECONOMIC DATA
# =====================================================

output_dir = Path.cwd() / "economic_data"

qcew = pd.read_csv(
    output_dir / "qcew_721_county_quarterly_2004_2023.csv",
    dtype={"area_fips": str}
)

qcew["area_fips"] = qcew["area_fips"].str.zfill(5)
qcew["year"] = pd.to_numeric(qcew["year"])
qcew["qtr"] = pd.to_numeric(qcew["qtr"])

# =====================================================
# 2. SUMMER EMPLOYMENT
# Q3 of year t-1 corresponds to winter year t
# =====================================================

summer = qcew.loc[qcew["qtr"] == 3, [
    "area_fips",
    "year",
    "quarter_avg_employment"
]].copy()

summer["winter_year"] = summer["year"] + 1

summer = summer.rename(columns={
    "quarter_avg_employment": "summer_employment"
})

summer = summer.drop(columns="year")

# =====================================================
# 3. WINTER EMPLOYMENT
# Q4 of t-1 plus Q1 of t
# =====================================================

winter = qcew[qcew["qtr"].isin([1, 4])].copy()

winter["winter_year"] = np.where(
    winter["qtr"] == 4,
    winter["year"] + 1,
    winter["year"]
)

winter = winter[
    winter["winter_year"].between(2005, 2023)
].copy()

winter_summary = (
    winter.groupby(["area_fips", "winter_year"])
    .agg(
        winter_quarters=(
            "quarter_avg_employment", "count"
        ),
        winter_employment=(
            "quarter_avg_employment", "mean"
        )
    )
    .reset_index()
)

# Require both winter quarters
winter_summary.loc[
    winter_summary["winter_quarters"] != 2,
    "winter_employment"
] = np.nan

# =====================================================
# 4. MERGE WINTER AND SUMMER
# =====================================================

seasonal = winter_summary.merge(
    summer[[
        "area_fips",
        "winter_year",
        "summer_employment"
    ]],
    on=["area_fips", "winter_year"],
    how="left",
    validate="one_to_one"
)

# =====================================================
# 5. CONSTRUCT SEASONAL ECONOMIC OUTCOMES
# =====================================================

seasonal["winter_summer_ratio"] = (
    seasonal["winter_employment"]
    / seasonal["summer_employment"]
)

seasonal["log_winter_summer_ratio"] = np.where(
    (seasonal["winter_employment"] > 0)
    & (seasonal["summer_employment"] > 0),
    np.log(seasonal["winter_summer_ratio"]),
    np.nan
)

seasonal["winter_premium_pct"] = (
    100 * (seasonal["winter_summer_ratio"] - 1)
)

# =====================================================
# 6. SAVE
# =====================================================

seasonal = seasonal.sort_values(
    ["area_fips", "winter_year"]
).reset_index(drop=True)

save_path = (
    output_dir
    / "qcew_721_winter_summer_panel_2005_2023.csv"
)

seasonal.to_csv(save_path, index=False)

# =====================================================
# 7. DIAGNOSTICS
# =====================================================

print("=" * 60)
print("WINTER-TO-SUMMER ECONOMIC PANEL")
print("=" * 60)

print("Total county-winters:", len(seasonal))
print(
    "Usable winter employment:",
    seasonal["winter_employment"].notna().sum()
)
print(
    "Usable summer employment:",
    seasonal["summer_employment"].notna().sum()
)
print(
    "Usable seasonal ratios:",
    seasonal["log_winter_summer_ratio"].notna().sum()
)

print("\nSeasonal ratio summary:")
print(
    seasonal[[
        "winter_summer_ratio",
        "winter_premium_pct"
    ]].describe().round(3)
)

print("\nCoverage by county:")
print(
    seasonal.groupby("area_fips")
    ["log_winter_summer_ratio"]
    .count()
    .to_string()
)

print("\nSaved to:")
print(save_path)


In [ ]:

import statsmodels.formula.api as smf

# Set to True only after checking the SNODAS code
SNOW_YEAR_CONFIRMED = True

if not SNOW_YEAR_CONFIRMED:
    raise RuntimeError(
        "Confirm SNODAS winter-year alignment first, "
        "then change SNOW_YEAR_CONFIRMED to True."
    )

snow = model_df.copy()

snow["fips"] = (
    snow["fips"].astype(str)
    .str.replace(r"\.0$", "", regex=True)
    .str.zfill(5)
)

snow["year"] = pd.to_numeric(snow["year"])

seasonal_merged = snow.merge(
    seasonal,
    left_on=["fips", "year"],
    right_on=["area_fips", "winter_year"],
    how="inner",
    validate="one_to_one"
)

seasonal_model_df = seasonal_merged.dropna(
    subset=[
        "log_winter_summer_ratio",
        "snow_days_10cm_2000m"
    ]
).copy()

# Exclude Clear Creek, consistent with prior models
seasonal_model_df = seasonal_model_df[
    seasonal_model_df["fips"] != "08019"
].copy()

seasonal_fe = smf.ols(
    "log_winter_summer_ratio ~ "
    "snow_days_10cm_2000m + C(fips) + C(year)",
    data=seasonal_model_df
).fit(
    cov_type="cluster",
    cov_kwds={
        "groups": seasonal_model_df["fips"]
    }
)

beta = seasonal_fe.params["snow_days_10cm_2000m"]

print("=" * 60)
print("WINTER-TO-SUMMER EMPLOYMENT REGRESSION")
print("=" * 60)

print("Observations:", int(seasonal_fe.nobs))
print("Counties:", seasonal_model_df["fips"].nunique())

print("Snow coefficient:", round(beta, 6))
print(
    "Clustered SE:",
    round(seasonal_fe.bse["snow_days_10cm_2000m"], 6)
)
print(
    "P-value:",
    round(seasonal_fe.pvalues["snow_days_10cm_2000m"], 4)
)

print(
    "Estimated effect of 10 additional snow days:",
    round(100 * (np.exp(10 * beta) - 1), 2),
    "%"
)


Snow Anomalies

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

df = seasonal_model_df.copy()
snow_col = "snow_days_10cm_2000m"

# County-specific mean snow duration
df["mean_snow"] = df.groupby("fips")[snow_col].transform("mean")
df["snow_anomaly"] = df[snow_col] - df["mean_snow"]

# Separate below-average and above-average snow
df["snow_deficit"] = df["snow_anomaly"].clip(upper=0)
df["snow_surplus"] = df["snow_anomaly"].clip(lower=0)

outcomes = [
    "log_winter_summer_ratio",
    "log_winter_employment"
]

for outcome in outcomes:
    if outcome not in df.columns:
        print(f"Skipping {outcome}: column not found")
        continue

    for label, terms in [
        ("Linear anomaly", "snow_anomaly"),
        ("Asymmetric anomaly", "snow_deficit + snow_surplus")
    ]:
        sample = df.dropna(
            subset=[outcome, "snow_anomaly", "fips", "year"]
        ).copy()

        model = smf.ols(
            f"{outcome} ~ {terms} + C(fips) + C(year)",
            data=sample
        ).fit(
            cov_type="cluster",
            cov_kwds={"groups": sample["fips"]}
        )

        print(f"\n{outcome} | {label}")
        print("Observations:", int(model.nobs))
        print("Counties:", sample["fips"].nunique())

        for var in ["snow_anomaly", "snow_deficit", "snow_surplus"]:
            if var in model.params:
                print(
                    f"{var}: beta={model.params[var]:.6f}, "
                    f"SE={model.bse[var]:.6f}, "
                    f"p={model.pvalues[var]:.4f}"
                )

Do snow effects differ aross counties?

In [ ]:
df = seasonal_model_df.copy()

# County's high-elevation terrain share
df["high_elev_share"] = df.groupby("fips")[
    "frac_above_2500m"
].transform("mean")

df["high_elev_centered"] = (
    df["high_elev_share"] - df["high_elev_share"].mean()
)

for outcome in [
    "log_winter_summer_ratio",
    "log_winter_employment"
]:
    if outcome not in df.columns:
        continue

    sample = df.dropna(subset=[
        outcome, "snow_days_10cm_2000m",
        "high_elev_centered", "fips", "year"
    ]).copy()

    model = smf.ols(
        f"{outcome} ~ snow_days_10cm_2000m * "
        "high_elev_centered + C(fips) + C(year)",
        data=sample
    ).fit(
        cov_type="cluster",
        cov_kwds={"groups": sample["fips"]}
    )

    print(f"\n{'='*60}")
    print(outcome)
    print(f"{'='*60}")

    for term in model.params.index:
        if (
            "snow_days_10cm_2000m" in term
            or "high_elev_centered" in term
        ):
            print(
                f"{term}: "
                f"beta={model.params[term]:.6f}, "
                f"p={model.pvalues[term]:.4f}"
            )

In [ ]:
from pathlib import Path
import pandas as pd

root = Path.home() / "Files/thesis/snow_ml"

print("GDP / VALUE-ADDED FILES")
print("=" * 65)

gdp_files = []

for file in root.rglob("*"):
    if not file.is_file():
        continue

    if file.suffix.lower() not in [".csv", ".xlsx"]:
        continue

    if any(term in file.name.lower() for term in [
        "gdp", "orsa", "value_added", "valueadded",
        "cagdp", "sagdp"
    ]):
        gdp_files.append(file)

for i, file in enumerate(sorted(gdp_files), 1):
    print(f"{i}. {file.relative_to(root)}")

print(f"\nTotal files: {len(gdp_files)}")

# Inspect a small sample of the files
for file in sorted(gdp_files)[:5]:
    print("\n" + "=" * 65)
    print(file.name)

    try:
        if file.suffix.lower() == ".csv":
            sample = pd.read_csv(file, nrows=8)
        else:
            sample = pd.read_excel(file, nrows=8)

        print("Columns:", sample.columns.tolist())
        print(sample.head(5).to_string(index=False)[:1800])

    except Exception as e:
        print("Could not preview:", e)

In [ ]:
import statsmodels.formula.api as smf

df = seasonal_model_df.copy()

df["high_elev_share"] = (
    df.groupby("fips")["frac_above_2500m"]
    .transform("mean")
)

# Center at the county-level mean
county_mean = (
    df.groupby("fips")["high_elev_share"]
    .first()
    .mean()
)

df["high_elev_centered"] = (
    df["high_elev_share"] - county_mean
)

heterogeneity_model = smf.ols(
    "log_winter_summer_ratio ~ "
    "snow_days_10cm_2000m + "
    "snow_days_10cm_2000m:high_elev_centered + "
    "C(fips) + C(year)",
    data=df
).fit(
    cov_type="cluster",
    cov_kwds={"groups": df["fips"]}
)

print(heterogeneity_model.summary().tables[1])

In [ ]:
import pandas as pd
from pathlib import Path

root = Path.home() / "Files/thesis/snow_ml"

files = [
    root / "ml_model/gdp/CAGDP2_CO_2001_2024.csv",
    root / "gdp_data/orsa.csv",
    root / "gdp_data/bea_total_gdp.csv"
]

for path in files:
    print("\n" + "=" * 75)
    print("FILE:", path.name)
    print("=" * 75)

    if not path.exists():
        print("File not found")
        continue

    # Raw preview to inspect BEA metadata/header layout
    print("\nFIRST 12 RAW LINES:")
    with open(path, encoding="utf-8-sig", errors="replace") as f:
        for i in range(12):
            line = f.readline()
            if not line:
                break
            print(f"{i+1:2}: {line[:220].rstrip()}")

    print("\nPANDAS PREVIEW:")
    try:
        sample = pd.read_csv(path, nrows=5)
        print("Columns:", sample.columns.tolist()[:25])
        print(sample.head(3).to_string(index=False)[:1800])
    except Exception as e:
        print("Needs custom parsing:", str(e)[:300])

In [ ]:
from pathlib import Path
import pandas as pd

root = Path.home() / "Files/thesis/snow_ml"
gdp_dir = root / "ml_model/gdp"

files = sorted(gdp_dir.glob("CAGDP2_*.csv"))

frames = []

for file in files:
    temp = pd.read_csv(file, dtype={"GeoFIPS": str})

    # Remove quotation marks and whitespace from identifiers
    temp["GeoFIPS"] = (
        temp["GeoFIPS"].astype(str)
        .str.replace('"', "", regex=False)
        .str.strip()
        .str.zfill(5)
    )

    temp["Description"] = (
        temp["Description"].astype(str).str.strip()
    )

    frames.append(temp)

gdp_raw = pd.concat(frames, ignore_index=True)

# Remove any non-geographic footer rows
gdp_raw = gdp_raw[
    gdp_raw["GeoFIPS"].str.fullmatch(r"\d{5}")
].copy()

print("Total rows:", len(gdp_raw))
print("Unique geographic areas:", gdp_raw["GeoFIPS"].nunique())

print("\nAVAILABLE INDUSTRIES")
print("=" * 75)

industries = (
    gdp_raw[[
        "LineCode",
        "IndustryClassification",
        "Description"
    ]]
    .drop_duplicates()
    .sort_values("LineCode")
)

print(industries.to_string(index=False))

print("\nYOUR SNOW COUNTIES")
print("=" * 75)

snow_counties = (
    model_df[["fips", "county", "state"]]
    .drop_duplicates()
    .copy()
)

snow_counties["fips"] = (
    snow_counties["fips"]
    .astype(int)
    .astype(str)
    .str.zfill(5)
)

snow_counties["GDP_file_available"] = (
    snow_counties["fips"].isin(gdp_raw["GeoFIPS"])
)

print(snow_counties.to_string(index=False))
print(
    "\nCounties matched:",
    snow_counties["GDP_file_available"].sum(),
    "of",
    len(snow_counties)
)

In [ ]:
import pandas as pd
from pathlib import Path

orsa_path = (
    Path.home()
    / "Files/thesis/snow_ml/gdp_data/orsa.csv"
)

orsa = pd.read_csv(orsa_path, skiprows=3)

orsa["Description"] = (
    orsa["Description"].astype(str).str.strip()
)

print("Rows:", len(orsa))
print("Columns:", orsa.columns.tolist())

print("\nOUTDOOR RECREATION CATEGORIES")
print("=" * 80)

categories = (
    orsa[["LineCode", "Description"]]
    .drop_duplicates()
    .sort_values("LineCode")
)

print(categories.to_string(index=False))

print("\nSTATES")
print(orsa["GeoName"].dropna().unique())

County GDP

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

root = Path.home() / "Files/thesis/snow_ml"
gdp_dir = root / "ml_model/gdp"

# --------------------------------------------------
# 1. Load county GDP files
# --------------------------------------------------

frames = []

for file in sorted(gdp_dir.glob("CAGDP2_*.csv")):
    temp = pd.read_csv(file, dtype={"GeoFIPS": str})

    temp["GeoFIPS"] = (
        temp["GeoFIPS"]
        .astype(str)
        .str.replace('"', "", regex=False)
        .str.strip()
        .str.zfill(5)
    )

    temp = temp[
        temp["GeoFIPS"].str.fullmatch(r"\d{5}")
    ].copy()

    frames.append(temp)

gdp_raw = pd.concat(frames, ignore_index=True)

# --------------------------------------------------
# 2. Select economic outcomes
# --------------------------------------------------

industry_codes = {
    1: "total_gdp",
    75: "tourism_broad_gdp",
    76: "arts_recreation_gdp",
    79: "accommodation_food_gdp"
}

gdp_selected = gdp_raw[
    gdp_raw["LineCode"].isin(industry_codes)
].copy()

gdp_selected["industry"] = (
    gdp_selected["LineCode"].map(industry_codes)
)

# --------------------------------------------------
# 3. Convert wide years into long format
# --------------------------------------------------

year_cols = [
    str(y) for y in range(2004, 2024)
]

gdp_long = gdp_selected.melt(
    id_vars=["GeoFIPS", "industry"],
    value_vars=year_cols,
    var_name="year",
    value_name="gdp_thousands"
)

gdp_long["year"] = gdp_long["year"].astype(int)

# BEA suppression codes become missing values
gdp_long["gdp_thousands"] = pd.to_numeric(
    gdp_long["gdp_thousands"],
    errors="coerce"
)

gdp_wide = (
    gdp_long.pivot_table(
        index=["GeoFIPS", "year"],
        columns="industry",
        values="gdp_thousands",
        aggfunc="first"
    )
    .reset_index()
)

gdp_wide.columns.name = None

# --------------------------------------------------
# 4. Prepare SNODAS county panel
# --------------------------------------------------

snow = model_df.copy()

snow["fips"] = (
    snow["fips"]
    .astype(int)
    .astype(str)
    .str.zfill(5)
)

snow["year"] = snow["year"].astype(int)

# --------------------------------------------------
# 5. Merge
# --------------------------------------------------

county_gdp_panel = snow.merge(
    gdp_wide,
    left_on=["fips", "year"],
    right_on=["GeoFIPS", "year"],
    how="left",
    validate="one_to_one"
)

# --------------------------------------------------
# 6. Check coverage
# --------------------------------------------------

print("COUNTY GDP PANEL")
print("=" * 60)

print("Observations:", len(county_gdp_panel))
print("Counties:", county_gdp_panel["fips"].nunique())
print(
    "Years:",
    county_gdp_panel["year"].min(),
    "-",
    county_gdp_panel["year"].max()
)

print("\nNON-MISSING GDP OBSERVATIONS")

for col in industry_codes.values():
    if col in county_gdp_panel.columns:
        print(
            f"{col}:",
            county_gdp_panel[col].notna().sum()
        )

print("\nGDP SUMMARY")
print(
    county_gdp_panel[
        [c for c in industry_codes.values()
         if c in county_gdp_panel.columns]
    ].describe().round(2)
)

# Save
output = root / "ml_model/county_snodas_gdp_panel.csv"
county_gdp_panel.to_csv(output, index=False)

print("\nSaved:", output)

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

df = county_gdp_panel.copy()

outcomes = {
    "total_gdp": "Total GDP",
    "tourism_broad_gdp": "Arts, Recreation, Accommodation & Food",
    "arts_recreation_gdp": "Arts, Entertainment & Recreation",
    "accommodation_food_gdp": "Accommodation & Food Services"
}

results = []

for variable, label in outcomes.items():

    for specification, exclude_covid in [
        ("Full sample", False),
        ("Exclude 2020-2021", True)
    ]:

        sample = df.copy()

        if exclude_covid:
            sample = sample[
                ~sample["year"].isin([2020, 2021])
            ]

        sample = sample[
            (sample[variable] > 0)
            & sample["snow_days_10cm_2000m"].notna()
        ].copy()

        sample["log_gdp"] = np.log(sample[variable])

        model = smf.ols(
            "log_gdp ~ snow_days_10cm_2000m "
            "+ C(fips) + C(year)",
            data=sample
        ).fit(
            cov_type="cluster",
            cov_kwds={"groups": sample["fips"]}
        )

        beta = model.params["snow_days_10cm_2000m"]
        se = model.bse["snow_days_10cm_2000m"]
        p = model.pvalues["snow_days_10cm_2000m"]

        results.append({
            "Outcome": label,
            "Specification": specification,
            "Observations": int(model.nobs),
            "Counties": sample["fips"].nunique(),
            "Coefficient": beta,
            "Clustered SE": se,
            "P-value": p,
            "10-day effect (%)": 100 * (np.exp(10 * beta) - 1)
        })

county_results = pd.DataFrame(results)

print("\nCOUNTY GDP REGRESSION RESULTS")
print("=" * 100)

print(
    county_results.round({
        "Coefficient": 6,
        "Clustered SE": 6,
        "P-value": 4,
        "10-day effect (%)": 2
    }).to_string(index=False)
)

Skiing VA

In [ ]:
# --------------------------------------------------
# 1. Load ORSA
# --------------------------------------------------

orsa = pd.read_csv(
    root / "gdp_data/orsa.csv",
    skiprows=3,
    dtype={"GeoFIPS": str}
)

orsa["GeoFIPS"] = (
    orsa["GeoFIPS"]
    .astype(str)
    .str.strip()
    .str.zfill(5)
)

# --------------------------------------------------
# 2. Select outdoor recreation categories
# --------------------------------------------------

orsa_codes = {
    1: "total_outdoor_va",
    18: "snow_activities_va",
    19: "ski_snowboard_va",
    41: "outdoor_lodging_va"
}

orsa_selected = orsa[
    orsa["LineCode"].isin(orsa_codes)
].copy()

orsa_selected["industry"] = (
    orsa_selected["LineCode"].map(orsa_codes)
)

year_cols = [
    str(y) for y in range(2012, 2024)
]

orsa_long = orsa_selected.melt(
    id_vars=["GeoFIPS", "GeoName", "industry"],
    value_vars=year_cols,
    var_name="year",
    value_name="value_added_thousands"
)

orsa_long["year"] = orsa_long["year"].astype(int)

orsa_long["value_added_thousands"] = pd.to_numeric(
    orsa_long["value_added_thousands"],
    errors="coerce"
)

orsa_wide = (
    orsa_long.pivot_table(
        index=["GeoFIPS", "GeoName", "year"],
        columns="industry",
        values="value_added_thousands",
        aggfunc="first"
    )
    .reset_index()
)

orsa_wide.columns.name = None

# --------------------------------------------------
# 3. Construct state-level snow index
# --------------------------------------------------

snow = model_df.copy()

snow["state_fips"] = (
    snow["fips"]
    .astype(int)
    .astype(str)
    .str.zfill(5)
    .str[:2]
)

snow["year"] = snow["year"].astype(int)

state_snow = (
    snow.groupby(["state_fips", "year"])
    .agg(
        snow_days_10cm=(
            "snow_days_10cm_2000m", "mean"
        ),
        snow_days_5cm=(
            "snow_days_5cm_2000m", "mean"
        ),
        mean_depth_cm=(
            "mean_depth_cm_2000m", "mean"
        ),
        n_counties=("fips", "nunique")
    )
    .reset_index()
)

# --------------------------------------------------
# 4. Merge
# --------------------------------------------------

orsa_wide["state_fips"] = (
    orsa_wide["GeoFIPS"].str[:2]
)

state_orsa_panel = orsa_wide.merge(
    state_snow,
    on=["state_fips", "year"],
    how="inner",
    validate="one_to_one"
)

print("STATE ORSA PANEL")
print("=" * 60)

print("Observations:", len(state_orsa_panel))
print("States:", state_orsa_panel["GeoName"].nunique())
print(
    "Years:",
    state_orsa_panel["year"].min(),
    "-",
    state_orsa_panel["year"].max()
)

print("\nNON-MISSING OUTCOMES")

for col in orsa_codes.values():
    print(
        col,
        state_orsa_panel[col].notna().sum()
    )

print("\nSTATE SNOW COVERAGE")

print(
    state_orsa_panel.groupby("GeoName")[
        "n_counties"
    ].agg(["min", "max"])
)

output = root / "ml_model/state_snodas_orsa_panel.csv"
state_orsa_panel.to_csv(output, index=False)

print("\nSaved:", output)

In [ ]:
df = state_orsa_panel.copy()

outcomes = {
    "ski_snowboard_va": "Skiing & Snowboarding",
    "snow_activities_va": "All Snow Activities",
    "total_outdoor_va": "Total Outdoor Recreation"
}

snow_measures = {
    "snow_days_10cm": "Snow days above 10 cm",
    "snow_days_5cm": "Snow days above 5 cm",
    "mean_depth_cm": "Mean snow depth"
}

results = []

for outcome, outcome_label in outcomes.items():

    for snow_var, snow_label in snow_measures.items():

        sample = df[
            (df[outcome] > 0)
            & df[snow_var].notna()
        ].copy()

        sample["log_va"] = np.log(sample[outcome])

        model = smf.ols(
            f"log_va ~ {snow_var} "
            "+ C(state_fips) + C(year)",
            data=sample
        ).fit(
            cov_type="cluster",
            cov_kwds={"groups": sample["state_fips"]}
        )

        results.append({
            "Outcome": outcome_label,
            "Snow measure": snow_label,
            "Observations": int(model.nobs),
            "States": sample["state_fips"].nunique(),
            "Coefficient": model.params[snow_var],
            "Clustered SE": model.bse[snow_var],
            "P-value": model.pvalues[snow_var]
        })

state_results = pd.DataFrame(results)

print("\nSTATE OUTDOOR RECREATION REGRESSIONS")
print("=" * 110)

print(
    state_results.round({
        "Coefficient": 6,
        "Clustered SE": 6,
        "P-value": 4
    }).to_string(index=False)
)

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

df = county_gdp_panel.copy().sort_values(["fips", "year"])

snow_var = "snow_days_10cm_2000m"

# Explicitly match snow season years to GDP years
snow_lookup = df[["fips", "year", snow_var]].drop_duplicates().copy()

results = []

outcomes = [
    "total_gdp",
    "tourism_broad_gdp",
    "arts_recreation_gdp",
    "accommodation_food_gdp"
]

for shift, label in [
    (-1, "Following snow season (t+1)"),
    (0, "Current snow season (t)"),
    (1, "Previous snow season (t-1)")
]:
    shifted = snow_lookup.copy()

    # A snow observation from year s is assigned to GDP year s - shift
    shifted["year"] = shifted["year"] - shift
    shifted = shifted.rename(columns={snow_var: "snow_test"})

    sample_base = df.drop(columns=[snow_var]).merge(
        shifted,
        on=["fips", "year"],
        how="left",
        validate="one_to_one"
    )

    for outcome in outcomes:
        sample = sample_base[
            (sample_base[outcome] > 0)
            & sample_base["snow_test"].notna()
        ].copy()

        sample["log_gdp"] = np.log(sample[outcome])

        model = smf.ols(
            "log_gdp ~ snow_test + C(fips) + C(year)",
            data=sample
        ).fit(
            cov_type="cluster",
            cov_kwds={"groups": sample["fips"]}
        )

        results.append({
            "Outcome": outcome,
            "Timing": label,
            "N": int(model.nobs),
            "Beta": model.params["snow_test"],
            "SE": model.bse["snow_test"],
            "P-value": model.pvalues["snow_test"],
            "10-day effect (%)": 100 * (
                np.exp(10 * model.params["snow_test"]) - 1
            )
        })

timing_results = pd.DataFrame(results)

print(
    timing_results.round({
        "Beta": 6,
        "SE": 6,
        "P-value": 4,
        "10-day effect (%)": 2
    }).to_string(index=False)
)

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

df = county_gdp_panel.copy()

# --------------------------------------------------
# 1. Construct baseline tourism dependence
# --------------------------------------------------

baseline = df[
    df["year"].between(2004, 2008)
].copy()

baseline["tourism_share"] = (
    baseline["tourism_broad_gdp"]
    / baseline["total_gdp"]
)

county_dependence = (
    baseline.groupby("fips")
    .agg(
        tourism_share_baseline=("tourism_share", "mean"),
        county=("county", "first"),
        state=("state", "first")
    )
    .reset_index()
)

# --------------------------------------------------
# 2. Display county tourism dependence
# --------------------------------------------------

print("BASELINE TOURISM DEPENDENCE (2004–2008)")
print("=" * 75)

display_table = county_dependence.copy()
display_table["tourism_share_pct"] = (
    100 * display_table["tourism_share_baseline"]
)

print(
    display_table[
        ["county", "state", "tourism_share_pct"]
    ]
    .sort_values("tourism_share_pct", ascending=False)
    .round(2)
    .to_string(index=False)
)

# --------------------------------------------------
# 3. Merge dependence into GDP panel
# --------------------------------------------------

df = df.merge(
    county_dependence[
        ["fips", "tourism_share_baseline"]
    ],
    on="fips",
    how="left",
    validate="many_to_one"
)

# Center tourism dependence
df["tourism_share_centered"] = (
    df["tourism_share_baseline"]
    - county_dependence["tourism_share_baseline"].mean()
)

# Restrict regressions to post-baseline years
df = df[df["year"] >= 2009].copy()

# --------------------------------------------------
# 4. Run interaction regressions
# --------------------------------------------------

outcomes = {
    "total_gdp": "Total GDP",
    "tourism_broad_gdp": "Broad Tourism GDP",
    "arts_recreation_gdp": "Arts & Recreation GDP",
    "accommodation_food_gdp": "Accommodation & Food GDP"
}

results = []

for outcome, label in outcomes.items():

    sample = df[
        (df[outcome] > 0)
        & df["snow_days_10cm_2000m"].notna()
        & df["tourism_share_centered"].notna()
    ].copy()

    sample["log_gdp"] = np.log(sample[outcome])

    model = smf.ols(
        "log_gdp ~ snow_days_10cm_2000m "
        "+ snow_days_10cm_2000m:tourism_share_centered "
        "+ C(fips) + C(year)",
        data=sample
    ).fit(
        cov_type="cluster",
        cov_kwds={"groups": sample["fips"]}
    )

    snow_var = "snow_days_10cm_2000m"
    interaction_var = (
        "snow_days_10cm_2000m:tourism_share_centered"
    )

    results.append({
        "Outcome": label,
        "N": int(model.nobs),
        "Counties": sample["fips"].nunique(),
        "Snow beta": model.params[snow_var],
        "Snow p": model.pvalues[snow_var],
        "Interaction beta": model.params[interaction_var],
        "Interaction p": model.pvalues[interaction_var]
    })

interaction_results = pd.DataFrame(results)

print("\nTOURISM DEPENDENCE INTERACTION RESULTS")
print("=" * 100)

print(
    interaction_results.round({
        "Snow beta": 6,
        "Snow p": 4,
        "Interaction beta": 6,
        "Interaction p": 4
    }).to_string(index=False)
)

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

df = county_gdp_panel.copy()

# Rebuild baseline tourism dependence
baseline = df[df["year"].between(2004, 2008)].copy()

baseline["tourism_share"] = (
    baseline["tourism_broad_gdp"] / baseline["total_gdp"]
)

dependence = (
    baseline.groupby("fips")["tourism_share"]
    .mean()
    .rename("tourism_share_baseline")
    .reset_index()
)

df = df.merge(
    dependence,
    on="fips",
    how="left",
    validate="many_to_one"
)

df = df[df["year"] >= 2009].copy()

# Fixed centering reference across all exclusions
reference_share = dependence["tourism_share_baseline"].mean()

df["tourism_share_centered"] = (
    df["tourism_share_baseline"] - reference_share
)

df = df[
    (df["accommodation_food_gdp"] > 0)
    & df["snow_days_10cm_2000m"].notna()
    & df["tourism_share_centered"].notna()
].copy()

df["log_gdp"] = np.log(df["accommodation_food_gdp"])

interaction = "snow_days_10cm_2000m:tourism_share_centered"

formula = (
    "log_gdp ~ snow_days_10cm_2000m "
    "+ snow_days_10cm_2000m:tourism_share_centered "
    "+ C(fips) + C(year)"
)

def estimate(sample):
    model = smf.ols(formula, data=sample).fit(
        cov_type="cluster",
        cov_kwds={"groups": sample["fips"]}
    )

    return {
        "Interaction beta": model.params[interaction],
        "SE": model.bse[interaction],
        "P-value": model.pvalues[interaction],
        "N": int(model.nobs),
        "Counties": sample["fips"].nunique()
    }

results = []

# Full model
results.append({
    "Excluded county": "None (full model)",
    **estimate(df)
})

# Exclude one county at a time
for fips in sorted(df["fips"].unique()):
    county_rows = df[df["fips"] == fips]

    county_name = county_rows["county"].iloc[0]
    state = county_rows["state"].iloc[0]

    sample = df[df["fips"] != fips].copy()

    results.append({
        "Excluded county": f"{county_name} ({state})",
        **estimate(sample)
    })

loo_results = pd.DataFrame(results)

print("\nLEAVE-ONE-COUNTY-OUT RESULTS")
print("=" * 100)

print(
    loo_results.round({
        "Interaction beta": 6,
        "SE": 6,
        "P-value": 4
    }).to_string(index=False)
)

print("\nSUMMARY")
print("Negative coefficients:",
      (loo_results.iloc[1:]["Interaction beta"] < 0).sum(),
      "of 20")

print("P < 0.05:",
      (loo_results.iloc[1:]["P-value"] < 0.05).sum(),
      "of 20")

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

df = county_gdp_panel.copy()

# Baseline tourism dependence
baseline = df[df["year"].between(2004, 2008)].copy()

baseline["tourism_share"] = (
    baseline["tourism_broad_gdp"] / baseline["total_gdp"]
)

dependence = (
    baseline.groupby("fips")["tourism_share"]
    .mean()
    .rename("tourism_share_baseline")
    .reset_index()
)

df = df.merge(
    dependence,
    on="fips",
    how="left",
    validate="many_to_one"
)

df = df[df["year"] >= 2009].copy()

df["tourism_share_centered"] = (
    df["tourism_share_baseline"]
    - dependence["tourism_share_baseline"].mean()
)

df["time_trend"] = df["year"] - 2009

df = df[
    (df["accommodation_food_gdp"] > 0)
    & df["snow_days_10cm_2000m"].notna()
    & df["tourism_share_centered"].notna()
].copy()

df["log_gdp"] = np.log(df["accommodation_food_gdp"])

snow = "snow_days_10cm_2000m"
interaction = f"{snow}:tourism_share_centered"

specifications = {
    "Original FE": (
        f"log_gdp ~ {snow} + {interaction} "
        "+ C(fips) + C(year)"
    ),
    "County-specific trends": (
        f"log_gdp ~ {snow} + {interaction} "
        "+ C(fips) + C(year) "
        "+ C(fips):time_trend"
    ),
    "Exclude COVID years": (
        f"log_gdp ~ {snow} + {interaction} "
        "+ C(fips) + C(year)"
    )
}

results = []

for name, formula in specifications.items():

    sample = df.copy()

    if name == "Exclude COVID years":
        sample = sample[
            ~sample["year"].isin([2020, 2021])
        ].copy()

    model = smf.ols(
        formula,
        data=sample
    ).fit(
        cov_type="cluster",
        cov_kwds={"groups": sample["fips"]}
    )

    results.append({
        "Specification": name,
        "N": int(model.nobs),
        "Counties": sample["fips"].nunique(),
        "Snow beta": model.params[snow],
        "Snow p": model.pvalues[snow],
        "Interaction beta": model.params[interaction],
        "Interaction SE": model.bse[interaction],
        "Interaction p": model.pvalues[interaction]
    })

trend_results = pd.DataFrame(results)

print("\nCOUNTY TREND ROBUSTNESS")
print("=" * 105)

print(
    trend_results.round({
        "Snow beta": 6,
        "Snow p": 4,
        "Interaction beta": 6,
        "Interaction SE": 6,
        "Interaction p": 4
    }).to_string(index=False)
)

In [ ]:
# Create county trends relative to a reference county
reference_fips = sorted(df["fips"].unique())[0]

trend_columns = []

for fips in sorted(df["fips"].unique()):
    if fips == reference_fips:
        continue

    col = f"trend_{int(fips)}"

    df[col] = (
        (df["fips"] == fips).astype(int)
        * df["time_trend"]
    )

    trend_columns.append(col)

formula = (
    "log_gdp ~ snow_days_10cm_2000m "
    "+ snow_days_10cm_2000m:tourism_share_centered "
    "+ C(fips) + C(year) "
    "+ " + " + ".join(trend_columns)
)

model = smf.ols(
    formula,
    data=df
).fit(
    cov_type="cluster",
    cov_kwds={"groups": df["fips"]}
)

print(model.params[
    "snow_days_10cm_2000m:tourism_share_centered"
])

print(model.pvalues[
    "snow_days_10cm_2000m:tourism_share_centered"
])

In [ ]:
snow_measures = [
    "snow_days_5cm_2000m",
    "snow_days_10cm_2000m",
    "mean_depth_cm_2000m"
]

results = []

for snow in snow_measures:

    sample = df[
        df[snow].notna()
        & (df["accommodation_food_gdp"] > 0)
    ].copy()

    interaction = f"{snow}:tourism_share_centered"

    formula = (
        f"log_gdp ~ {snow} "
        f"+ {interaction} "
        "+ C(fips) + C(year) "
        "+ " + " + ".join(trend_columns)
    )

    model = smf.ols(
        formula,
        data=sample
    ).fit(
        cov_type="cluster",
        cov_kwds={"groups": sample["fips"]}
    )

    results.append({
        "Snow measure": snow,
        "N": int(model.nobs),
        "Snow beta": model.params[snow],
        "Snow p": model.pvalues[snow],
        "Interaction beta": model.params[interaction],
        "Interaction SE": model.bse[interaction],
        "Interaction p": model.pvalues[interaction]
    })

snow_comparison = pd.DataFrame(results)

print("\nALTERNATIVE SNOW MEASURE RESULTS")
print("=" * 110)

print(
    snow_comparison.round({
        "Snow beta": 6,
        "Snow p": 4,
        "Interaction beta": 6,
        "Interaction SE": 6,
        "Interaction p": 4
    }).to_string(index=False)
)

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

df_fd = county_gdp_panel.copy()
df_fd = df_fd.sort_values(["fips", "year"])

# Baseline tourism dependence
baseline = df_fd[df_fd["year"].between(2004, 2008)].copy()

baseline["tourism_share"] = (
    baseline["tourism_broad_gdp"] / baseline["total_gdp"]
)

dependence = (
    baseline.groupby("fips")["tourism_share"]
    .mean()
    .rename("tourism_share_baseline")
    .reset_index()
)

df_fd = df_fd.merge(
    dependence,
    on="fips",
    how="left",
    validate="many_to_one"
)

df_fd["tourism_share_centered"] = (
    df_fd["tourism_share_baseline"]
    - dependence["tourism_share_baseline"].mean()
)

# Log accommodation and food GDP
df_fd["log_gdp"] = np.where(
    df_fd["accommodation_food_gdp"] > 0,
    np.log(df_fd["accommodation_food_gdp"].clip(lower=1)),
    np.nan
)

snow_measures = [
    "snow_days_5cm_2000m",
    "snow_days_10cm_2000m",
    "mean_depth_cm_2000m"
]

# Compute annual differences within each county
df_fd["d_log_gdp"] = (
    df_fd.groupby("fips")["log_gdp"].diff()
)

for snow in snow_measures:
    df_fd[f"d_{snow}"] = (
        df_fd.groupby("fips")[snow].diff()
    )

# Keep changes from 2009 onward
df_fd = df_fd[df_fd["year"] >= 2009].copy()

results = []

for snow in snow_measures:

    d_snow = f"d_{snow}"
    interaction = f"{d_snow}:tourism_share_centered"

    sample = df_fd.dropna(
        subset=[
            "d_log_gdp",
            d_snow,
            "tourism_share_centered"
        ]
    ).copy()

    model = smf.ols(
        f"d_log_gdp ~ {d_snow} "
        f"+ {interaction} "
        "+ C(year)",
        data=sample
    ).fit(
        cov_type="cluster",
        cov_kwds={"groups": sample["fips"]}
    )

    results.append({
        "Snow measure": snow,
        "N": int(model.nobs),
        "Counties": sample["fips"].nunique(),
        "Snow beta": model.params[d_snow],
        "Snow p": model.pvalues[d_snow],
        "Interaction beta": model.params[interaction],
        "Interaction SE": model.bse[interaction],
        "Interaction p": model.pvalues[interaction]
    })

fd_results = pd.DataFrame(results)

print("\nFIRST-DIFFERENCE GDP RESULTS")
print("=" * 110)

print(
    fd_results.round({
        "Snow beta": 6,
        "Snow p": 4,
        "Interaction beta": 6,
        "Interaction SE": 6,
        "Interaction p": 4
    }).to_string(index=False)
)

In [ ]:
from pathlib import Path
import re

root = Path.home() / "Files/thesis/snow_ml"

terms = [
    "snow_days_10cm_2000m",
    "snow_days_5cm_2000m",
    "mean_depth_cm_2000m",
    "season_days",
    "SNODAS"
]

extensions = {".py", ".js", ".ipynb"}

matches = []

for path in root.rglob("*"):
    if not path.is_file() or path.suffix.lower() not in extensions:
        continue

    if any(part in {".venv", "node_modules", ".git"}
           for part in path.parts):
        continue

    try:
        content = path.read_text(
            encoding="utf-8", errors="ignore"
        )
    except Exception:
        continue

    found = [
        term for term in terms
        if re.search(re.escape(term), content, re.IGNORECASE)
    ]

    if found:
        matches.append((path, found))

print("FILES CONTAINING SNODAS REFERENCES")
print("=" * 75)

for path, found in matches:
    print(path.relative_to(root))
    print("  Matches:", ", ".join(found))

In [ ]:
from pathlib import Path
import json

root = Path.home() / "Files/thesis/snow_ml"
notebook_path = root / "code/snow_model.ipynb"

with open(notebook_path, "r", encoding="utf-8") as f:
    notebook = json.load(f)

search_terms = [
    "snow_days_10cm_2000m",
    "snow_days_5cm_2000m",
    "mean_depth_cm_2000m",
    "season_days",
    "read_csv",
    "SNODAS"
]

print("SNOW MODEL NOTEBOOK — RELEVANT CELLS")
print("=" * 80)

for i, cell in enumerate(notebook["cells"]):

    if cell["cell_type"] != "code":
        continue

    source = "".join(cell.get("source", []))

    matches = [
        term for term in search_terms
        if term.lower() in source.lower()
    ]

    if matches:
        print(f"\n{'=' * 80}")
        print(f"CELL {i}")
        print("MATCHES:", ", ".join(matches))
        print("-" * 80)

        # Show the beginning of each relevant cell
        lines = source.splitlines()

        for line in lines[:35]:
            print(line)

        if len(lines) > 35:
            print(f"... [{len(lines) - 35} additional lines]")

In [ ]:
from pathlib import Path

root = Path.home() / "Files/thesis/snow_ml"

keywords = [
    "snodas",
    "snow",
    "gee",
    "earth_engine",
    "earthengine"
]

extensions = {
    ".js", ".py", ".ipynb",
    ".csv", ".tif", ".tiff",
    ".json", ".geojson"
}

print("POTENTIAL SNODAS SOURCE FILES")
print("=" * 85)

for path in sorted(root.rglob("*")):

    if not path.is_file():
        continue

    if path.suffix.lower() not in extensions:
        continue

    if any(
        word in path.name.lower()
        for word in keywords
    ):
        print(path.relative_to(root))

In [ ]:
from pathlib import Path
import pandas as pd

root = Path.home() / "Files/thesis/snow_ml"

files = [
    "code/ski_snow_states/snodas_depth_panel.csv",
    "code/ski_snow_states/snodas_local_depth_panel.csv",
    "ml_model/county_snodas_panel_v1.csv",
    "ml_model/county_snow_panel.csv",
    "ml_model/county_snow_validation_v3.csv"
]

for filename in files:

    path = root / filename

    print("\n" + "=" * 85)
    print("FILE:", filename)
    print("=" * 85)

    if not path.exists():
        print("FILE NOT FOUND")
        continue

    df_check = pd.read_csv(path)

    print("\nSHAPE:", df_check.shape)

    print("\nCOLUMNS:")
    print(df_check.columns.tolist())

    print("\nFIRST 3 ROWS:")
    print(df_check.head(3).to_string(index=False))

    print("\nDATE / YEAR COVERAGE:")

    for col in df_check.columns:
        if any(term in col.lower() for term in
               ["year", "date", "season"]):

            print(
                col,
                "min:", df_check[col].min(),
                "max:", df_check[col].max()
            )

    print("\nGEOGRAPHIC COVERAGE:")

    for col in ["state", "county", "fips", "GeoName",
                "resort", "location"]:

        if col in df_check.columns:
            print(
                col,
                "unique values:",
                df_check[col].nunique()
            )
            print(
                df_check[col].dropna().unique()[:15]
            )

In [ ]:
from pathlib import Path
import pandas as pd

root = Path.home() / "Files/thesis/snow_ml"

snow = pd.read_csv(
    root / "ml_model/county_snodas_panel_v1.csv",
    dtype={"fips": str}
)

snow["fips"] = snow["fips"].str.zfill(5)

print("SNODAS COUNTY COVERAGE")
print("=" * 80)

coverage = (
    snow.groupby(["fips", "state", "county"])
    .agg(
        observations=("year", "size"),
        first_year=("year", "min"),
        last_year=("year", "max"),
        mean_snow_days=("snow_days_10cm_2000m", "mean"),
        mean_depth=("mean_depth_cm_2000m", "mean"),
        valid_fraction=("valid_fraction_2000m", "mean")
    )
    .reset_index()
    .sort_values(["state", "county"])
)

print(coverage.round(2).to_string(index=False))

print("\nDATA QUALITY")
print("=" * 80)

print("Rows:", len(snow))
print("Unique FIPS:", snow["fips"].nunique())
print("Duplicate county-year rows:",
      snow.duplicated(["fips", "year"]).sum())

print("\nMISSING VALUES")
print(
    snow[
        [
            "snow_days_10cm_2000m",
            "mean_depth_cm_2000m",
            "peak_depth_cm_2000m",
            "valid_fraction_2000m"
        ]
    ].isna().sum()
)

print("\nSNOW VARIABLE SUMMARY")
print("=" * 80)

print(
    snow[
        [
            "snow_days_5cm_2000m",
            "snow_days_10cm_2000m",
            "mean_depth_cm_2000m",
            "peak_depth_cm_2000m",
            "valid_fraction_2000m"
        ]
    ].describe().round(2).to_string()
)

print("\nSEASON LENGTH BY YEAR")
print(
    snow.groupby("year")["season_days"]
    .agg(["min", "max"])
    .to_string()
)

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from scipy.stats import linregress

root = Path.home() / "Files/thesis/snow_ml"

snow = pd.read_csv(
    root / "ml_model/county_snodas_panel_v1.csv",
    dtype={"fips": str}
)

snow["fips"] = snow["fips"].str.zfill(5)
snow["year"] = pd.to_numeric(snow["year"])

# Exclude counties without valid snow observations
snow = snow.dropna(
    subset=[
        "snow_days_10cm_2000m",
        "mean_depth_cm_2000m"
    ]
).copy()

variables = {
    "snow_days_10cm_2000m": "Snow duration (days)",
    "mean_depth_cm_2000m": "Mean snow depth (cm)"
}

results = []

for (fips, county, state), group in snow.groupby(
    ["fips", "county", "state"]
):

    for variable, label in variables.items():

        sample = group.dropna(
            subset=["year", variable]
        )

        if len(sample) < 10:
            continue

        fit = linregress(
            sample["year"],
            sample[variable]
        )

        results.append({
            "State": state,
            "County": county,
            "Measure": label,
            "Annual trend": fit.slope,
            "P-value": fit.pvalue,
            "2004 value": sample.loc[
                sample["year"] == 2004, variable
            ].mean(),
            "2023 value": sample.loc[
                sample["year"] == 2023, variable
            ].mean()
        })

trends = pd.DataFrame(results)

print("COUNTY SNOW TRENDS, 2004–2023")
print("=" * 100)

print(
    trends.round(3).to_string(index=False)
)

print("\nSUMMARY BY SNOW MEASURE")
print("=" * 80)

summary = trends.groupby("Measure").agg(
    Counties=("County", "count"),
    Mean_annual_trend=("Annual trend", "mean"),
    Negative_trends=("Annual trend", lambda x: (x < 0).sum()),
    Positive_trends=("Annual trend", lambda x: (x > 0).sum()),
    Significant_negative=(
        "P-value",
        lambda x: (
            (x < 0.05) &
            (trends.loc[x.index, "Annual trend"] < 0)
        ).sum()
    )
)

print(summary.round(3).to_string())

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from scipy.stats import linregress

root = Path.home() / "Files/thesis/snow_ml"

df = pd.read_csv(
    root / "ml_model/county_snodas_panel_v1.csv",
    dtype={"fips": str}
)

df["fips"] = df["fips"].str.zfill(5)
df["year"] = pd.to_numeric(df["year"])

df = df.dropna(
    subset=[
        "snow_days_10cm_2000m",
        "mean_depth_cm_2000m",
        "valid_fraction_2000m"
    ]
).copy()

# Annual means across counties
annual = (
    df.groupby("year")
    .agg(
        snow_days=("snow_days_10cm_2000m", "mean"),
        mean_depth=("mean_depth_cm_2000m", "mean"),
        valid_fraction=("valid_fraction_2000m", "mean"),
        counties=("fips", "nunique")
    )
    .reset_index()
)

# Exclude Box Elder
annual_no_box = (
    df[df["fips"] != "49003"]
    .groupby("year")
    .agg(
        snow_days=("snow_days_10cm_2000m", "mean"),
        mean_depth=("mean_depth_cm_2000m", "mean")
    )
    .reset_index()
)

print("ANNUAL SNODAS SUMMARY")
print("=" * 80)
print(annual.round(3).to_string(index=False))

print("\nTREND COMPARISON")
print("=" * 80)

for label, data in [
    ("All 20 counties", annual),
    ("Excluding Box Elder", annual_no_box)
]:
    for variable in ["snow_days", "mean_depth"]:
        fit = linregress(data["year"], data[variable])

        print(
            f"{label:25s} | "
            f"{variable:12s} | "
            f"slope={fit.slope:+.3f}/year | "
            f"p={fit.pvalue:.4f}"
        )

print("\nVALID DATA FRACTION TREND")
print("=" * 80)

fit = linregress(
    annual["year"],
    annual["valid_fraction"]
)

print(
    f"Annual change in valid fraction: "
    f"{fit.slope:+.6f} | p={fit.pvalue:.4f}"
)

print("\nBOX ELDER YEAR-BY-YEAR")
print("=" * 80)

box = df[df["fips"] == "49003"][
    [
        "year",
        "snow_days_10cm_2000m",
        "mean_depth_cm_2000m",
        "valid_fraction_2000m"
    ]
]

print(box.round(3).to_string(index=False))

In [ ]:
import pandas as pd
import numpy as np

print("AVAILABLE PREDICTION COLUMNS")
print("=" * 70)

prediction_cols = [
    col for col in model_df.columns
    if any(term in col.lower()
           for term in ["pred", "snow", "year"])
]

print(prediction_cols)

print("\nMODEL PREDICTIONS")
print("=" * 70)

cols = [
    "fips",
    "county",
    "year",
    "snow_days_10cm_2000m",
    "rf_predicted"
]

available = [
    col for col in cols
    if col in model_df.columns
]

print(
    model_df[available]
    .head(15)
    .round(2)
    .to_string(index=False)
)

print("\nPREDICTION PERFORMANCE")
print("=" * 70)

if {"snow_days_10cm_2000m", "rf_predicted"}.issubset(model_df.columns):

    from sklearn.metrics import (
        r2_score,
        mean_absolute_error
    )

    valid = model_df[
        ["snow_days_10cm_2000m", "rf_predicted"]
    ].dropna()

    print(
        "R²:",
        round(
            r2_score(
                valid["snow_days_10cm_2000m"],
                valid["rf_predicted"]
            ), 3
        )
    )

    print(
        "MAE:",
        round(
            mean_absolute_error(
                valid["snow_days_10cm_2000m"],
                valid["rf_predicted"]
            ), 2
        ),
        "days"
    )

In [ ]:
import pandas as pd
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import numpy as np

target = "snow_days_10cm_2000m"

models = {
    "Random Forest": "rf_predicted",
    "XGBoost": "xgb_predicted",
    "Linear Regression (FE)": "fe_predicted",
    "Baseline": "baseline_predicted"
}

results = []

for name, prediction in models.items():

    sample = model_df[[target, prediction]].dropna()

    results.append({
        "Model": name,
        "N": len(sample),
        "R²": r2_score(sample[target], sample[prediction]),
        "MAE (days)": mean_absolute_error(
            sample[target], sample[prediction]
        ),
        "RMSE (days)": np.sqrt(
            mean_squared_error(sample[target], sample[prediction])
        )
    })

comparison = pd.DataFrame(results).sort_values("MAE (days)")

print("MODEL COMPARISON")
print("=" * 75)
print(comparison.round(3).to_string(index=False))

In [ ]:
print("MODEL COLUMNS")
print(model_df.columns.tolist())

print("\nPOTENTIAL REGRESSION OBJECTS")

for name in [
    "fe_model",
    "fe_results",
    "fe_fit",
    "ols_model",
    "ols_results",
    "feature_cols"
]:
    if name in globals():
        obj = globals()[name]
        print(f"\n{name}: {type(obj).__name__}")

        if hasattr(obj, "params"):
            print(obj.params)

        elif isinstance(obj, (list, tuple)):
            print(obj)

In [ ]:
print("FIXED-EFFECTS MODEL PIPELINE")
print("=" * 70)

for name, step in fe_model.named_steps.items():
    print(f"\nStep: {name}")
    print(step)

print("\nMODEL INPUT FEATURES")
print("=" * 70)

if hasattr(fe_model, "feature_names_in_"):
    print(fe_model.feature_names_in_.tolist())
else:
    print("Feature names not stored in pipeline.")

print("\nREGRESSION COEFFICIENTS")
print("=" * 70)

try:
    coefficients = fe_model.named_steps[
        list(fe_model.named_steps.keys())[-1]
    ].coef_

    features = fe_model[:-1].get_feature_names_out()

    import pandas as pd

    coef_df = pd.DataFrame({
        "Feature": features,
        "Coefficient": coefficients.ravel()
    })

    print(
        coef_df.sort_values(
            "Coefficient"
        ).round(4).to_string(index=False)
    )

except Exception as e:
    print("Could not extract coefficients:", e)

In [ ]:
import pandas as pd
import numpy as np

# Use your existing county-year model dataset
df = model_df.copy()

features = [
    "positive_degree_days_2000m",
    "precip_djf_2000m",
    "precip_mar_apr_2000m",
    "fips"
]

# Ensure consistent county identifiers
df["fips"] = df["fips"].astype(str).str.zfill(5)

# Keep observations with complete predictors
df = df.dropna(subset=features).copy()

# Predict baseline snow conditions
df["baseline_snow_pred"] = fe_model.predict(df[features])

scenarios = {
    "PDD +25": 25,
    "PDD +50": 50,
    "PDD +100": 100
}

results = []

for scenario, increase in scenarios.items():

    scenario_data = df[features].copy()

    scenario_data["positive_degree_days_2000m"] += increase

    scenario_predictions = fe_model.predict(scenario_data)

    results.append({
        "Scenario": scenario,
        "Mean baseline snow days": df["baseline_snow_pred"].mean(),
        "Mean scenario snow days": scenario_predictions.mean(),
        "Change in snow days": (
            scenario_predictions - df["baseline_snow_pred"]
        ).mean(),
        "Percent change": (
            scenario_predictions.mean()
            / df["baseline_snow_pred"].mean() - 1
        ) * 100
    })

scenario_results = pd.DataFrame(results)

print("FIXED-EFFECTS SNOW SCENARIOS")
print("=" * 80)

print(
    scenario_results.round(2).to_string(index=False)
)

In [ ]:
import pandas as pd
import numpy as np

models_to_check = {
    "Fixed Effects": "fe_model",
    "Random Forest": "rf_model",
    "XGBoost": "xgb_model"
}

for label, name in models_to_check.items():

    print(f"\n{'=' * 65}")
    print(label.upper())
    print("=" * 65)

    if name not in globals():
        print(f"{name} not found")
        continue

    model = globals()[name]

    print("Object type:", type(model).__name__)

    if hasattr(model, "feature_names_in_"):
        print(
            "Input features:",
            model.feature_names_in_.tolist()
        )

    if hasattr(model, "named_steps"):
        print("\nPipeline steps:")
        for step_name, step in model.named_steps.items():
            print(f"{step_name}: {step}")

    if hasattr(model, "get_booster"):
        print(
            "\nXGBoost features:",
            model.get_booster().feature_names
        )

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.pipeline import Pipeline

print("SEARCHING FOR TRAINED MODELS")
print("=" * 70)

for name, obj in list(globals().items()):

    if name.startswith("_"):
        continue

    try:
        if isinstance(obj, (RandomForestRegressor, XGBRegressor)):
            print(f"\n{name}: {type(obj).__name__}")

            if hasattr(obj, "feature_names_in_"):
                print("Features:", obj.feature_names_in_.tolist())

        elif isinstance(obj, Pipeline):

            model_types = [
                type(step).__name__
                for step in obj.named_steps.values()
            ]

            if any(
                "RandomForest" in t or "XGB" in t
                for t in model_types
            ):
                print(f"\n{name}: Pipeline")
                print("Steps:", model_types)

                if hasattr(obj, "feature_names_in_"):
                    print("Features:", obj.feature_names_in_.tolist())

    except Exception:
        continue

In [ ]:
import pandas as pd
import numpy as np

# Existing trained models
models = {
    "Fixed Effects": fe_model,
    "Random Forest": final_rf,
    "XGBoost": xgb
}

# Use the same county-year observations for all models
df = model_df.copy()
df["fips"] = df["fips"].astype(str).str.zfill(5)

# Identify the features used by each model
features = {
    name: list(model.feature_names_in_)
    for name, model in models.items()
}

# Restrict to observations with complete inputs for every model
all_features = sorted(set().union(*features.values()))

df = df.dropna(subset=all_features).copy()

print("Common observations:", len(df))

scenarios = {
    "Baseline": 0,
    "PDD +25": 25,
    "PDD +50": 50,
    "PDD +100": 100
}

results = []
county_results = []

for model_name, model in models.items():

    model_features = features[model_name]
    baseline_inputs = df[model_features].copy()
    baseline_predictions = model.predict(baseline_inputs)

    for scenario_name, pdd_increase in scenarios.items():

        scenario_inputs = baseline_inputs.copy()

        scenario_inputs["positive_degree_days_2000m"] += pdd_increase

        scenario_predictions = model.predict(scenario_inputs)

        # Snow duration is physically bounded by the season length
        baseline_bounded = np.clip(
            baseline_predictions, 0, df["season_days"].to_numpy()
        )

        scenario_bounded = np.clip(
            scenario_predictions, 0, df["season_days"].to_numpy()
        )

        change = scenario_bounded - baseline_bounded

        results.append({
            "Model": model_name,
            "Scenario": scenario_name,
            "Baseline snow days": baseline_bounded.mean(),
            "Scenario snow days": scenario_bounded.mean(),
            "Change in snow days": change.mean(),
            "Percent change": (
                100 * change.mean() / baseline_bounded.mean()
            )
        })

        for i, (_, row) in enumerate(df.iterrows()):
            county_results.append({
                "fips": row["fips"],
                "county": row["county"],
                "state": row["state"],
                "year": row["year"],
                "Model": model_name,
                "Scenario": scenario_name,
                "Baseline snow days": baseline_bounded[i],
                "Scenario snow days": scenario_bounded[i],
                "Change in snow days": change[i]
            })

scenario_comparison = pd.DataFrame(results)
county_scenarios = pd.DataFrame(county_results)

print("\nTHREE-MODEL SNOW SCENARIO COMPARISON")
print("=" * 95)

print(
    scenario_comparison
    .round(2)
    .to_string(index=False)
)

In [ ]:
import pandas as pd
import numpy as np

econ = county_scenarios.copy()

# Hypothetical demand elasticities:
# NOT empirical estimates from the literature.
elasticities = {
    "Low response": 0.25,
    "Medium response": 0.50,
    "High response": 1.00
}

# Prevent undefined percentage changes
econ = econ[econ["Baseline snow days"] > 0].copy()

econ["snow_ratio"] = (
    econ["Scenario snow days"]
    / econ["Baseline snow days"]
)

results = []

for label, elasticity in elasticities.items():

    temp = econ.copy()

    temp["Demand assumption"] = label
    temp["Elasticity"] = elasticity

    # Constant-elasticity demand response
    temp["Demand ratio"] = (
        temp["snow_ratio"].clip(lower=0)
        ** elasticity
    )

    temp["Demand change (%)"] = (
        temp["Demand ratio"] - 1
    ) * 100

    temp["Baseline demand index"] = 100

    temp["Scenario demand index"] = (
        100 * temp["Demand ratio"]
    )

    results.append(temp)

economic_scenarios = pd.concat(
    results, ignore_index=True
)

# Each county-year receives equal weight.
# This is NOT an industry-revenue-weighted result.
economic_summary = (
    economic_scenarios
    .groupby(
        ["Model", "Scenario", "Demand assumption"],
        as_index=False
    )
    .agg(
        Mean_demand_change_pct=(
            "Demand change (%)", "mean"
        ),
        Mean_demand_index=(
            "Scenario demand index", "mean"
        )
    )
)

print("THREE-MODEL ECONOMIC SENSITIVITY ANALYSIS")
print("=" * 100)

print(
    economic_summary
    .round(2)
    .to_string(index=False)
)

# Save results
from pathlib import Path

output_dir = (
    Path.home() / "Files/thesis/snow_ml/ml_model"
)

economic_scenarios.to_csv(
    output_dir / "economic_scenarios_county.csv",
    index=False
)

economic_summary.to_csv(
    output_dir / "economic_sensitivity_summary.csv",
    index=False
)

print("\nSaved economic scenario files.")

In [ ]:
import pandas as pd

# Rename the state column for consistency
orsa = orsa.rename(columns={"GeoName": "state"})

print("STATE COVERAGE")
print("=" * 60)

print(
    orsa.groupby("state")["year"]
    .agg(["min", "max", "count"])
    .to_string()
)

print("\n2023 SKI INDUSTRY VALUE ADDED")
print("=" * 70)

print(
    orsa.loc[
        orsa["year"] == 2023,
        [
            "state",
            "ski_snowboard_va",
            "snow_activities_va",
            "total_outdoor_va"
        ]
    ].to_string(index=False)
)

print("\nECONOMIC DATA SUMMARY")
print("=" * 70)

print(
    orsa[
        [
            "ski_snowboard_va",
            "snow_activities_va",
            "total_outdoor_va"
        ]
    ].describe().round(2)
)

In [ ]:
# Map state FIPS codes to state names
state_fips_map = {
    8: "Colorado",
    16: "Idaho",
    30: "Montana",
    49: "Utah",
    56: "Wyoming"
}

# Convert economic scenario state codes to names
economic_scenarios["state"] = (
    pd.to_numeric(economic_scenarios["state"], errors="coerce")
    .map(state_fips_map)
)

# Check the conversion
print("Economic scenario states:")
print(economic_scenarios["state"].value_counts(dropna=False))

print("\nORSA states:")
print(orsa["state"].unique())

In [ ]:
import pandas as pd
import numpy as np

# 2023 economic baseline
orsa_2023 = (
    orsa.loc[
        orsa["year"] == 2023,
        ["state", "ski_snowboard_va"]
    ]
    .copy()
)

# Assume BEA data are in thousands of dollars
orsa_2023["baseline_va_millions"] = (
    orsa_2023["ski_snowboard_va"] / 1000
)

# Average modeled demand changes across
# historical county-years within each state
state_demand = (
    economic_scenarios
    .groupby(
        ["state", "Model", "Scenario",
         "Demand assumption"],
        as_index=False
    )
    .agg(
        demand_change_pct=(
            "Demand change (%)", "mean"
        )
    )
)

# Join state economic baseline
state_impacts = state_demand.merge(
    orsa_2023[
        ["state", "baseline_va_millions"]
    ],
    on="state",
    how="left",
    validate="many_to_one"
)

assert state_impacts["baseline_va_millions"].notna().all(), (
    "Some states did not match the ORSA data."
)

# Convert demand changes into value-added changes
state_impacts["va_change_millions"] = (
    state_impacts["baseline_va_millions"]
    * state_impacts["demand_change_pct"]
    / 100
)

state_impacts["scenario_va_millions"] = (
    state_impacts["baseline_va_millions"]
    + state_impacts["va_change_millions"]
)

# Aggregate across the five states
total_impacts = (
    state_impacts
    .groupby(
        ["Model", "Scenario", "Demand assumption"],
        as_index=False
    )
    .agg(
        total_va_change_millions=(
            "va_change_millions", "sum"
        ),
        total_baseline_va_millions=(
            "baseline_va_millions", "sum"
        )
    )
)

total_impacts["total_change_pct"] = (
    100
    * total_impacts["total_va_change_millions"]
    / total_impacts["total_baseline_va_millions"]
)

# Show the medium-elasticity results
medium_results = total_impacts.loc[
    (total_impacts["Demand assumption"] == "Medium response")
    & (total_impacts["Scenario"] != "Baseline")
].copy()

print("PRELIMINARY ECONOMIC IMPACTS")
print("Medium assumed elasticity = 0.50")
print("=" * 90)

print(
    medium_results[
        [
            "Model",
            "Scenario",
            "total_va_change_millions",
            "total_change_pct"
        ]
    ]
    .round(2)
    .to_string(index=False)
)

In [ ]:
# State-level results for medium elasticity
# and the PDD +100 scenario

state_breakdown = state_impacts.loc[
    (state_impacts["Demand assumption"] == "Medium response")
    & (state_impacts["Scenario"] == "PDD +100")
].copy()

state_breakdown = state_breakdown[
    [
        "state",
        "Model",
        "baseline_va_millions",
        "demand_change_pct",
        "va_change_millions"
    ]
].sort_values(["Model", "va_change_millions"])

print("STATE-LEVEL ECONOMIC IMPACTS")
print("=" * 90)

print(
    state_breakdown
    .round(2)
    .to_string(index=False)
)

print("\nCOUNTY COVERAGE BY STATE")
print("=" * 50)

print(
    model_df.groupby("state")["fips"]
    .nunique()
    .to_string()
)

In [ ]:
# Identify the counties included in the snow model

county_coverage = (
    model_df[
        ["state", "county", "fips", "mean_elev_m",
         "frac_above_2000m"]
    ]
    .drop_duplicates(subset=["fips"])
    .copy()
)

state_names = {
    8: "Colorado",
    16: "Idaho",
    30: "Montana",
    49: "Utah",
    56: "Wyoming"
}

county_coverage["state_name"] = (
    pd.to_numeric(
        county_coverage["state"],
        errors="coerce"
    ).map(state_names)
)

county_coverage = county_coverage.sort_values(
    ["state_name", "county"]
)

print("COUNTIES INCLUDED IN SNOW MODEL")
print("=" * 90)

print(
    county_coverage[
        [
            "state_name",
            "county",
            "fips",
            "mean_elev_m",
            "frac_above_2000m"
        ]
    ]
    .round(3)
    .to_string(index=False)
)

In [ ]:
# County-level economic responses
# PDD +100, medium assumed elasticity

county_comparison = (
    economic_scenarios.loc[
        (economic_scenarios["Scenario"] == "PDD +100")
        & (
            economic_scenarios["Demand assumption"]
            == "Medium response"
        )
    ]
    .groupby(
        ["state", "county", "fips", "Model"],
        as_index=False
    )
    .agg(
        demand_change_pct=(
            "Demand change (%)", "mean"
        )
    )
)

# Show counties side by side across models
county_pivot = county_comparison.pivot_table(
    index=["state", "county", "fips"],
    columns="Model",
    values="demand_change_pct"
).reset_index()

print("COUNTY-LEVEL DEMAND RESPONSE (%)")
print("=" * 100)

print(
    county_pivot
    .round(2)
    .to_string(index=False)
)

# Check within-state variation
state_variation = (
    county_comparison
    .groupby(["state", "Model"])
    ["demand_change_pct"]
    .agg(["min", "max", "mean", "std"])
    .round(2)
)

print("\nWITHIN-STATE VARIATION")
print("=" * 80)
print(state_variation.to_string())

In [ ]:
# Geographic sensitivity test:
# Compare all counties against a restricted sample

excluded_fips = ["08067", "49003"]

def calculate_state_impacts(data, label):

    state_demand = (
        data.groupby(
            ["state", "Model", "Scenario",
             "Demand assumption"],
            as_index=False
        )
        .agg(
            demand_change_pct=(
                "Demand change (%)", "mean"
            )
        )
    )

    state_demand = state_demand.merge(
        orsa_2023[
            ["state", "baseline_va_millions"]
        ],
        on="state",
        how="left",
        validate="many_to_one"
    )

    state_demand["va_change_millions"] = (
        state_demand["baseline_va_millions"]
        * state_demand["demand_change_pct"] / 100
    )

    totals = (
        state_demand.groupby(
            ["Model", "Scenario", "Demand assumption"],
            as_index=False
        )["va_change_millions"]
        .sum()
    )

    totals["Sample"] = label

    return totals


full_sample = calculate_state_impacts(
    economic_scenarios,
    "All counties"
)

restricted_sample = calculate_state_impacts(
    economic_scenarios.loc[
        ~economic_scenarios["fips"]
        .astype(str).str.zfill(5)
        .isin(excluded_fips)
    ],
    "Excluding La Plata and Box Elder"
)

geographic_sensitivity = pd.concat(
    [full_sample, restricted_sample],
    ignore_index=True
)

comparison = geographic_sensitivity.loc[
    (geographic_sensitivity["Scenario"] == "PDD +100")
    & (
        geographic_sensitivity["Demand assumption"]
        == "Medium response"
    )
].pivot_table(
    index="Model",
    columns="Sample",
    values="va_change_millions"
)

comparison["Difference ($M)"] = (
    comparison["Excluding La Plata and Box Elder"]
    - comparison["All counties"]
)

print("GEOGRAPHIC SENSITIVITY ANALYSIS")
print("=" * 90)

print(comparison.round(2).to_string())

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Use the existing state-level ORSA panel
historical = orsa.copy()

# Keep valid observations
historical = historical.dropna(
    subset=[
        "ski_snowboard_va",
        "snow_days_10cm"
    ]
)

historical = historical.loc[
    (historical["ski_snowboard_va"] > 0)
    & (historical["snow_days_10cm"] > 0)
].copy()

# Log variables
historical["log_ski_va"] = np.log(
    historical["ski_snowboard_va"]
)

historical["log_snow_days"] = np.log(
    historical["snow_days_10cm"]
)

# Examine within-state correlations
print("SNOW DURATION AND SKI VALUE ADDED")
print("=" * 65)

print(
    historical.groupby("state")
    .apply(
        lambda x: x["log_snow_days"].corr(
            x["log_ski_va"]
        ),
        include_groups=False
    )
    .round(3)
)

# Plot historical relationship by state
fig, ax = plt.subplots(figsize=(9, 6))

for state, group in historical.groupby("state"):
    ax.scatter(
        group["snow_days_10cm"],
        group["ski_snowboard_va"] / 1000,
        label=state,
        alpha=0.75
    )

ax.set_xlabel("Snow-cover days (10 cm threshold)")
ax.set_ylabel("Ski/snowboard value added ($ millions)")
ax.set_title("Historical Snow Conditions and Ski Industry Value Added")
ax.legend()
ax.grid(alpha=0.25)

plt.tight_layout()
plt.show()

In [ ]:
import statsmodels.formula.api as smf
import numpy as np

historical = orsa.copy()

historical = historical.dropna(
    subset=["state", "year",
            "ski_snowboard_va", "snow_days_10cm"]
)

historical = historical.loc[
    (historical["ski_snowboard_va"] > 0)
    & (historical["snow_days_10cm"] > 0)
].copy()

historical["log_ski_va"] = np.log(
    historical["ski_snowboard_va"]
)

historical["log_snow_days"] = np.log(
    historical["snow_days_10cm"]
)

# Model 1: State fixed effects
model_state = smf.ols(
    "log_ski_va ~ log_snow_days + C(state)",
    data=historical
).fit(cov_type="HC3")

# Model 2: State and year fixed effects
model_twfe = smf.ols(
    "log_ski_va ~ log_snow_days + C(state) + C(year)",
    data=historical
).fit(cov_type="HC3")

print("HISTORICAL SNOW–ECONOMY REGRESSION")
print("=" * 80)

for name, model in [
    ("State FE", model_state),
    ("State + Year FE", model_twfe)
]:
    print(f"\n{name}")
    print(
        "Snow elasticity:",
        round(model.params["log_snow_days"], 4)
    )
    print(
        "P-value:",
        round(model.pvalues["log_snow_days"], 4)
    )
    print(
        "95% confidence interval:",
        model.conf_int()
        .loc["log_snow_days"]
        .round(4)
        .tolist()
    )
    print(
        "R-squared:",
        round(model.rsquared, 4)
    )

## NSAA Data

In [ ]:
import pandas as pd
import numpy as np

nsaa = pd.DataFrame({
    "year": list(range(2004, 2026)),
    "skier_visits_millions": [
        18.868, 19.606, 20.717, 20.849,
        21.324, 19.974, 20.378, 20.900,
        19.130, 19.800, 21.100, 20.768,
        22.287, 21.736, 20.792, 24.408,
        20.107, 22.638, 25.250, 28.174,
        26.757, 26.503
    ]
})

# year = ending year of the ski season
# Example: 2023 means 2022/23

print(nsaa.tail())

In [ ]:
# Create regional snow-cover proxy
regional_snow = (
    orsa.groupby("year", as_index=False)
    .agg(
        snow_days_10cm=("snow_days_10cm", "mean"),
        snow_days_5cm=("snow_days_5cm", "mean"),
        mean_depth_cm=("mean_depth_cm", "mean")
    )
)

# Merge with NSAA skier visits
regional_panel = nsaa.merge(
    regional_snow,
    on="year",
    how="inner",
    validate="one_to_one"
)

print("MATCHED SNOW AND SKIER VISIT DATA")
print("=" * 75)
print(regional_panel.to_string(index=False))
print("\nObservations:", len(regional_panel))

In [ ]:
import statsmodels.formula.api as smf

reg = regional_panel.copy()

reg = reg.loc[
    (reg["snow_days_10cm"] > 0)
    & (reg["skier_visits_millions"] > 0)
].copy()

reg["log_visits"] = np.log(
    reg["skier_visits_millions"]
)

reg["log_snow"] = np.log(
    reg["snow_days_10cm"]
)

reg["time_trend"] = reg["year"] - reg["year"].min()

# Without time trend
m1 = smf.ols(
    "log_visits ~ log_snow",
    data=reg
).fit(cov_type="HC3")

# With time trend
m2 = smf.ols(
    "log_visits ~ log_snow + time_trend",
    data=reg
).fit(cov_type="HC3")

print("NSAA SKIER VISITS VS SNOW DURATION")
print("=" * 75)

for name, model in [
    ("Snow only", m1),
    ("Snow + trend", m2)
]:
    print(f"\n{name}")
    print("Elasticity:", round(model.params["log_snow"], 4))
    print("P-value:", round(model.pvalues["log_snow"], 4))
    print(
        "95% CI:",
        model.conf_int()
        .loc["log_snow"]
        .round(4)
        .tolist()
    )
    print("R²:", round(model.rsquared, 4))

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf

# Create regional snow series using all available county-years
regional_snow_full = (
    model_df
    .groupby("year", as_index=False)
    .agg(
        snow_days_10cm=(
            "snow_days_10cm_2000m", "mean"
        ),
        snow_days_5cm=(
            "snow_days_5cm_2000m", "mean"
        ),
        mean_depth_cm=(
            "mean_depth_cm_2000m", "mean"
        )
    )
)

# Match NSAA visits to the full snow series
regional_full = nsaa.merge(
    regional_snow_full,
    on="year",
    how="inner",
    validate="one_to_one"
)

regional_full = regional_full.dropna(
    subset=["snow_days_10cm", "skier_visits_millions"]
)

regional_full = regional_full.loc[
    (regional_full["snow_days_10cm"] > 0)
    & (regional_full["skier_visits_millions"] > 0)
].copy()

regional_full["log_visits"] = np.log(
    regional_full["skier_visits_millions"]
)

regional_full["log_snow"] = np.log(
    regional_full["snow_days_10cm"]
)

regional_full["time_trend"] = (
    regional_full["year"]
    - regional_full["year"].min()
)

# COVID-disrupted seasons (ending years)
regional_full["covid"] = (
    regional_full["year"].isin([2020, 2021])
).astype(int)

models = {
    "Snow only": "log_visits ~ log_snow",
    "Snow + trend": "log_visits ~ log_snow + time_trend",
    "Snow + trend + COVID": (
        "log_visits ~ log_snow + time_trend + covid"
    )
}

print("EXTENDED NSAA–SNODAS REGRESSIONS")
print("=" * 85)

for name, formula in models.items():

    fitted = smf.ols(
        formula,
        data=regional_full
    ).fit(cov_type="HC3")

    print(f"\n{name}")
    print("N:", int(fitted.nobs))
    print(
        "Snow elasticity:",
        round(fitted.params["log_snow"], 4)
    )
    print(
        "P-value:",
        round(fitted.pvalues["log_snow"], 4)
    )
    print(
        "95% CI:",
        fitted.conf_int()
        .loc["log_snow"]
        .round(4)
        .tolist()
    )
    print("R²:", round(fitted.rsquared, 4))

# Additional robustness check:
# Exclude COVID-disrupted seasons entirely
noncovid = regional_full.loc[
    ~regional_full["year"].isin([2020, 2021])
].copy()

fitted_noncovid = smf.ols(
    "log_visits ~ log_snow + time_trend",
    data=noncovid
).fit(cov_type="HC3")

print("\nSnow + trend, excluding COVID seasons")
print("N:", int(fitted_noncovid.nobs))
print(
    "Snow elasticity:",
    round(fitted_noncovid.params["log_snow"], 4)
)
print(
    "P-value:",
    round(fitted_noncovid.pvalues["log_snow"], 4)
)
print(
    "95% CI:",
    fitted_noncovid.conf_int()
    .loc["log_snow"]
    .round(4)
    .tolist()
)

In [ ]:
import numpy as np
import pandas as pd

# Provisional NSAA elasticity:
# Snow + trend + COVID specification
nsaa_elasticity = 0.3816

# Start from existing county-level snow scenarios
nsaa_econ = county_scenarios.copy()

# Ensure state identifiers are consistent
state_map = {
    8: "Colorado",
    16: "Idaho",
    30: "Montana",
    49: "Utah",
    56: "Wyoming"
}

nsaa_econ["state"] = (
    pd.to_numeric(
        nsaa_econ["state"],
        errors="coerce"
    ).map(state_map)
)

# Keep valid baseline observations
nsaa_econ = nsaa_econ.loc[
    nsaa_econ["Baseline snow days"] > 0
].copy()

# Calculate snow-duration ratios
nsaa_econ["snow_ratio"] = (
    nsaa_econ["Scenario snow days"]
    / nsaa_econ["Baseline snow days"]
)

# Apply empirically estimated elasticity
nsaa_econ["demand_change_pct"] = (
    nsaa_econ["snow_ratio"].clip(lower=0)
    ** nsaa_elasticity - 1
) * 100

# Average demand changes by state/model/scenario
state_nsaa = (
    nsaa_econ.groupby(
        ["state", "Model", "Scenario"],
        as_index=False
    )
    .agg(
        demand_change_pct=(
            "demand_change_pct", "mean"
        )
    )
)

# Add state-level BEA economic baseline
state_nsaa = state_nsaa.merge(
    orsa_2023[
        ["state", "baseline_va_millions"]
    ],
    on="state",
    how="left",
    validate="many_to_one"
)

assert state_nsaa["baseline_va_millions"].notna().all()

# Estimate economic changes
state_nsaa["va_change_millions"] = (
    state_nsaa["baseline_va_millions"]
    * state_nsaa["demand_change_pct"] / 100
)

# Aggregate five-state economic impacts
nsaa_total = (
    state_nsaa.groupby(
        ["Model", "Scenario"],
        as_index=False
    )
    .agg(
        va_change_millions=(
            "va_change_millions", "sum"
        )
    )
)

print("NSAA-CALIBRATED ECONOMIC SCENARIOS")
print("=" * 80)

print(
    nsaa_total.loc[
        nsaa_total["Scenario"] != "Baseline"
    ]
    .round(2)
    .to_string(index=False)
)

In [ ]:
import pandas as pd
import numpy as np

elasticity_cases = {
    "Zero response": 0.0,
    "NSAA central": 0.3816,
    "NSAA upper 95%": 0.8950
}

results = []

for label, elasticity in elasticity_cases.items():

    temp = county_scenarios.copy()

    state_map = {
        8: "Colorado",
        16: "Idaho",
        30: "Montana",
        49: "Utah",
        56: "Wyoming"
    }

    temp["state"] = (
        pd.to_numeric(temp["state"], errors="coerce")
        .map(state_map)
    )

    temp = temp.loc[
        temp["Baseline snow days"] > 0
    ].copy()

    snow_ratio = (
        temp["Scenario snow days"]
        / temp["Baseline snow days"]
    ).clip(lower=0)

    temp["demand_change_pct"] = (
        snow_ratio ** elasticity - 1
    ) * 100

    state_results = (
        temp.groupby(
            ["state", "Model", "Scenario"],
            as_index=False
        )["demand_change_pct"]
        .mean()
    )

    state_results = state_results.merge(
        orsa_2023[
            ["state", "baseline_va_millions"]
        ],
        on="state",
        how="left",
        validate="many_to_one"
    )

    state_results["va_change_millions"] = (
        state_results["baseline_va_millions"]
        * state_results["demand_change_pct"] / 100
    )

    totals = (
        state_results.groupby(
            ["Model", "Scenario"],
            as_index=False
        )["va_change_millions"]
        .sum()
    )

    totals["Elasticity case"] = label
    results.append(totals)

elasticity_uncertainty = pd.concat(
    results, ignore_index=True
)

print("NSAA ELASTICITY SENSITIVITY")
print("=" * 85)

print(
    elasticity_uncertainty.loc[
        elasticity_uncertainty["Scenario"] == "PDD +100"
    ]
    .round(2)
    .to_string(index=False)
)

In [ ]:
from pathlib import Path

root = Path.home() / "Files/thesis/snow_ml"

print("Folder exists:", root.exists())

keywords = [
    "season_days",
    "snow_days_10cm",
    "season_start",
    "season_end",
    "water_year",
    "datetime",
    "date_range"
]

matches = []

for path in root.rglob("*"):
    if path.suffix not in [".py", ".ipynb"]:
        continue

    try:
        content = path.read_text(errors="ignore")
    except Exception:
        continue

    found = [k for k in keywords if k in content]

    if found:
        matches.append((path, found))

print("\nMatching files:", len(matches))

for path, found in matches[:30]:
    print(f"\n{path.relative_to(root)}")
    print("Keywords:", ", ".join(found))

In [ ]:
import json
from pathlib import Path

notebook_path = (
    Path.home()
    / "Files/thesis/snow_ml/code/snow_model.ipynb"
)

with open(notebook_path, "r") as f:
    nb = json.load(f)

keywords = [
    "season_start",
    "season_end",
    "water_year",
    "date_range"
]

for i, cell in enumerate(nb["cells"]):
    if cell["cell_type"] != "code":
        continue

    source = "".join(cell["source"])

    if any(k in source for k in keywords):
        print(f"\n{'=' * 70}")
        print(f"CELL {i}")
        print("=" * 70)
        print(source[:3000])

In [ ]:
import json
from pathlib import Path

notebook_path = (
    Path.home()
    / "Files/thesis/snow_ml/code/snow_model.ipynb"
)

with open(notebook_path, "r") as f:
    nb = json.load(f)

keywords = [
    "read_csv",
    "read_parquet",
    "read_excel",
    "model_df =",
    "county_snodas",
    "snodas_panel",
]

for i, cell in enumerate(nb["cells"]):
    if cell["cell_type"] != "code":
        continue

    source = "".join(cell["source"])

    if any(k in source.lower() for k in keywords):
        print(f"\n{'=' * 70}")
        print(f"CELL {i}")
        print("=" * 70)
        print(source[:2000])

In [ ]:
from pathlib import Path
import pandas as pd

path = (
    Path.home()
    / "Files/thesis/snow_ml/ml_model/county_snodas_panel_v1.csv"
)

snow_check = pd.read_csv(path)

print("COLUMNS:")
print(snow_check.columns.tolist())

print("\n2023 OBSERVATIONS:")
print(
    snow_check.loc[
        snow_check["year"] == 2023
    ].head(5).to_string(index=False)
)

print("\nSEASON LENGTH:")
print(
    snow_check.groupby("year")["season_days"]
    .first()
    .to_string()
)

In [ ]:
from pathlib import Path

root = Path.home() / "Files/thesis/snow_ml"

terms = [
    "county_snodas_panel_v1.csv",
    "season_start",
    "season_end",
    "november",
    "april"
]

for path in root.rglob("*"):
    if not path.is_file():
        continue

    if any(part in ["venv", ".venv", ".git"] for part in path.parts):
        continue

    if path.suffix not in [".py", ".ipynb"]:
        continue

    try:
        content = path.read_text(errors="ignore")
    except Exception:
        continue

    found = [term for term in terms if term.lower() in content.lower()]

    if found:
        print(path.relative_to(root))
        print("Matches:", ", ".join(found))
        print()

In [ ]:
import json
from pathlib import Path

root = Path.home() / "Files/thesis/snow_ml"

notebooks = [
    root / "code/snow_model.ipynb",
    root / "code/gdp_snow.ipynb",
    root / "code/FDOS_annotated.ipynb"
]

terms = [
    "season_start",
    "season_end",
    "water_year",
    "11, 1",
    "4, 30",
    "11-01",
    "04-30",
    "november",
    "april"
]

for path in notebooks:
    with open(path, encoding="utf-8") as f:
        nb = json.load(f)

    print(f"\n{'=' * 70}")
    print(path.name)
    print("=" * 70)

    for i, cell in enumerate(nb["cells"]):
        source = "".join(cell.get("source", []))

        # Skip our recent diagnostic cells
        if "keywords = [" in source or "terms = [" in source:
            continue

        matching_lines = [
            (j, line)
            for j, line in enumerate(source.splitlines())
            if any(t in line.lower() for t in terms)
        ]

        if matching_lines:
            print(f"\nCELL {i}")
            for line_number, line in matching_lines[:12]:
                print(f"  Line {line_number + 1}: {line[:180]}")

In [ ]:
import json
from pathlib import Path

path = Path.home() / "Files/thesis/snow_ml/code/snow_model.ipynb"

with open(path, encoding="utf-8") as f:
    nb = json.load(f)

print("CELL 101 — SNODAS DATE DEFINITION")
print("=" * 70)
print("".join(nb["cells"][101]["source"]))

In [ ]:
import json
from pathlib import Path

root = Path.home() / "Files/thesis/snow_ml"

# Search project notebooks and scripts, excluding virtual environments
terms = [
    "season_days =",
    '"season_days":',
    "'season_days':",
    "season_days.append",
    "county_snodas_panel_v1.csv"
]

for path in root.rglob("*"):
    if not path.is_file():
        continue

    if any(p in {"venv", ".venv", ".git"} for p in path.parts):
        continue

    if path.suffix not in {".py", ".ipynb"}:
        continue

    try:
        if path.suffix == ".ipynb":
            with open(path, encoding="utf-8") as f:
                notebook = json.load(f)

            sources = [
                (i, "".join(cell.get("source", [])))
                for i, cell in enumerate(notebook["cells"])
                if cell["cell_type"] == "code"
            ]
        else:
            sources = [(None, path.read_text(errors="ignore"))]

        for cell_num, source in sources:
            lines = source.splitlines()

            for i, line in enumerate(lines):
                if any(term in line for term in terms):
                    # Skip diagnostic/search code
                    if "terms =" in source or "keywords =" in source:
                        continue

                    print("\n" + "=" * 65)
                    print("FILE:", path.relative_to(root))
                    if cell_num is not None:
                        print("CELL:", cell_num)

                    print("\n".join(
                        lines[max(0, i-5):min(len(lines), i+6)]
                    ))

    except Exception as e:
        print("Error:", path, e)

In [ ]:
from pathlib import Path

root = Path.home() / "Files/thesis/snow_ml"

for folder in [root / "ml_model", root / "code"]:
    print(f"\n{'=' * 65}")
    print("FOLDER:", folder)
    print("=" * 65)

    if not folder.exists():
        continue

    for path in sorted(folder.rglob("*")):
        if not path.is_file():
            continue

        if any(p in {"venv", ".venv", ".git", "__pycache__"}
               for p in path.parts):
            continue

        if path.suffix.lower() not in {
            ".py", ".r", ".ipynb", ".sh"
        }:
            continue

        name = path.name.lower()

        if any(term in name for term in [
            "snodas", "snow", "county", "panel", "extract"
        ]):
            print(path.relative_to(root))

In [ ]:
print("FIXED EFFECTS MODEL")
print("=" * 60)
print(fe_model)

print("\nRANDOM FOREST FEATURES")
print("=" * 60)
print(final_rf.feature_names_in_
      if hasattr(final_rf, "feature_names_in_")
      else "Feature names not stored in model")

print("\nXGBOOST FEATURES")
print("=" * 60)
print(xgb.feature_names_in_
      if hasattr(xgb, "feature_names_in_")
      else "Feature names not stored in model")

print("\nAVAILABLE TEMPERATURE VARIABLES")
print("=" * 60)

temp_cols = [
    c for c in model_df.columns
    if any(term in c.lower() for term in [
        "temp", "tmin", "tmax", "degree", "freeze", "thaw"
    ])
]

print(temp_cols)

In [ ]:
import json
from pathlib import Path

root = Path.home() / "Files/thesis/snow_ml"

terms = [
    "positive_degree_days",
    "freezing_degree_days",
    "thaw_days"
]

for path in root.rglob("*"):

    if not path.is_file():
        continue

    if any(p in {"venv", ".venv", ".git"}
           for p in path.parts):
        continue

    if path.suffix not in {".py", ".ipynb"}:
        continue

    if path.suffix == ".ipynb":
        with open(path, encoding="utf-8") as f:
            nb = json.load(f)

        sources = [
            (i, "".join(cell.get("source", [])))
            for i, cell in enumerate(nb["cells"])
            if cell["cell_type"] == "code"
        ]
    else:
        sources = [(None, path.read_text(errors="ignore"))]

    for cell_num, source in sources:

        lines = source.splitlines()

        for i, line in enumerate(lines):

            # Find assignments/calculations, not just column references
            if not any(term in line for term in terms):
                continue

            if not any(op in line for op in [
                "=", "np.maximum", "np.minimum", "clip"
            ]):
                continue

            # Ignore recent diagnostic cells
            if "terms = [" in source:
                continue

            print("\n" + "=" * 65)
            print("FILE:", path.relative_to(root))
            print("CELL:", cell_num)
            print("\n".join(
                lines[max(0, i-3):min(len(lines), i+5)]
            ))

In [ ]:
from pathlib import Path

root = Path.home() / "Files/thesis/snow_ml"

extensions = {".csv", ".parquet", ".nc", ".nc4", ".zarr"}

climate_files = []

for path in root.rglob("*"):
    if any(p in {"venv", ".venv", ".git"} for p in path.parts):
        continue

    if path.is_file() and path.suffix.lower() in extensions:
        if any(term in path.name.lower() for term in [
            "climate", "temp", "daily", "daymet", "gridmet",
            "era5", "prism", "degree", "weather"
        ]):
            climate_files.append(path)

print("POTENTIAL CLIMATE DATA FILES")
print("=" * 70)

for path in sorted(climate_files):
    print(path.relative_to(root))

print("\nTotal:", len(climate_files))

In [ ]:
import pandas as pd
from pathlib import Path

root = Path.home() / "Files/thesis/snow_ml/ml_model"

files = [
    "county_climate_maca_projections.csv",
    "county_climate_panel_v3.csv"
]

for filename in files:
    path = root / filename

    print("\n" + "=" * 75)
    print("FILE:", filename)
    print("=" * 75)

    df_check = pd.read_csv(path, nrows=5)

    print("\nCOLUMNS:")
    print(df_check.columns.tolist())

    print("\nFIRST 3 ROWS:")
    print(df_check.head(3).to_string(index=False))

    print("\nFILE SIZE:")
    print(round(path.stat().st_size / 1e6, 2), "MB")

In [ ]:
import pandas as pd
from pathlib import Path

path = (
    Path.home()
    / "Files/thesis/snow_ml/ml_model/"
    "county_climate_maca_projections.csv"
)

maca = pd.read_csv(path, dtype={"fips": str})
maca["fips"] = maca["fips"].str.zfill(5)

print("MACA DATASET")
print("=" * 70)
print("Rows:", len(maca))
print("Counties:", maca["fips"].nunique())

print("\nSCENARIOS AND PERIODS")
print(
    maca[
        ["scenario", "period", "year_start", "year_end"]
    ]
    .drop_duplicates()
    .sort_values(["scenario", "year_start"])
    .to_string(index=False)
)

print("\nCOUNTIES PER SCENARIO AND PERIOD")
print(
    maca.groupby(["scenario", "period"])["fips"]
    .nunique()
    .to_string()
)

print("\nMISSING VALUES")
print(maca.isna().sum().to_string())

print("\nHISTORICAL BASELINE EXAMPLE")
print(
    maca[maca["period"] == "historical"]
    .head(3)
    .to_string(index=False)
)

In [ ]:
import pandas as pd

# Keep only counties included in the trained snow models
model_counties = set(
    model_df["fips"]
    .astype(str)
    .str.replace(r"\.0$", "", regex=True)
    .str.zfill(5)
)

maca_model = maca[
    maca["fips"].isin(model_counties)
].copy()

climate_vars = [
    "tmin_djf",
    "tmin_apr",
    "precip_on",
    "precip_djf"
]

# Historical baseline for each county and RCP
historical = maca_model[
    maca_model["period"] == "historical"
][["fips", "scenario"] + climate_vars].copy()

historical = historical.rename(
    columns={v: f"{v}_historical" for v in climate_vars}
)

# Future climate periods
future = maca_model[
    maca_model["period"] != "historical"
].copy()

future = future.merge(
    historical,
    on=["fips", "scenario"],
    how="left",
    validate="many_to_one"
)

# Temperature changes in degrees Celsius
for var in ["tmin_djf", "tmin_apr"]:
    future[f"delta_{var}"] = (
        future[var] - future[f"{var}_historical"]
    )

# Precipitation changes in percent
for var in ["precip_on", "precip_djf"]:
    future[f"pct_change_{var}"] = (
        100 * (
            future[var] / future[f"{var}_historical"] - 1
        )
    )

summary = (
    future.groupby(["scenario", "period"])
    .agg(
        counties=("fips", "nunique"),
        winter_warming_C=("delta_tmin_djf", "mean"),
        april_warming_C=("delta_tmin_apr", "mean"),
        oct_nov_precip_pct=("pct_change_precip_on", "mean"),
        winter_precip_pct=("pct_change_precip_djf", "mean")
    )
    .reset_index()
)

print("MACA CLIMATE CHANGES — 20 SNOW-MODEL COUNTIES")
print("=" * 85)
print(summary.round(2).to_string(index=False))

In [ ]:
import pandas as pd
import numpy as np

variables = [
    "tmin_djf_2000m",
    "tmax_djf_2000m",
    "tmin_mar_2000m",
    "tmax_mar_2000m",
    "tmin_apr_2000m",
    "tmax_apr_2000m",
    "positive_degree_days_2000m",
    "freezing_degree_days_2000m",
    "thaw_days_2000m"
]

climate_hist = model_df[
    ["fips", "year"] + variables
].dropna().copy()

print("HISTORICAL CLIMATE DATA")
print("=" * 70)
print("Observations:", len(climate_hist))
print("Counties:", climate_hist["fips"].nunique())
print("Years:", climate_hist["year"].min(),
      "-", climate_hist["year"].max())

print("\nTEMPERATURE AND DEGREE-DAY CORRELATIONS")
print("=" * 70)

corr = climate_hist[variables].corr()

print(
    corr.loc[
        [
            "positive_degree_days_2000m",
            "freezing_degree_days_2000m",
            "thaw_days_2000m"
        ],
        [
            "tmin_djf_2000m",
            "tmax_djf_2000m",
            "tmin_mar_2000m",
            "tmax_mar_2000m",
            "tmin_apr_2000m",
            "tmax_apr_2000m"
        ]
    ].round(3).to_string()
)

print("\nHISTORICAL PREDICTOR RANGES")
print("=" * 70)

print(
    climate_hist[variables]
    .agg(["min", "mean", "max"])
    .round(2)
    .to_string()
)

In [ ]:
import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score, mean_absolute_error

# Historical observations
hist = model_df.copy()

hist["fips"] = (
    hist["fips"]
    .astype(str)
    .str.replace(r"\.0$", "", regex=True)
    .str.zfill(5)
)

# Predictors available in MACA
predictors = [
    "fips",
    "tmin_djf_2000m",
    "tmin_apr_2000m"
]

# Variables needed by RF and XGBoost but missing from MACA
targets = [
    "tmax_djf_2000m",
    "tmin_mar_2000m",
    "tmax_mar_2000m",
    "tmax_apr_2000m",
    "positive_degree_days_2000m",
    "freezing_degree_days_2000m",
    "thaw_days_2000m"
]

hist = hist.dropna(
    subset=predictors + targets
).copy()

train = hist[hist["year"] <= 2018]
test = hist[hist["year"] >= 2019]

results = []

for target in targets:

    preprocessor = ColumnTransformer(
        transformers=[
            (
                "county",
                OneHotEncoder(handle_unknown="ignore"),
                ["fips"]
            ),
            (
                "temperature",
                "passthrough",
                [
                    "tmin_djf_2000m",
                    "tmin_apr_2000m"
                ]
            )
        ]
    )

    auxiliary_model = Pipeline([
        ("preprocessor", preprocessor),
        ("regression", Ridge(alpha=1.0))
    ])

    auxiliary_model.fit(
        train[predictors],
        train[target]
    )

    predictions = auxiliary_model.predict(
        test[predictors]
    )

    results.append({
        "Variable": target,
        "R2": r2_score(test[target], predictions),
        "MAE": mean_absolute_error(
            test[target], predictions
        ),
        "Observed SD": test[target].std()
    })

auxiliary_results = pd.DataFrame(results)

auxiliary_results["MAE / SD"] = (
    auxiliary_results["MAE"]
    / auxiliary_results["Observed SD"]
)

print("AUXILIARY CLIMATE MODEL VALIDATION")
print("=" * 85)
print("Training years: 2004–2018")
print("Testing years: 2019–2023")
print("Training observations:", len(train))
print("Testing observations:", len(test))

print(
    auxiliary_results
    .round(3)
    .to_string(index=False)
)

In [ ]:
import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge

# Use historical climate data
hist = model_df.copy()

hist["fips"] = (
    hist["fips"]
    .astype(str)
    .str.replace(r"\.0$", "", regex=True)
    .str.zfill(5)
)

predictors = [
    "fips",
    "tmin_djf_2000m",
    "tmin_apr_2000m"
]

targets = [
    "tmax_djf_2000m",
    "tmin_mar_2000m",
    "tmax_mar_2000m",
    "tmax_apr_2000m",
    "positive_degree_days_2000m",
    "freezing_degree_days_2000m",
    "thaw_days_2000m"
]

hist = hist.dropna(
    subset=predictors + targets
).copy()

train = hist[hist["year"] <= 2018].copy()
test = hist[hist["year"] >= 2019].copy()

results = []

for target in targets:

    preprocessor = ColumnTransformer([
        (
            "county",
            OneHotEncoder(handle_unknown="ignore"),
            ["fips"]
        ),
        (
            "temperature",
            "passthrough",
            [
                "tmin_djf_2000m",
                "tmin_apr_2000m"
            ]
        )
    ])

    model = Pipeline([
        ("preprocessor", preprocessor),
        ("regression", Ridge(alpha=1.0))
    ])

    model.fit(train[predictors], train[target])

    # Predict both historical periods
    train[f"pred_{target}"] = model.predict(
        train[predictors]
    )

    test[f"pred_{target}"] = model.predict(
        test[predictors]
    )

    # Average within county and period
    early = train.groupby("fips")[
        [target, f"pred_{target}"]
    ].mean()

    late = test.groupby("fips")[
        [target, f"pred_{target}"]
    ].mean()

    comparison = late - early

    actual_change = comparison[target]
    predicted_change = comparison[f"pred_{target}"]

    results.append({
        "Variable": target,
        "Actual mean change": actual_change.mean(),
        "Predicted mean change": predicted_change.mean(),
        "Change correlation": actual_change.corr(
            predicted_change
        ),
        "Change MAE": np.mean(
            np.abs(actual_change - predicted_change)
        )
    })

change_validation = pd.DataFrame(results)

print("HISTORICAL CLIMATE-CHANGE VALIDATION")
print("=" * 95)
print("Early period: 2004–2018")
print("Late period: 2019–2023")
print(
    change_validation
    .round(3)
    .to_string(index=False)
)

In [ ]:
from pathlib import Path

root = Path.home() / "Files/thesis/snow_ml"

for name in [
    "README.md",
    "README.txt",
    "requirements.txt",
    "environment.yml"
]:
    for path in root.rglob(name):
        print("\nFILE:", path.relative_to(root))
        print("-" * 60)
        print(path.read_text(errors="ignore")[:3000])

In [ ]:
from pathlib import Path

root = Path.home() / "Files/thesis"

extensions = {
    ".nc", ".nc4", ".grib", ".grb",
    ".tif", ".tiff", ".parquet",
    ".zarr", ".csv"
}

keywords = [
    "daymet", "gridmet", "prism",
    "maca", "daily", "temperature",
    "climate", "tmin", "tmax"
]

matches = []

for path in root.rglob("*"):

    if any(part in {
        "venv", ".venv", "site-packages",
        ".git", "__pycache__"
    } for part in path.parts):
        continue

    if not path.is_file():
        continue

    if path.suffix.lower() not in extensions:
        continue

    if any(word in path.name.lower() for word in keywords):
        matches.append(path)

print("POTENTIAL CLIMATE SOURCE FILES")
print("=" * 70)

for path in sorted(matches):
    print(path.relative_to(root))

print("\nTotal files:", len(matches))

In [ ]:
from pathlib import Path
import json

root = Path.home() / "Files/thesis/snow_ml"

keywords = [
    "NASA/ORNL/DAYMET",
    "IDAHO_EPSCOR",
    "GRIDMET",
    "ECMWF",
    "PRISM",
    "ee.ImageCollection",
    "positive_degree_days"
]

for path in (root / "code").glob("*.ipynb"):
    with open(path, encoding="utf-8") as f:
        nb = json.load(f)

    for i, cell in enumerate(nb["cells"]):
        if cell["cell_type"] != "code":
            continue

        source = "".join(cell.get("source", []))

        if any(k.lower() in source.lower() for k in keywords):
            print(f"\n{path.name} — Cell {i}")
            for line in source.splitlines():
                if any(k.lower() in line.lower() for k in keywords):
                    print(line[:200])

In [ ]:
%pip install earthengine-api

In [ ]:
import ee

try:
    ee.Initialize()
    print("Google Earth Engine initialized successfully.")
except Exception as e:
    print("Earth Engine initialization failed:")
    print(str(e))

In [ ]:
import ee

ee.Authenticate(auth_mode="localhost")

In [ ]:
import ee

ee.Initialize(project="fdos-491217")

print("Google Earth Engine initialized successfully!")

In [ ]:
daymet = ee.ImageCollection("NASA/ORNL/DAYMET_V4")

print("Daymet images:", daymet.size().getInfo())
print("Available bands:", daymet.first().bandNames().getInfo())

In [ ]:
import ee
import pandas as pd

ee.Initialize(project="fdos-491217")

# Routt County, Colorado
counties = ee.FeatureCollection("TIGER/2018/Counties")

routt = counties.filter(
    ee.Filter.eq("GEOID", "08107")
).first()

geometry = routt.geometry()

# Elevation data
dem = ee.Image("USGS/SRTMGL1_003")

# Above 2,000 meters
high_elevation = dem.gte(2000)

# Calculate area above 2,000 meters
area_image = ee.Image.pixelArea()

total_area = area_image.reduceRegion(
    reducer=ee.Reducer.sum(),
    geometry=geometry,
    scale=1000,
    maxPixels=1e9
).getInfo()

high_area = area_image.updateMask(
    high_elevation
).reduceRegion(
    reducer=ee.Reducer.sum(),
    geometry=geometry,
    scale=1000,
    maxPixels=1e9
).getInfo()

total_km2 = total_area["area"] / 1e6
high_km2 = high_area["area"] / 1e6

print("ROUTT COUNTY ELEVATION CHECK")
print("=" * 50)
print(f"Total area: {total_km2:,.1f} km²")
print(f"Area above 2,000m: {high_km2:,.1f} km²")
print(f"Fraction above 2,000m: {high_km2 / total_km2:.3f}")

In [ ]:
print(
    model_df.loc[
        model_df["fips"].astype(str).str.zfill(5) == "08107",
        ["county", "frac_above_2000m"]
    ].drop_duplicates().to_string(index=False)
)

In [ ]:
import ee
import pandas as pd

ee.Initialize(project="fdos-491217")

# Routt County
routt = (
    ee.FeatureCollection("TIGER/2018/Counties")
    .filter(ee.Filter.eq("GEOID", "08107"))
    .first()
)

geom = routt.geometry()

# Elevation mask
dem = ee.Image("USGS/SRTMGL1_003")
mask_2000m = dem.gte(2000)

# Daily Daymet data
daymet = (
    ee.ImageCollection("NASA/ORNL/DAYMET_V4")
    .filterDate("2021-11-01", "2022-05-01")
    .select(["tmin", "tmax"])
)

def daily_climate(image):
    # Daily mean temperature
    tmean = (
        image.select("tmin")
        .add(image.select("tmax"))
        .divide(2)
        .rename("tmean")
    )

    # Candidate degree-day definitions
    pdd = tmean.max(0).rename("pdd")
    fdd = tmean.min(0).abs().rename("fdd")
    thaw = tmean.gt(0).rename("thaw")

    output = (
        image
        .addBands([tmean, pdd, fdd, thaw])
        .updateMask(mask_2000m)
    )

    values = output.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=geom,
        scale=1000,
        maxPixels=1e9
    )

    return ee.Feature(None, values).set(
        "date",
        image.date().format("YYYY-MM-dd")
    )

daily_fc = daymet.map(daily_climate)

# Retrieve daily county averages
daily_data = pd.DataFrame([
    f["properties"]
    for f in daily_fc.getInfo()["features"]
])

daily_data["date"] = pd.to_datetime(daily_data["date"])

print("DAILY DATA")
print("=" * 60)
print("Days:", len(daily_data))
print("Start:", daily_data["date"].min())
print("End:", daily_data["date"].max())

print("\nRECONSTRUCTED CLIMATE VARIABLES")
print("=" * 60)

print("Positive degree days:", round(daily_data["pdd"].sum(), 2))
print("Freezing degree days:", round(daily_data["fdd"].sum(), 2))
print("Thaw days:", round(daily_data["thaw"].sum(), 2))

# Existing historical values
existing = model_df[
    (model_df["fips"].astype(str).str.zfill(5) == "08107")
    & (model_df["year"] == 2022)
]

print("\nEXISTING MODEL VALUES")
print("=" * 60)

print(
    existing[
        [
            "positive_degree_days_2000m",
            "freezing_degree_days_2000m",
            "thaw_days_2000m"
        ]
    ].to_string(index=False)
)

print("\nSEASONAL TEMPERATURES")
print("=" * 60)

for label, months in [
    ("DJF", [12, 1, 2]),
    ("March", [3]),
    ("April", [4])
]:
    subset = daily_data[
        daily_data["date"].dt.month.isin(months)
    ]

    print(
        label,
        "Tmin:", round(subset["tmin"].mean(), 2),
        "Tmax:", round(subset["tmax"].mean(), 2)
    )

In [ ]:
# Compare possible thaw-day definitions

thaw_mean = daily_data["thaw"].sum()

# Daily maximum temperature above freezing
# Note: this applies the threshold to the county-mean Tmax,
# so it is a diagnostic, not an exact pixel-level reconstruction.
thaw_tmax = (daily_data["tmax"] > 0).sum()

# Daily minimum temperature above freezing
thaw_tmin = (daily_data["tmin"] > 0).sum()

original_thaw = existing["thaw_days_2000m"].iloc[0]

print("THAW-DAY DEFINITION CHECK")
print("=" * 55)

print(f"Original dataset:           {original_thaw:.2f}")
print(f"Daily mean > 0°C:           {thaw_mean:.2f}")
print(f"County-mean Tmax > 0°C:     {thaw_tmax:.2f}")
print(f"County-mean Tmin > 0°C:     {thaw_tmin:.2f}")

In [ ]:
# Thaw day = daily maximum temperature > 0°C
# Calculate at each pixel, then average across pixels above 2,000m

def calculate_thaw(image):

    thaw = (
        image.select("tmax")
        .gt(0)
        .rename("thaw_tmax")
        .updateMask(mask_2000m)
    )

    value = thaw.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=geom,
        scale=1000,
        maxPixels=1e9
    ).get("thaw_tmax")

    return ee.Feature(None, {
        "date": image.date().format("YYYY-MM-dd"),
        "thaw_fraction": value
    })

thaw_fc = daymet.map(calculate_thaw)

thaw_daily = pd.DataFrame([
    f["properties"]
    for f in thaw_fc.getInfo()["features"]
])

reconstructed_thaw = thaw_daily["thaw_fraction"].sum()
original_thaw = existing["thaw_days_2000m"].iloc[0]

print("THAW-DAY VALIDATION — ROUTT COUNTY, 2022")
print("=" * 55)
print(f"Original thaw days:      {original_thaw:.3f}")
print(f"Reconstructed thaw days: {reconstructed_thaw:.3f}")
print(f"Difference:              {reconstructed_thaw - original_thaw:+.3f}")
print(f"Percent difference:      {100 * (reconstructed_thaw / original_thaw - 1):+.3f}%")

In [ ]:
# Second validation: Teton County, WY — 2015

test_fips = "56039"
test_year = 2015

test_county = (
    ee.FeatureCollection("TIGER/2018/Counties")
    .filter(ee.Filter.eq("GEOID", test_fips))
    .first()
)

test_geom = test_county.geometry()

test_daymet = (
    ee.ImageCollection("NASA/ORNL/DAYMET_V4")
    .filterDate(f"{test_year-1}-11-01", f"{test_year}-05-01")
    .select(["tmin", "tmax"])
)

def climate_bands(img):
    tmean = img.select("tmin").add(img.select("tmax")).divide(2)

    return ee.Image.cat([
        tmean.max(0).rename("pdd"),
        tmean.min(0).abs().rename("fdd"),
        img.select("tmax").gt(0).rename("thaw")
    ])

# Sum daily values at each pixel, then average over high-elevation county area
season_totals = (
    test_daymet
    .map(climate_bands)
    .sum()
    .updateMask(mask_2000m)
)

reconstructed = season_totals.reduceRegion(
    reducer=ee.Reducer.mean(),
    geometry=test_geom,
    scale=1000,
    maxPixels=1e9
).getInfo()

original = model_df.loc[
    (model_df["fips"].astype(str).str.zfill(5) == test_fips)
    & (model_df["year"] == test_year)
].iloc[0]

print("VALIDATION — TETON COUNTY, WY, 2015")
print("=" * 65)

for reconstructed_var, original_var in [
    ("pdd", "positive_degree_days_2000m"),
    ("fdd", "freezing_degree_days_2000m"),
    ("thaw", "thaw_days_2000m")
]:
    actual = original[original_var]
    predicted = reconstructed[reconstructed_var]

    print(
        f"{original_var:32s} "
        f"Original: {actual:8.2f}  "
        f"Reconstructed: {predicted:8.2f}  "
        f"Difference: {predicted-actual:+7.2f}"
    )

In [ ]:
# Teton County, WY — 2015
# Uniform warming scenarios: +0, +1, +2, +3°C

def warming_bands(img, warming):
    tmin = img.select("tmin").add(warming)
    tmax = img.select("tmax").add(warming)
    tmean = tmin.add(tmax).divide(2)

    return ee.Image.cat([
        tmean.max(0).rename("pdd"),
        tmean.min(0).abs().rename("fdd"),
        tmax.gt(0).rename("thaw")
    ])

results = []

for warming in [0, 1, 2, 3]:
    totals = (
        test_daymet
        .map(lambda img: warming_bands(img, warming))
        .sum()
        .updateMask(mask_2000m)
    )

    values = totals.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=test_geom,
        scale=1000,
        maxPixels=1e9
    ).getInfo()

    results.append({
        "warming_C": warming,
        "pdd": values["pdd"],
        "fdd": values["fdd"],
        "thaw_days": values["thaw"]
    })

warming_test = pd.DataFrame(results)

print("TETON COUNTY, WY — 2015 WARMING SCENARIOS")
print("=" * 65)
print(warming_test.round(2).to_string(index=False))

In [ ]:
print("RANDOM FOREST FEATURES")
print("=" * 55)

print(final_rf.feature_names_in_ if hasattr(final_rf, "feature_names_in_")
      else "Feature names not stored directly")

print("\nXGBOOST FEATURES")
print("=" * 55)

print(xgb.feature_names_in_ if hasattr(xgb, "feature_names_in_")
      else "Feature names not stored directly")

print("\nMODEL DATA COLUMNS")
print("=" * 55)

print(model_df.columns.tolist())

In [ ]:
import ee
import pandas as pd

ee.Initialize(project="fdos-491217")

# Cancel the original large export
old_status = task.status()["state"]

if old_status in ["READY", "RUNNING"]:
    task.cancel()
    print("Original export cancellation requested.")
else:
    print("Original export status:", old_status)

# Colorado county-years only
co_rows = (
    model_df.loc[
        model_df["state"].astype(str).isin(["8", "08", "Colorado"]),
        ["fips", "year"]
    ]
    .drop_duplicates()
    .copy()
)

co_rows["fips"] = co_rows["fips"].astype(str).str.zfill(5)

# Fallback using FIPS prefix
if co_rows.empty:
    co_rows = model_df[["fips", "year"]].drop_duplicates().copy()
    co_rows["fips"] = co_rows["fips"].astype(str).str.zfill(5)
    co_rows = co_rows[co_rows["fips"].str.startswith("08")]

co_rows["year"] = co_rows["year"].astype(int)

tiger = ee.FeatureCollection("TIGER/2018/Counties")
daymet = ee.ImageCollection("NASA/ORNL/DAYMET_V4").select(
    ["tmin", "tmax"]
)
mask = ee.Image("USGS/SRTMGL1_003").gte(2000)

warmings = [0, 1, 2, 3]

def build_county_year(fips, year):
    geom = tiger.filter(
        ee.Filter.eq("GEOID", fips)
    ).first().geometry()

    daily = daymet.filterDate(
        f"{year - 1}-11-01",
        f"{year}-05-01"
    )

    # Calculate all warming scenarios in one daily image
    def daily_bands(img):
        tmean = img.select("tmin").add(
            img.select("tmax")
        ).divide(2)

        tmax = img.select("tmax")

        bands = []

        for w in warmings:
            shifted_mean = tmean.add(w)
            shifted_max = tmax.add(w)

            bands.extend([
                shifted_mean.max(0).rename(f"pdd_{w}"),
                shifted_mean.min(0).abs().rename(f"fdd_{w}"),
                shifted_max.gt(0).rename(f"thaw_{w}")
            ])

        return ee.Image.cat(bands)

    # One seasonal sum for all warming levels
    totals = daily.map(daily_bands).sum().updateMask(mask)

    # One spatial reduction per county-year
    values = totals.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=geom,
        scale=1000,
        maxPixels=1e9,
        tileScale=4
    )

    return ee.Feature(None, values).set({
        "fips": fips,
        "year": year
    })

features = [
    build_county_year(row.fips, row.year)
    for row in co_rows.itertuples(index=False)
]

co_fc = ee.FeatureCollection(features)

co_task = ee.batch.Export.table.toDrive(
    collection=co_fc,
    description="warming_scenarios_colorado",
    fileFormat="CSV"
)

co_task.start()

print("Colorado county-years:", len(co_rows))
print("Warming scenarios:", warmings)
print("Export status:", co_task.status()["state"])
print("Task ID:", co_task.id)

In [ ]:
status = co_task.status()

print("State:", status["state"])
print("Attempt:", status.get("attempt", 0))
print("Compute seconds:", status.get("batch_eecu_usage_seconds", 0))

if status.get("error_message"):
    print("Error:", status["error_message"])

In [ ]:
import pandas as pd
from pathlib import Path

path = Path.home() / "Files/thesis/snow_ml/ml_model/warming_scenarios_colorado.csv"

co_warming = pd.read_csv(path, dtype={"fips": str})
co_warming["fips"] = co_warming["fips"].str.zfill(5)

# Baseline warming = 0°C
baseline = co_warming[
    ["fips", "year", "pdd_0", "fdd_0", "thaw_0"]
]

original = model_df.copy()
original["fips"] = original["fips"].astype(str).str.zfill(5)

comparison = baseline.merge(
    original[
        ["fips", "year",
         "positive_degree_days_2000m",
         "freezing_degree_days_2000m",
         "thaw_days_2000m"]
    ],
    on=["fips", "year"],
    validate="one_to_one"
)

print("Matched county-years:", len(comparison))

for new, old in [
    ("pdd_0", "positive_degree_days_2000m"),
    ("fdd_0", "freezing_degree_days_2000m"),
    ("thaw_0", "thaw_days_2000m")
]:
    errors = comparison[new] - comparison[old]
    print(
        f"{old}: "
        f"Mean absolute difference = {errors.abs().mean():.3f}, "
        f"Maximum difference = {errors.abs().max():.3f}"
    )

In [ ]:
# Submit optimized warming exports for remaining four states

remaining_states = {
    "utah": "49",
    "wyoming": "56",
    "montana": "30",
    "idaho": "16"
}

state_tasks = {}

for state_name, prefix in remaining_states.items():

    rows = model_df[["fips", "year"]].drop_duplicates().copy()
    rows["fips"] = rows["fips"].astype(str).str.zfill(5)

    rows = rows[rows["fips"].str.startswith(prefix)]
    rows["year"] = rows["year"].astype(int)

    features = [
        build_county_year(row.fips, row.year)
        for row in rows.itertuples(index=False)
    ]

    fc = ee.FeatureCollection(features)

    export_task = ee.batch.Export.table.toDrive(
        collection=fc,
        description=f"warming_scenarios_{state_name}",
        fileFormat="CSV"
    )

    export_task.start()
    state_tasks[state_name] = export_task

    print(
        f"{state_name.title():10s} "
        f"County-years: {len(rows):3d}  "
        f"Status: {export_task.status()['state']}"
    )

In [ ]:
from pathlib import Path
import pandas as pd

folder = Path.home() / "Files/thesis/snow_ml/ml_model"

states = ["colorado", "utah", "wyoming", "montana", "idaho"]

frames = []

for state in states:
    path = folder / f"warming_scenarios_{state}.csv"
    df = pd.read_csv(path, dtype={"fips": str})
    df["fips"] = df["fips"].str.zfill(5)
    df["year"] = df["year"].astype(int)
    frames.append(df)
    print(f"{state.title():10s}: {len(df)} rows")

warming_all = pd.concat(frames, ignore_index=True)

# Check uniqueness
assert not warming_all.duplicated(["fips", "year"]).any(), \
    "Duplicate county-year records found"

# Compare historical baseline to original data
original = model_df.copy()
original["fips"] = original["fips"].astype(str).str.zfill(5)
original["year"] = original["year"].astype(int)

comparison = warming_all.merge(
    original[
        ["fips", "year",
         "positive_degree_days_2000m",
         "freezing_degree_days_2000m",
         "thaw_days_2000m"]
    ],
    on=["fips", "year"],
    validate="one_to_one"
)

print("\nFULL DATASET VALIDATION")
print("=" * 60)
print("County-years:", len(warming_all))
print("Unique counties:", warming_all["fips"].nunique())
print("Matched observations:", len(comparison))

for new, old in [
    ("pdd_0", "positive_degree_days_2000m"),
    ("fdd_0", "freezing_degree_days_2000m"),
    ("thaw_0", "thaw_days_2000m")
]:
    error = comparison[new] - comparison[old]

    print(
        f"{old:32s} "
        f"MAE: {error.abs().mean():.4f}  "
        f"Max: {error.abs().max():.4f}"
    )

# Ensure no observations are missing
assert len(warming_all) == len(original) == len(comparison) == 400
assert warming_all[
    [f"{v}_{w}" for v in ["pdd", "fdd", "thaw"] for w in [0, 1, 2, 3]]
].notna().all().all(), "Missing scenario values found"

# Save combined scenarios
output = folder / "county_warming_scenarios_all.csv"
warming_all.to_csv(output, index=False)

print("\nCombined dataset saved:", output)

In [ ]:
import numpy as np
import pandas as pd

# Prepare historical model data
base = model_df.copy()
base["fips"] = base["fips"].astype(str).str.zfill(5)
base["year"] = base["year"].astype(int)

# All six seasonal temperature predictors
temp_cols = [
    "tmin_djf_2000m",
    "tmax_djf_2000m",
    "tmin_mar_2000m",
    "tmax_mar_2000m",
    "tmin_apr_2000m",
    "tmax_apr_2000m"
]

scenario_frames = []

for warming in [0, 1, 2, 3]:

    scenario = base.merge(
        warming_all[
            ["fips", "year",
             f"pdd_{warming}",
             f"fdd_{warming}",
             f"thaw_{warming}"]
        ],
        on=["fips", "year"],
        validate="one_to_one"
    )

    # Increase minimum and maximum temperatures
    for col in temp_cols:
        scenario[col] = scenario[col] + warming

    # Replace threshold-derived variables
    scenario["positive_degree_days_2000m"] = scenario[f"pdd_{warming}"]
    scenario["freezing_degree_days_2000m"] = scenario[f"fdd_{warming}"]
    scenario["thaw_days_2000m"] = scenario[f"thaw_{warming}"]

    scenario["warming_C"] = warming

    scenario = scenario.drop(
        columns=[f"pdd_{warming}", f"fdd_{warming}", f"thaw_{warming}"]
    )

    scenario_frames.append(scenario)

warming_model_df = pd.concat(
    scenario_frames,
    ignore_index=True
)

# Verify dimensions and temperature changes
assert len(warming_model_df) == 1600
assert not warming_model_df.duplicated(
    ["fips", "year", "warming_C"]
).any()

print("WARMING SCENARIO DATASET")
print("=" * 60)
print("Total observations:", len(warming_model_df))
print("Counties:", warming_model_df["fips"].nunique())
print("Years:", warming_model_df["year"].nunique())
print("Warming levels:", sorted(warming_model_df["warming_C"].unique()))

print("\nAVERAGE CLIMATE VARIABLES BY WARMING LEVEL")
print("=" * 60)

summary_cols = [
    "tmin_djf_2000m",
    "positive_degree_days_2000m",
    "freezing_degree_days_2000m",
    "thaw_days_2000m"
]

print(
    warming_model_df.groupby("warming_C")[summary_cols]
    .mean()
    .round(2)
    .to_string()
)

In [ ]:
import numpy as np
import pandas as pd

# Use the trained models' original feature sets
rf_features = list(final_rf.feature_names_in_)
xgb_features = list(xgb.feature_names_in_)

# Create prediction dataset
snow_predictions = warming_model_df[
    ["fips", "state", "county", "year", "warming_C"]
].copy()

# Random Forest
snow_predictions["Random Forest"] = final_rf.predict(
    warming_model_df[rf_features]
)

# XGBoost
snow_predictions["XGBoost"] = xgb.predict(
    warming_model_df[xgb_features]
)

# Fixed Effects: use the original trained pipeline
fe_features = [
    "fips",
    "positive_degree_days_2000m",
    "precip_djf_2000m",
    "precip_mar_apr_2000m"
]

snow_predictions["Fixed Effects"] = fe_model.predict(
    warming_model_df[fe_features]
)

# Keep predictions within the possible snow-season duration
for model in ["Fixed Effects", "Random Forest", "XGBoost"]:
    snow_predictions[model] = snow_predictions[model].clip(
        lower=0,
        upper=warming_model_df["season_days"].to_numpy()
    )

# Compare each scenario against its own model's baseline
baseline = snow_predictions.loc[
    snow_predictions["warming_C"] == 0,
    ["fips", "year", "Fixed Effects", "Random Forest", "XGBoost"]
].copy()

baseline = baseline.rename(columns={
    model: f"{model}_baseline"
    for model in ["Fixed Effects", "Random Forest", "XGBoost"]
})

snow_predictions = snow_predictions.merge(
    baseline,
    on=["fips", "year"],
    validate="many_to_one"
)

for model in ["Fixed Effects", "Random Forest", "XGBoost"]:
    snow_predictions[f"{model}_change"] = (
        snow_predictions[model]
        - snow_predictions[f"{model}_baseline"]
    )

# Summarize average changes
summary = snow_predictions.groupby("warming_C")[
    [
        "Fixed Effects_change",
        "Random Forest_change",
        "XGBoost_change"
    ]
].mean()

print("AVERAGE CHANGE IN PREDICTED SNOW-COVER DAYS")
print("=" * 65)
print(summary.round(2).to_string())

# Save detailed predictions
output = (
    Path.home()
    / "Files/thesis/snow_ml/ml_model/snow_warming_predictions.csv"
)

snow_predictions.to_csv(output, index=False)

print("\nSaved:", output)

In [ ]:
# State-level average changes in predicted snow-cover days

state_names = {
    "08": "Colorado",
    "16": "Idaho",
    "30": "Montana",
    "49": "Utah",
    "56": "Wyoming"
}

state_results = snow_predictions.copy()

state_results["state_name"] = (
    state_results["fips"]
    .astype(str)
    .str.zfill(5)
    .str[:2]
    .map(state_names)
)

state_summary = (
    state_results[state_results["warming_C"] > 0]
    .groupby(["state_name", "warming_C"])[
        [
            "Fixed Effects_change",
            "Random Forest_change",
            "XGBoost_change"
        ]
    ]
    .mean()
    .round(2)
    .reset_index()
)

print("STATE-LEVEL SNOW-COVER CHANGES")
print("=" * 75)
print(state_summary.to_string(index=False))

In [ ]:
models = ["Fixed Effects", "Random Forest", "XGBoost"]

pct_results = snow_predictions.copy()

state_names = {
    "08": "Colorado",
    "16": "Idaho",
    "30": "Montana",
    "49": "Utah",
    "56": "Wyoming"
}

pct_results["state_name"] = (
    pct_results["fips"].astype(str).str.zfill(5).str[:2].map(state_names)
)

for model in models:
    pct_results[f"{model}_pct_change"] = (
        100
        * pct_results[f"{model}_change"]
        / pct_results[f"{model}_baseline"].replace(0, float("nan"))
    )

# State averages: aggregate baseline and scenario days first,
# then calculate percentage change
rows = []

for (state, warming), group in pct_results.groupby(
    ["state_name", "warming_C"]
):
    if warming == 0:
        continue

    result = {"state": state, "warming_C": warming}

    for model in models:
        baseline_mean = group[f"{model}_baseline"].mean()
        change_mean = group[f"{model}_change"].mean()

        result[model] = (
            100 * change_mean / baseline_mean
            if baseline_mean > 0 else float("nan")
        )

    rows.append(result)

state_pct_summary = pd.DataFrame(rows).sort_values(
    ["state", "warming_C"]
)

print("STATE-LEVEL PERCENTAGE CHANGE IN SNOW-COVER DAYS")
print("=" * 75)
print(state_pct_summary.round(2).to_string(index=False))

In [ ]:
import pandas as pd

# BEA 2023 state-level ski and snowboard value added
economic_base = orsa_2023[
    ["state", "baseline_va_millions"]
].copy()

# Check state names
print("BEA states:", economic_base["state"].tolist())

# Reshape snow-cover percentage changes
snow_pct_long = state_pct_summary.melt(
    id_vars=["state", "warming_C"],
    value_vars=["Fixed Effects", "Random Forest", "XGBoost"],
    var_name="Model",
    value_name="snow_change_pct"
)

# Merge with BEA economic baseline
economic_warming = snow_pct_long.merge(
    economic_base,
    on="state",
    how="left",
    validate="many_to_one"
)

assert economic_warming["baseline_va_millions"].notna().all(), \
    "Some states did not match BEA data"

# Illustrative demand elasticity scenarios
elasticities = [0.25, 0.50, 1.00]

economic_results = []

for elasticity in elasticities:

    scenario = economic_warming.copy()
    scenario["elasticity"] = elasticity

    # Assumed percentage change in economic activity
    scenario["economic_change_pct"] = (
        scenario["snow_change_pct"] * elasticity
    )

    # Change in annual value added, millions of 2023 dollars
    scenario["va_change_millions"] = (
        scenario["baseline_va_millions"]
        * scenario["economic_change_pct"] / 100
    )

    economic_results.append(scenario)

economic_results = pd.concat(
    economic_results,
    ignore_index=True
)

# Aggregate across five states
economic_totals = (
    economic_results
    .groupby(["warming_C", "Model", "elasticity"])[
        "va_change_millions"
    ]
    .sum()
    .reset_index()
)

print("\nILLUSTRATIVE ANNUAL CHANGE IN SKI VALUE ADDED")
print("Millions of 2023 dollars")
print("=" * 75)

print(
    economic_totals.pivot_table(
        index=["warming_C", "Model"],
        columns="elasticity",
        values="va_change_millions"
    ).round(2).to_string()
)

In [ ]:
# State-level economic results for the middle elasticity scenario
# Use +1°C, +2°C, and +3°C warming

state_economic = economic_results[
    economic_results["elasticity"] == 0.50
].copy()

state_economic_summary = (
    state_economic
    .pivot_table(
        index=["state", "warming_C"],
        columns="Model",
        values="va_change_millions"
    )
    .round(2)
    .reset_index()
)

print("STATE-LEVEL ECONOMIC IMPACTS")
print("Elasticity = 0.50 | Millions of 2023 dollars")
print("=" * 80)

print(state_economic_summary.to_string(index=False))

# Save for thesis tables
from pathlib import Path

output = (
    Path.home()
    / "Files/thesis/snow_ml/ml_model/state_warming_economic_impacts.csv"
)

economic_results.to_csv(output, index=False)

print("\nSaved:", output)

In [ ]:
print("ORSA columns:")
print(orsa.columns.tolist())

print("\n2023 BEA data:")
print(
    orsa.loc[
        orsa["year"] == 2023,
        ["state", "ski_snowboard_va"]
    ].to_string(index=False)
)

print("\nTotal reported value:")
print(
    orsa.loc[
        orsa["year"] == 2023,
        "ski_snowboard_va"
    ].sum()
)